# Biological System Modeling — dispensa-notebook completa in Python

Questa dispensa è un **notebook didattico operativo**: non solo formule, ma
codice, simulazioni, schemi elettrici, diagrammi a blocchi, grafici e
spiegazioni passo-passo. L'obiettivo è partire **da zero** — matematica,
Python, elettrotecnica, modellistica — e arrivare a un livello sufficiente
per leggere, implementare, modificare e discutere modelli fisiologici
dinamici a livello universitario / PhD.

## Come usare questa dispensa

1. **Leggi prima la teoria** di ciascuna sezione, poi esegui il codice.
2. **Esegui le celle in ordine** (alcune importano variabili condivise).
3. Quando vedi una matrice, chiediti *cosa rappresenta ogni riga*: i modelli
   fisiologici sono sempre bilanci, non magia.
4. Quando vedi un grafico, chiediti *quale equazione lo ha generato* e
   *cosa cambierebbe variando i parametri*.
5. Alla fine di ogni capitolo modifica i parametri proposti negli esercizi
   e osserva cosa cambia: la modellistica si impara giocando.

## Struttura della dispensa

| Parte | Contenuto |
|---|---|
| 0 | Setup Python operativo (numpy, scipy, matplotlib) |
| 1 | Recap matematico totale: vettori, matrici, autovalori, $e^{At}$, ODE, Taylor, Jacobiano, Laplace, funzioni di trasferimento, integrazione numerica |
| 2 | Elettrotecnica essenziale + analogie idraulico-elettriche per fisiologia |
| 3 | Teoria generale dei modelli: stato, ingressi, uscite, identificabilità, calibrazione, validazione, sensitività |
| 4 | Sistemi lineari LTI: stato-spazio, $e^{At}$, funzione di trasferimento, stabilità BIBO, classificazione 2D, feedback, Nyquist, ritardi |
| 5 | Sistemi non lineari: linearizzazione, Jacobiano, Hartman–Grobman, biforcazioni, Van der Pol, Hopf, caos (Lorenz, Rössler) |
| 6 | Dinamica delle popolazioni: Malthus, logistica, Lotka–Volterra, risposte funzionali |
| 7 | Modelli fisiologici: cardiovascolare con baroriflesso, emodialisi, meccanica respiratoria, scambio gas, controllo chemocettoriale, Cheyne–Stokes, Nernst, Hodgkin–Huxley |
| 8 | Metodi numerici avanzati: Euler esplicito/implicito, Runge–Kutta, stiffness, root finding |
| 9 | Esercizi guidati con soluzioni |
| App | Cheat-sheet finale, errori tipici, cosa fare dopo |

## Filosofia didattica

Per ogni concetto rilevante seguo sempre lo stesso schema:

1. **Definizione** rigorosa.
2. **Intuizione** fisica/biologica.
3. **Dimostrazione** o derivazione esplicita.
4. **Matrice o equazioni scalari** (mai un simbolo lasciato astratto).
5. **Interpretazione** fisiologica.
6. **Implementazione Python a mano** (per capire).
7. **Implementazione con libreria** (per scalare).
8. **Grafico esplicativo** con annotazioni.
9. **Esercizio** da modificare.

> *Le formule sono solo "fotografie compresse" di un processo. Il senso lo dà
> sempre il bilancio che ne sta dietro: massa, carica, energia, momento.*


## Fonti e filosofia del notebook

La dispensa sintetizza il programma del corso di **Biological System Modeling /
Teoria dei modelli** (lecture notes su sistemi dinamici, stabilità, feedback,
sistemi non lineari, popolazioni, emodialisi, respirazione, controllo
ventilatorio, elettrofisiologia cellulare, modello di Hodgkin–Huxley) e usa
come riferimento concettuale i testi di fisiologia matematica (Keener &
Sneyd) e di teoria dei sistemi (Strogatz, Khalil, Antonelli). I contenuti
sono **riscritti e parafrasati**: nessuna copia testuale.

Convenzioni di notazione usate ovunque nel notebook:

- $t$ = tempo (s, ms, min — esplicitato volta per volta);
- vettori di stato: lettera maiuscola, ad es. $\mathbf{X}(t)\in\mathbb{R}^n$;
- componenti: $x_i(t)$ con $i=1,\dots,n$;
- derivata temporale: $\dot x \equiv dx/dt$;
- matrice di sistema $A\in\mathbb{R}^{n\times n}$, matrice degli ingressi
  $B\in\mathbb{R}^{n\times m}$, matrice di uscita $C\in\mathbb{R}^{p\times n}$;
- pressioni in mmHg (fisiologia cardiocircolatoria) o cmH$_2$O (respiratoria);
- volumi in mL o L; flussi in mL/s o L/min; concentrazioni in mmol/L o mEq/L;
- potenziali transmembrana in mV; conduttanze in mS/cm$^2$; correnti in
  µA/cm$^2$; capacità in µF/cm$^2$.

> ⚠️ Le unità non sono un dettaglio decorativo. Quando un'equazione "non torna",
> il **primo** controllo è sempre dimensionale.


# Parte 0 — Setup Python: diventare operativi

Per modellare sistemi biologici servono pochi strumenti, ma vanno capiti bene.

| Libreria | A cosa serve |
|---|---|
| `numpy` | array, algebra lineare, broadcast, operazioni vettorializzate |
| `scipy.integrate.solve_ivp` | integrazione robusta di ODE (RK45, BDF, Radau...) |
| `scipy.linalg.expm` | esponenziale di matrice $e^{At}$ |
| `scipy.linalg.eig` | autovalori e autovettori |
| `scipy.optimize.root` | risoluzione di equazioni non lineari (equilibri, metodi impliciti) |
| `scipy.signal` | trasformate, filtri, funzioni di trasferimento |
| `matplotlib` | grafici temporali, phase plane, biforcazioni, diagrammi |
| `sympy` *(opzionale)* | derivazioni simboliche, Jacobiano in forma chiusa |

La cella di setup successiva fissa **una sola volta** lo stile dei grafici
e importa tutto ciò che usiamo dopo. Mantenerla all'inizio garantisce che
ogni figura del notebook abbia un aspetto coerente e pedagogicamente pulito.


In [ ]:
# =============================================================================
# SETUP GENERALE DEL NOTEBOOK
# =============================================================================
# Importa le librerie usate nell'intero notebook e fissa uno stile grafico
# coerente. Eseguire SEMPRE prima delle altre celle.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Rectangle, Circle, FancyBboxPatch, Polygon

# Algebra lineare e ODE
from scipy.integrate import solve_ivp
from scipy.linalg import expm, eig
from scipy.optimize import root, brentq, fsolve

# Stampe numpy leggibili
np.set_printoptions(precision=4, suppress=True, linewidth=110)

# Stile matplotlib pedagogico: figure leggibili, griglia, font medio-grande
plt.rcParams.update({
    "figure.figsize": (8.5, 5.0),
    "figure.dpi": 100,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "lines.linewidth": 1.8,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Palette consistente per tutto il notebook
COL = {
    "main":   "#1f4e79",
    "accent": "#c0392b",
    "ok":     "#1e8449",
    "warn":   "#d68910",
    "extra":  "#6c3483",
    "grey":   "#4d5656",
}

print("Ambiente pronto.")
import sys, scipy, matplotlib
print("Versioni:")
print(f"  python      {sys.version.split()[0]}")
print(f"  numpy       {np.__version__}")
print(f"  scipy       {scipy.__version__}")
print(f"  matplotlib  {matplotlib.__version__}")


### Funzioni helper per i grafici

Definiamo una volta sola **funzioni di utilità** per i grafici ricorrenti
(phase plane, vector field, schemi a blocchi). Usandole in tutte le sezioni
si ottiene uno stile uniforme, e si vede esplicitamente cosa entra in ogni
figura senza ripetere il codice.


In [ ]:
# =============================================================================
# HELPER PER GRAFICI E SCHEMI
# =============================================================================
# Funzioni di utilita' usate piu' volte: campo vettoriale, traiettorie,
# blocchi di schemi a blocchi, frecce annotate, etichette di punti.
# =============================================================================

def annotate_point(ax, x, y, text, dx=0.15, dy=0.15, color="black"):
    """Aggiunge un punto evidenziato e una callout testuale accanto."""
    ax.plot(x, y, "o", color=color, markersize=7, zorder=5)
    ax.annotate(text, xy=(x, y), xytext=(x+dx, y+dy),
                fontsize=10, color=color,
                arrowprops=dict(arrowstyle="-", color=color, lw=0.9))


def plot_vector_field(ax, f, xlim, ylim, n=22, scale=1.0):
    """Disegna il campo vettoriale di un sistema 2D dx/dt = f(t, [x,y])."""
    X, Y = np.meshgrid(np.linspace(*xlim, n), np.linspace(*ylim, n))
    DX = np.zeros_like(X); DY = np.zeros_like(Y)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            v = f(0.0, np.array([X[i, j], Y[i, j]]))
            DX[i, j], DY[i, j] = v[0], v[1]
    M = np.hypot(DX, DY); M[M == 0] = 1.0
    ax.quiver(X, Y, DX/M, DY/M, M, cmap="Blues", pivot="mid",
              scale=25/scale, width=0.0035, alpha=0.85)


def plot_trajectories(ax, f, ics, t_max=20.0, color=None, lw=1.6):
    """Integra il sistema da ogni condizione iniziale e disegna le traiettorie."""
    for i, x0 in enumerate(ics):
        sol = solve_ivp(f, (0, t_max), x0, dense_output=True,
                        rtol=1e-8, atol=1e-10, max_step=0.05)
        c = color if color is not None else plt.cm.tab10(i % 10)
        ax.plot(sol.y[0], sol.y[1], color=c, lw=lw)
        ax.plot(*x0, "o", color=c, markersize=6)


def draw_box(ax, x, y, w, h, label, fc="#fdfefe", ec="#1f4e79", fontsize=10):
    """Disegna un blocco rettangolare etichettato (per schemi a blocchi)."""
    box = FancyBboxPatch((x, y), w, h,
                         boxstyle="round,pad=0.02,rounding_size=0.05",
                         fc=fc, ec=ec, lw=1.6)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=fontsize)


def draw_arrow(ax, p0, p1, label=None, color="#1f4e79", offset=(0.0, 0.08),
               lw=1.5, mutation_scale=14):
    """Disegna una freccia tra due punti, con etichetta opzionale a meta'."""
    arr = FancyArrowPatch(p0, p1, arrowstyle="->", mutation_scale=mutation_scale,
                          color=color, lw=lw)
    ax.add_patch(arr)
    if label is not None:
        mx = (p0[0] + p1[0]) / 2 + offset[0]
        my = (p0[1] + p1[1]) / 2 + offset[1]
        ax.text(mx, my, label, ha="center", va="bottom", fontsize=10, color=color)


print("Helper grafici definiti.")


# Parte 0.4 — Python da zero per chi non ha mai programmato

Se non hai mai scritto codice prima d'ora, questa sezione è per te.
Spieghiamo, passo passo e *senza dare nulla per scontato*, i pochi
concetti di Python necessari per leggere, capire, e modificare tutto il
codice del notebook. Se sei già a tuo agio in Python, salta pure alla
Parte 0.5.

> Filosofia: in modellistica numerica usiamo Python come una **lavagna
> eseguibile**. Scrivi una formula, premi `Shift+Enter`, vedi il risultato.
> Le cose che useremo davvero sono **dieci**: variabili, tipi, liste,
> dizionari, funzioni, loop, condizioni, importazioni, numpy, matplotlib.
> Le copriamo tutte qui sotto.


## 0.4.1 Variabili e tipi

Una **variabile** è un nome a cui assegniamo un valore con `=`.
Python "indovina" da solo il tipo del valore.

```python
x = 5              # intero (int)
pi = 3.14          # decimale (float)
nome = "ciao"      # stringa (str)
ok  = True         # booleano (bool)
```

I **tipi** che useremo sono solo 5:

| Tipo | Esempio | Cosa contiene |
|---|---|---|
| `int` | `5`, `-2`, `100` | numero intero |
| `float` | `3.14`, `-0.5`, `1e-3` | numero decimale |
| `bool` | `True`, `False` | vero/falso |
| `str` | `"ciao"`, `'V_m'` | testo (tra virgolette) |
| `None` | `None` | "nessun valore" |

Per scoprire il tipo: `type(x)`. Per convertire: `int(3.7)`, `float("2.5")`, `str(42)`.


In [ ]:
# Esempi base di variabili e tipi
x = 5
pi = 3.14
nome = "membrana"
ok = True

print(x, type(x))
print(pi, type(pi))
print(nome, type(nome))
print(ok, type(ok))

# Operazioni elementari
print("5 + 3 =", 5 + 3)
print("10 / 3 =", 10 / 3)      # divisione "vera": resta float
print("10 // 3 =", 10 // 3)    # divisione intera
print("10 % 3 =", 10 % 3)      # modulo (resto)
print("2 ** 10 =", 2 ** 10)    # elevamento a potenza


## 0.4.2 Stringhe formattate (f-strings)

Per stampare risultati misti testo+numeri usiamo le **f-string**
(stringhe con `f` davanti). Dentro `{...}` puoi mettere variabili o
espressioni; con `:.2f` chiedi 2 decimali, `:.4e` notazione scientifica
a 4 decimali.

```python
V = -65.32
print(f"Il potenziale di membrana è {V:.1f} mV")
# Stampa: "Il potenziale di membrana è -65.3 mV"
```


In [ ]:
# Esempi di f-string
V = -65.32
tau = 3.333
print(f"V = {V} mV")
print(f"V = {V:.1f} mV (1 decimale)")
print(f"V = {V:+.2e} mV (notazione scientifica)")
print(f"tau = {tau:.3f} ms,  1/tau = {1/tau:.3f} 1/ms")
print(f"Sotto soglia: {V < -55}")


## 0.4.3 Liste, tuple, dizionari

Sono "contenitori" per più valori.

```python
# Lista: collezione ordinata, modificabile
poteni = [-70, -65, -50, +30]
print(poteni[0])      # -70 (indice 0 = primo elemento!)
print(poteni[-1])     # +30 (l'ultimo)
print(poteni[1:3])    # [-65, -50] (slicing dall'indice 1 escluso 3)
poteni.append(45)     # aggiungi in fondo

# Tupla: come una lista ma immutabile (non si può modificare)
punto = (1.5, 2.7)
x, y = punto          # "unpacking": x=1.5, y=2.7

# Dizionario: coppie chiave -> valore
parametri = {"C_m": 1.0, "g_L": 0.3, "E_L": -65.0}
print(parametri["C_m"])           # 1.0
parametri["g_K"] = 36.0           # aggiungo una chiave
for chiave, valore in parametri.items():
    print(chiave, "=", valore)
```

**Importante**: l'indicizzazione in Python parte da **0** (non da 1).
Lo slicing `[a:b]` prende gli elementi da `a` *incluso* a `b` *escluso*.


In [ ]:
# Esempi su liste, tuple, dizionari
poteni = [-70, -65, -50, +30]
print("Lista:", poteni)
print("Primo:", poteni[0], " Ultimo:", poteni[-1])
print("Slice [1:3]:", poteni[1:3])
poteni.append(45)
print("Dopo append(45):", poteni)
print("Lunghezza:", len(poteni))

# Tupla
punto = (1.5, 2.7)
x, y = punto
print(f"Punto: x={x}, y={y}")

# Dizionario
parametri = {"C_m": 1.0, "g_L": 0.3, "E_L": -65.0}
for k, v in parametri.items():
    print(f"{k:5s} = {v}")


## 0.4.4 Condizioni: `if` / `elif` / `else`

Le condizioni servono a *fare scelte*. La sintassi usa l'**indentazione**
(4 spazi) per delimitare i blocchi.

```python
V = -55.0
if V > -50:
    print("sopra soglia")
elif V > -65:
    print("vicino al riposo")
else:
    print("iperpolarizzato")
```

Gli **operatori di confronto**: `==` (uguale), `!=` (diverso),
`<`, `<=`, `>`, `>=`.

Gli **operatori logici**: `and`, `or`, `not`.


In [ ]:
# Esempi di condizioni
def stato_membrana(V):
    if V > -40:
        return "depolarizzata (spike?)"
    elif V > -55:
        return "vicino a soglia"
    elif V > -65:
        return "riposo o lieve iperpol."
    else:
        return "iperpolarizzata"

for V in [-80, -65, -52, -30, +10]:
    print(f"V = {V:+5.0f} mV  -> {stato_membrana(V)}")


## 0.4.5 Loop: `for` e `while`

Un loop ripete un blocco di codice. Il più usato è `for` su un iterabile:

```python
for V in [-80, -60, -40]:           # itera sulla lista
    print(V)

for k in range(5):                  # 0, 1, 2, 3, 4
    print(k)

for k in range(2, 10, 2):           # 2, 4, 6, 8 (start, stop esclusivo, step)
    print(k)
```

`while` ripete finché una condizione è vera:

```python
V = -65.0
while V < -55:                      # entra finché siamo sotto soglia
    V += 0.5                        # depolarizza di 0.5 mV
print(f"Soglia raggiunta: V={V}")
```


In [ ]:
# Loop for con range e while
print("range(5):")
for k in range(5):
    print(" ", k)

# while: simuliamo una depolarizzazione lenta
V = -65.0
n = 0
while V < -55 and n < 100:
    V += 0.5
    n += 1
print(f"Dopo {n} step, V = {V:+.1f} mV")


## 0.4.6 Funzioni: `def` e `lambda`

Una **funzione** è un blocco di codice riusabile. Si definisce con `def`
e si chiama con il suo nome seguito da `(...)`.

```python
def quadrato(x):
    return x * x

print(quadrato(5))     # 25
```

Le funzioni possono avere **parametri di default**:

```python
def membrana(V, E_L=-65.0, g_L=0.3):
    return -g_L * (V - E_L)
```

Chiamate possibili: `membrana(-60)`, `membrana(-60, E_L=-70)`,
`membrana(-60, -70, 0.5)`.

Una `lambda` è una **funzione "monoriga"** anonima:

```python
quadrato = lambda x: x * x
print(quadrato(5))      # 25
```

Le usiamo spessissimo come argomenti, per esempio nel risolutore di ODE:

```python
solve_ivp(lambda t, x: -x / tau,  (0, 5),  [1.0])
#         ↑↑↑ funzione "al volo" che riceve (t, x) e restituisce dx/dt
```


In [ ]:
# Definizione di funzioni e lambda
def quadrato(x):
    return x * x

def membrana(V, E_L=-65.0, g_L=0.3):
    return -g_L * (V - E_L)   # corrente di leak

print("quadrato(5) =", quadrato(5))
print("membrana(-60) =", membrana(-60))
print("membrana(-60, E_L=-70) =", membrana(-60, E_L=-70))

# Lambda equivalente
q = lambda x: x * x
print("lambda q(5) =", q(5))

# Funzioni come oggetti: possiamo passarle ad altre funzioni
def applica(f, valore):
    return f(valore)

print("applica(quadrato, 7) =", applica(quadrato, 7))
print("applica(lambda x: x+10, 7) =", applica(lambda x: x + 10, 7))


## 0.4.7 Numpy in 5 minuti

`numpy` è la libreria che ci permette di lavorare con **vettori e matrici**
in modo efficiente. La importiamo come `np`.

```python
import numpy as np

x = np.array([1.0, 2.0, 3.0])     # vettore (1D)
A = np.array([[1, 2],              # matrice (2D)
              [3, 4]])

print(x.shape, A.shape)            # (3,) e (2, 2)
print(x.dtype)                     # float64
```

**Operazioni element-wise** (riga per riga, colonna per colonna):

```python
x + 10        # somma 10 a ogni elemento
x * 2         # raddoppia ogni elemento
x ** 2        # eleva al quadrato ogni elemento
np.exp(x)     # esponenziale di ogni elemento
np.sin(x)     # seno di ogni elemento
```

**Algebra lineare**:

```python
A @ x         # prodotto matrice-vettore
A @ A         # prodotto matrice-matrice
np.linalg.inv(A)         # inversa
np.linalg.eig(A)         # autovalori, autovettori
```

**Creazione rapida** di vettori:

```python
np.zeros(5)              # [0, 0, 0, 0, 0]
np.ones(3)               # [1, 1, 1]
np.linspace(0, 10, 100)  # 100 punti equispaziati da 0 a 10
np.arange(0, 10, 0.1)    # da 0 a 10 (escl.) con passo 0.1
```

**Indicizzazione e slicing**:

```python
v = np.array([10, 20, 30, 40, 50])
v[0]            # 10
v[-1]           # 50 (ultimo)
v[1:4]          # [20, 30, 40]
v[v > 25]       # [30, 40, 50] (boolean indexing!)
```


In [ ]:
import numpy as np

# Crea un vettore e una matrice
x = np.array([1.0, 2.0, 3.0])
A = np.array([[1, 2],
              [3, 4]])
print("x =", x, " shape:", x.shape)
print("A =\n", A, " shape:", A.shape)

# Operazioni element-wise
print("x + 10 =", x + 10)
print("x ** 2 =", x ** 2)
print("np.exp(x) =", np.exp(x))

# Prodotto matrice-vettore
b = np.array([1.0, 0.0])
print("A @ b =", A @ b)

# Boolean indexing
v = np.array([10, 20, 30, 40, 50])
print("v[v > 25] =", v[v > 25])


## 0.4.8 Matplotlib in 5 minuti

`matplotlib` disegna grafici. Ne usiamo solo l'interfaccia "object-oriented":

```python
import matplotlib.pyplot as plt

t = np.linspace(0, 10, 200)
y = np.sin(t)

fig, ax = plt.subplots(figsize=(8, 4))   # crea figura + asse
ax.plot(t, y, label="sin(t)")             # disegna la curva
ax.set_xlabel("tempo")
ax.set_ylabel("y")
ax.set_title("Un seno")
ax.legend()
plt.show()
```

**Più curve insieme**: chiama `ax.plot(...)` più volte prima di `plt.show()`.

**Personalizzazioni** che usiamo spesso:

```python
ax.plot(t, y, color="red", lw=2, ls="--", label="...")   # linea
ax.plot(t, y, "o", color="blue", ms=6)                   # solo punti
ax.axhline(0, color="gray")                              # linea orizzontale
ax.axvline(5, color="gray", ls=":")                      # linea verticale
ax.fill_between(t, 0, y, alpha=0.3)                      # riempimento
ax.set_xlim(0, 10); ax.set_ylim(-1.2, 1.2)               # limiti
```


In [ ]:
import matplotlib.pyplot as plt

# Un grafico spiegato riga per riga
t = np.linspace(0, 10, 400)   # asse tempo: 400 punti tra 0 e 10
y1 = np.sin(t)                # prima curva
y2 = np.cos(t)                # seconda curva

fig, ax = plt.subplots(figsize=(9, 4))   # crea figura larga 9, alta 4
ax.plot(t, y1, color="#1f4e79", lw=2, label="sin(t)")
ax.plot(t, y2, color="#c0392b", lw=2, ls="--", label="cos(t)")
ax.axhline(0, color="gray", lw=0.7)
ax.set_xlabel("tempo t [s]")
ax.set_ylabel("ampiezza")
ax.set_title("Un primo grafico con due curve")
ax.legend()
plt.tight_layout()
plt.show()


## 0.4.9 Importazioni — come si "carica" una libreria

Tutto quello che non è core Python si carica con `import`:

```python
import numpy as np            # alias np (convenzione mondiale)
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp     # importa SOLO la funzione
from scipy.linalg import expm, eig        # più funzioni da scipy.linalg
```

In tutti i notebook tu vedrai `np.<qualcosa>` perché abbiamo importato
`numpy as np`. Quando vedi `solve_ivp(...)` senza prefisso, è perché
abbiamo fatto `from scipy.integrate import solve_ivp`.

## 0.4.10 Come si legge una cella che integra una ODE

Apriamo il "mattone" più importante del notebook: la chiamata a
`solve_ivp`. Decostruiamolo riga per riga.

```python
from scipy.integrate import solve_ivp

# 1) definiamo la "regola del moto": funzione che riceve (t, x) e
#    restituisce dx/dt come un array della stessa lunghezza di x.
def rhs(t, x):
    return np.array([-x[0]/3.0])   # dx/dt = -x/3 (decadimento)

# 2) condizione iniziale (un array, anche se di un solo elemento!)
x0 = np.array([1.0])

# 3) intervallo temporale di integrazione: (t_start, t_end)
t_span = (0.0, 10.0)

# 4) chiamata: solve_ivp tornerà un OGGETTO con dentro .t e .y
sol = solve_ivp(rhs, t_span, x0,
                dense_output=True,    # permette di valutare a tempi qualsiasi
                rtol=1e-8, atol=1e-10)  # tolleranze relativa/assoluta

# 5) usiamo i risultati: sol.t è il vettore tempi, sol.y[i] è la i-esima
#    componente dello stato
print(sol.t[:5])
print(sol.y[0][:5])
```

> Tutta la modellistica numerica del corso passa da questa singola
> chiamata. Capire bene questi 5 punti vale più di leggere 100 pagine.


In [ ]:
# Esempio di solve_ivp passo passo, con grafico finale
from scipy.integrate import solve_ivp

def rhs(t, x):
    # Sistema di prova: dx/dt = -x/tau
    tau = 3.0
    return np.array([-x[0]/tau])

x0 = np.array([1.0])
t_span = (0.0, 12.0)

sol = solve_ivp(rhs, t_span, x0,
                dense_output=True, rtol=1e-8, atol=1e-10, max_step=0.05)

print("solve_ivp ha prodotto", len(sol.t), "punti temporali")
print("Stato finale:", sol.y[0, -1], "(atteso ~ 0)")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sol.t, sol.y[0], color="#1f4e79", lw=2)
ax.set_xlabel("t"); ax.set_ylabel("x(t)")
ax.set_title("Output di solve_ivp per $\\dot x = -x/3$")
plt.tight_layout(); plt.show()


## 0.4.11 Riepilogo: "cheat-sheet Python operativo"

| Cosa vuoi fare | Sintassi |
|---|---|
| variabile | `x = 5` |
| stringa formattata | `f"x = {x:.2f}"` |
| lista | `[a, b, c]`; indicizza con `lst[0]`, `lst[-1]`, slice `lst[1:3]` |
| dizionario | `{"k": v}`; accedi con `d["k"]` |
| condizione | `if cond:` / `elif:` / `else:` |
| loop | `for k in range(n):` / `for x in iterable:` |
| funzione | `def f(x, y=0): return ...` |
| lambda | `lambda x: x + 1` |
| array numpy | `np.array([...])`, `np.linspace(a, b, n)`, `np.zeros(n)` |
| operazione element-wise | `x + 1`, `x ** 2`, `np.exp(x)` |
| prodotto matriciale | `A @ x`, `A @ B` |
| autovalori | `np.linalg.eig(A)` |
| matrice esponenziale | `scipy.linalg.expm(A * t)` |
| integrare ODE | `solve_ivp(rhs, (t0, tf), x0, rtol=..., atol=...)` |
| trovare zero di funzione | `scipy.optimize.root(F, x0)` |
| grafico | `fig, ax = plt.subplots(); ax.plot(t, y); plt.show()` |

**Sei pronto.** Da qui in poi tutto il codice del notebook userà
*esclusivamente* questi mattoni.


# Parte 0.5 — Matematica davvero da zero

Questa sezione è per chi sente di **non avere basi** in matematica.
Niente paura: vediamo insieme cosa significano i simboli che usiamo dopo,
con esempi numerici espliciti e grafici. Se sei già a tuo agio con
frazioni, potenze, logaritmi e derivate, salta pure a Parte 1.

> *Il vocabolario matematico è come un dizionario di parole tecniche: una
> volta capito il senso di ogni simbolo, leggere un'equazione diventa
> come leggere una frase. Costruiamoci insieme questo vocabolario.*

## 0.5.1 Numeri, frazioni, percentuali

- **Numero intero**: 1, 2, 100, −7. Niente virgola.
- **Numero decimale**: 0.5, 3.14, −0.01. C'è una "parte di unità".
- **Frazione**: $\frac{a}{b}$ significa "$a$ diviso $b$". Esempio:
  $\frac{3}{4}=0.75$.
- **Percentuale**: una frazione su 100. $25\% = \frac{25}{100} = 0.25$.

In fisiologia li mescoliamo sempre: la **saturazione** dell'emoglobina
$\text{SaO}_2$ è "frazione di emoglobina con O$_2$ legato". Se $\text{SaO}_2=0.98$,
diciamo che è "il 98%".

### Regole utili sulle frazioni

$$
\frac{a}{b}+\frac{c}{d}=\frac{ad+bc}{bd},\qquad
\frac{a}{b}\cdot \frac{c}{d}=\frac{ac}{bd},\qquad
\frac{a/b}{c/d}=\frac{a}{b}\cdot \frac{d}{c}=\frac{ad}{bc}.
$$

Esempio (utile per le concentrazioni):
"la quantità di Na in 0.5 L di plasma a 140 mmol/L" è
$0.5\,\text{L}\cdot 140\,\frac{\text{mmol}}{\text{L}}=70\,\text{mmol}$.
Nota come le "L" si elidono: questa è l'**analisi dimensionale**.

## 0.5.2 Potenze e radici

$$
a^n=\underbrace{a\cdot a \cdots a}_{n\text{ volte}},\quad a^0=1,\quad a^{-n}=\frac{1}{a^n}.
$$

Esempi:
- $10^3=1000$,
- $10^{-2}=0.01$,
- $2^5=32$.

**Notazione scientifica**: $3{.}4\cdot 10^{-3}=0.0034$. Onnipresente in
fisiologia: la concentrazione di H$^+$ in soluzione a pH 7 è $10^{-7}$ M.

**Radice quadrata**: $\sqrt{a}$ è il numero positivo $b$ tale che $b^2=a$.
$\sqrt{9}=3$, $\sqrt{2}\approx 1.414$. In modellistica appare ad esempio
nella formula della pulsazione di un sistema lineare:
$\omega_n=\sqrt{k/m}$.

## 0.5.3 Logaritmi: il "contro-potenza"

Se $y = a^x$, allora $x = \log_a y$. In parole: $\log_a y$ = "a che potenza
devo elevare $a$ per ottenere $y$?".

I due logaritmi più importanti:

- $\log_{10}$ (logaritmo decimale): $\log_{10} 1000 = 3$ perché $10^3=1000$;
- $\ln$ o $\log_e$ (logaritmo naturale, base $e\approx 2.718$):
  $\ln e = 1$, $\ln 1 = 0$, $\ln e^x = x$.

**Proprietà fondamentali** (entrambe utilissime):

$$
\log(ab)=\log a + \log b,\qquad
\log\left(\frac{a}{b}\right)=\log a - \log b,\qquad
\log(a^k)=k\,\log a.
$$

**Perché ci serve**: l'equazione di Nernst ha proprio un logaritmo
$\log_{10}([X]_o/[X]_i)$. Il pH è definito come $-\log_{10}[H^+]$.

## 0.5.4 Esponenziale $e^x$: la "crescita compatta"

$e$ (numero di Eulero) $\approx 2.71828$. La funzione $f(x)=e^x$ ha una
proprietà unica: **la sua pendenza in ogni punto è uguale al suo valore**:

$$
\frac{d e^x}{dx}=e^x.
$$

Conseguenza: l'equazione $\dot x = a x$ ha come soluzione $x(t)=x_0 e^{at}$.
Vedi figura sotto: per $a>0$ esplode esponenzialmente; per $a<0$ decade
esponenzialmente; per $a=0$ resta costante. Quasi *tutto* in fisiologia
lineare è esponenziale: rilassamento di tensione, decadimento di un
farmaco, riempimento di un compartimento.


In [ ]:
# Grafico esplicativo: esponenziali a diversi tassi
t = np.linspace(0, 5, 400)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# Crescita (a > 0)
ax = axes[0]
for a, c in zip([0.2, 0.6, 1.2], [COL["ok"], COL["main"], COL["accent"]]):
    ax.plot(t, np.exp(a*t), lw=2, color=c, label=f"$a={a}$")
ax.set_xlabel("t"); ax.set_ylabel(r"$e^{at}$")
ax.set_title(r"Crescita esponenziale ($a>0$)")
ax.legend()

# Decadimento (a < 0)
ax = axes[1]
for a, c in zip([-0.2, -0.6, -1.2], [COL["ok"], COL["main"], COL["accent"]]):
    ax.plot(t, np.exp(a*t), lw=2, color=c, label=f"$a={a}$")
    # Marker della costante di tempo tau = -1/a (per a<0)
    tau = -1/a
    if tau < 5:
        ax.axvline(tau, color=c, lw=0.7, ls=":")
        ax.plot(tau, np.exp(-1), "o", color=c, ms=6)
ax.axhline(np.exp(-1), color="gray", lw=0.6, ls="--")
ax.text(0.1, np.exp(-1)+0.03, r"$1/e\approx 0.37$", color="gray")
ax.set_xlabel("t"); ax.set_ylabel(r"$e^{at}$")
ax.set_title(r"Decadimento esponenziale ($a<0$)" + "\nI puntini = costante di tempo $\\tau=-1/a$")
ax.legend()
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — a sinistra: $e^{at}$ esplode più
ripidamente man mano che $a$ cresce. A destra: $e^{at}$ con $a<0$ decade,
e a $t=\tau=-1/a$ vale $1/e\approx 0.37$ del valore iniziale (37% in
gergo). Dopo $3\tau$ siamo al 5%, dopo $5\tau$ all'1%: in pratica
"praticamente a zero".

**Regola pratica per la dispensa**: ogni volta che vediamo "costante di
tempo $\tau$" significa esattamente "dopo $\tau$ secondi ne resta il 37%".

## 0.5.5 Derivata: la "pendenza istantanea"

La **derivata** $\dot x = dx/dt$ è il limite del rapporto incrementale:

$$
\dot x(t) = \lim_{\Delta t\to 0}\frac{x(t+\Delta t)-x(t)}{\Delta t}.
$$

In parole: **quanto cambia $x$ per ogni unità di tempo, *istante per istante***.

- Se $x$ è un volume in mL e $t$ in secondi, $\dot x$ è in mL/s — un flusso.
- Se $x$ è una concentrazione in mmol/L e $t$ in min, $\dot x$ è in
  mmol/(L·min).

Graficamente, $\dot x(t)$ è la **pendenza della tangente** alla curva
$x(t)$ nel punto $t$.

**Le regole base** (le useremo costantemente):

| Funzione | Derivata |
|---|---|
| $c$ (costante) | $0$ |
| $t$ | $1$ |
| $t^n$ | $n\,t^{n-1}$ |
| $e^{at}$ | $a\,e^{at}$ |
| $\sin(\omega t)$ | $\omega\,\cos(\omega t)$ |
| $\cos(\omega t)$ | $-\omega\,\sin(\omega t)$ |
| $\ln t$ | $1/t$ |
| $f(t)+g(t)$ | $\dot f+\dot g$ |
| $c\cdot f(t)$ | $c\,\dot f$ |
| $f(t)\,g(t)$ | $\dot f\,g+f\,\dot g$ (Leibniz) |
| $f(g(t))$ | $\dot f(g)\cdot \dot g$ (catena) |


In [ ]:
# Visualizzazione: derivata come pendenza della tangente
t = np.linspace(-1, 3, 400)
x = t**3 - 2*t**2 + 1
xdot = 3*t**2 - 4*t

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ax = axes[0]
ax.plot(t, x, color=COL["main"], lw=2.2, label="$x(t)=t^3-2t^2+1$")

# Tangenti in alcuni punti
for t0, c in zip([0.0, 1.0, 2.0], [COL["accent"], COL["ok"], COL["extra"]]):
    x0 = t0**3 - 2*t0**2 + 1
    s  = 3*t0**2 - 4*t0
    tt = np.linspace(t0 - 0.6, t0 + 0.6, 50)
    ax.plot(tt, x0 + s*(tt - t0), color=c, lw=1.6,
            label=f"tangente in $t={t0}$, $\\dot x(t)$={s:.1f}")
    ax.plot(t0, x0, "o", color=c, ms=8)
ax.set_xlabel("t"); ax.set_ylabel("x(t)")
ax.set_title("Significato geometrico della derivata = pendenza della tangente")
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(t, xdot, color=COL["accent"], lw=2.2, label=r"$\dot x(t)=3t^2-4t$")
ax.axhline(0, color="gray", lw=0.7)
# Equilibri (dot x = 0)
for t_eq in [0.0, 4/3]:
    ax.plot(t_eq, 0, "o", color=COL["ok"], ms=8)
    ax.annotate(f"$\\dot x=0$\n$t={t_eq:.2f}$", xy=(t_eq, 0),
                xytext=(t_eq+0.15, 1.5),
                arrowprops=dict(arrowstyle="->", color=COL["ok"]),
                color=COL["ok"], fontsize=9)
ax.set_xlabel("t"); ax.set_ylabel(r"$\dot x(t)$")
ax.set_title(r"La derivata $\dot x(t)$ in funzione di $t$" + "\nDove vale 0: punti stazionari di $x$")
ax.legend()
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — a sinistra: la curva $x(t)$ con tangenti
disegnate in 3 punti. La pendenza della tangente in $t=0$ è 0 (massimo
locale), in $t=1$ è $-1$ (curva decrescente), in $t=2$ è $+4$ (curva
crescente ripida). A destra: la stessa derivata $\dot x(t)$ disegnata in
funzione di $t$. I suoi zeri sono i punti **stazionari** di $x$.

## 0.5.6 Integrale: la "somma cumulativa"

L'**integrale definito** è "l'area sotto la curva":

$$
\int_a^b f(t)\,dt = \text{somma dei contributi } f(t)\,\Delta t \text{ da } a \text{ a } b.
$$

**Fatto fondamentale (Teorema fondamentale del calcolo)**: se
$F(t)$ è una "primitiva" di $f$ (cioè $\dot F = f$), allora

$$
\int_a^b f(t)\,dt = F(b) - F(a).
$$

In modellistica usiamo gli integrali continuamente:

- $V(t) = V_0 + \int_0^t Q(\tau)\,d\tau$: volume = volume iniziale + integrale del flusso;
- $\text{carico totale} = \int_0^T (\text{infusione})(\tau)\,d\tau$;
- $\text{energia} = \int_0^T P(\tau)\,d\tau$ se $P$ è una potenza.

### Connessione fondamentale tra derivata e integrale

Sono **operazioni inverse**:

$$
\frac{d}{dt}\int_0^t f(\tau)\,d\tau = f(t),\qquad
\int_0^t \dot x(\tau)\,d\tau = x(t)-x(0).
$$

Quindi *"derivata e integrale si annullano a vicenda"*: utile per
risolvere ODE (Parte 1).


In [ ]:
# Visualizzazione: integrale come area sotto la curva
t = np.linspace(0, 5, 400)
f = 0.3 * t + 0.5 * np.sin(2*t) + 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
ax = axes[0]
ax.plot(t, f, color=COL["main"], lw=2.2)
mask = (t >= 1) & (t <= 4)
ax.fill_between(t[mask], 0, f[mask], color=COL["accent"], alpha=0.35, label="area $\\int_1^4 f(t)\\,dt$")
ax.axvline(1, color="gray", ls=":")
ax.axvline(4, color="gray", ls=":")
ax.set_xlabel("t"); ax.set_ylabel("f(t)")
ax.set_title("Integrale come area sotto la curva")
ax.legend()

# Integrale numerico cumulativo (per esempio: volume da flusso)
F_cum = np.cumsum(f) * (t[1]-t[0])
ax = axes[1]
ax.plot(t, F_cum, color=COL["accent"], lw=2.2)
ax.set_xlabel("t"); ax.set_ylabel(r"$F(t)=\int_0^t f(\tau)\,d\tau$")
ax.set_title("Funzione integrale cumulativa")
plt.tight_layout(); plt.show()

print(f"Esempio: se f(t) e' un FLUSSO in mL/s, F(t) e' un VOLUME in mL.")
# np.trapezoid sostituisce np.trapz nelle versioni recenti di numpy
_trap = getattr(np, "trapezoid", getattr(np, "trapz", None))
print(f"  Integrale tra t=1 e t=4: {_trap(f[mask], t[mask]):.3f}")


## 0.5.7 Equazione differenziale ordinaria — cos'è davvero

Un'**ODE** è una **regola di evoluzione**: ti dice, dato lo stato attuale,
quanto vale la derivata. Esempio "scarica di un condensatore":

$$
\dot V = -V/\tau.
$$

In parole: "il tasso di variazione di $V$ in questo istante è
$-V/\tau$, cioè $V$ tenderà sempre a zero, con velocità proporzionale al
proprio valore". È **dinamica autoregolata**: più $V$ è grande, più
velocemente scarica; più $V$ si avvicina a zero, più lentamente scarica.

Visualmente, è una **legge del moto** sul "mondo dei valori di $V$":
ovunque tu sia, la legge ti dice in che direzione muoverti.

### Risolvere un'ODE significa: trovare la funzione $V(t)$

Per $\dot V=-V/\tau$ con $V(0)=V_0$, la **soluzione** è $V(t)=V_0\,e^{-t/\tau}$.
Verifichiamo: la derivata di $V_0 e^{-t/\tau}$ è $-V_0/\tau \cdot e^{-t/\tau}
= -V(t)/\tau$. ✓.

In modellistica:

1. **Scriviamo l'ODE** dai bilanci fisici.
2. **La risolviamo** (analiticamente se è lineare e semplice, numericamente
   se è non lineare o complessa).
3. **Confrontiamo la soluzione con i dati**.
4. **Calibriamo i parametri** finché il modello replica i dati.


In [ ]:
# Visualizzazione: campo di direzione di dot V = -V/tau e qualche traiettoria
tau = 1.5
def f(t, V): return -V/tau

T, n = 8, 25
t_grid = np.linspace(0, T, n)
V_grid = np.linspace(-2, 2, n)
TT, VV = np.meshgrid(t_grid, V_grid)
F = -VV/tau

fig, ax = plt.subplots(figsize=(11, 5.5))
# Campo come piccole frecce
ax.quiver(TT, VV, np.ones_like(F), F, F, cmap="RdBu", pivot="mid",
          scale=22, width=0.0030)
# Traiettorie da varie condizioni iniziali
for V0 in [-1.5, -1.0, -0.5, 0.5, 1.0, 1.5]:
    sol = solve_ivp(f, (0, T), [V0], dense_output=True, max_step=0.05)
    ax.plot(sol.t, sol.y[0], lw=2.0, color=COL["main"])
    ax.plot(0, V0, "o", color=COL["main"], ms=6)
ax.axhline(0, color="gray", lw=1)
ax.set_xlabel("t"); ax.set_ylabel("V")
ax.set_title(r"ODE $\dot V=-V/\tau$ come 'legge del moto': frecce + traiettorie")
ax.text(6.5, 0.1, "tutte le traiettorie\nfiniscono a V=0", color="gray", fontsize=10)
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — le frecce sono "il vento del campo": in
ogni punto $(t,V)$ ti dicono dove andare. Le linee continue sono traiettorie
seguendo quel vento, partendo da diverse condizioni iniziali $V_0$. Tutte
convergono a $V=0$: questo è il significato di **equilibrio asintoticamente
stabile**.

## 0.5.8 Notazione vettoriale: "perché tante variabili?"

In modellistica raramente abbiamo *una* variabile da seguire: in cardiovascolare
ne abbiamo 3 (pressioni), in HH ne abbiamo 4 ($V_m,m,h,n$). Le impacchettiamo
in un vettore $\mathbf{X}(t)$. La regola di evoluzione diventa una
**rete di equazioni accoppiate**:

$$
\dot{\mathbf{X}} = \mathbf{f}(\mathbf{X}),
$$

dove $\mathbf{f}$ è la *regola del moto* in spazio multidimensionale.
Vediamo passo passo come si legge: con $n=2$,

$$
\mathbf{X} = \begin{bmatrix} x_1 \\ x_2 \end{bmatrix},\quad
\dot{\mathbf{X}} = \begin{bmatrix} \dot x_1 \\ \dot x_2 \end{bmatrix},\quad
\mathbf{f}(\mathbf{X})=\begin{bmatrix} f_1(x_1,x_2) \\ f_2(x_1,x_2) \end{bmatrix}.
$$

Questa è **la** forma standard. Tutto il notebook gira intorno a essa.

## 0.5.9 Riassunto operativo della "matematica da zero"

| Concetto | Simbolo | Cosa fa |
|---|---|---|
| esponenziale | $e^{at}$ | cresce/decade molto velocemente |
| logaritmo | $\ln, \log_{10}$ | inverte la potenza |
| derivata | $\dot x, dx/dt$ | tasso di variazione istantaneo |
| integrale | $\int$ | somma cumulativa = area |
| ODE | $\dot x=f(x)$ | regola del moto |
| vettore di stato | $\mathbf{X}$ | n variabili impacchettate |
| matrice | $A$ | n×n coefficienti di accoppiamento |

Con questi sette oggetti puoi seguire qualsiasi modello del corso.


# Parte 0.6 — Ricette Python per la modellistica

Saper scrivere `x = 5` non basta per fare modellistica. Servono dei
**pattern ricorrenti** — *ricette* — che si ripetono identici in ogni
esercizio. Una volta imparati questi pattern, qualunque modello del
notebook diventa "compilabile" mentalmente in pochi minuti.

In questa parte vediamo, *con esempi eseguibili*, le 12 ricette più importanti:

| # | Ricetta | Quando si usa |
|---|---|---|
| 1 | Dalla formula matematica alla funzione `rhs(t, x)` | sempre |
| 2 | Parametri come **dizionario** | modelli con tanti parametri |
| 3 | Simulazione end-to-end con `solve_ivp` | sempre |
| 4 | Studio di sensitività con un `for` sui parametri | "cosa cambia se varia X?" |
| 5 | Stimolazione/ingresso *dipendente dal tempo* | corrente iniettata, gradino, sinusoide |
| 6 | Calcolo dell'**equilibrio numerico** con `scipy.optimize.root` | trovare $\mathbf{x}^*$ |
| 7 | Jacobiano via differenze finite + autovalori | studio di stabilità locale |
| 8 | **Phase plane** con campo vettoriale + nullcline | sistemi 2D non lineari |
| 9 | **Bifurcation diagram** numerico | esplorare cosa succede al variare di $r$ |
| 10 | Feedback: aggiungere uno **stato di controllo lento** | baroriflesso, controllo ventilatorio |
| 11 | Ritardo discreto con **buffer circolare** | Cheyne–Stokes, controlli con $\tau_d$ |
| 12 | Calibrazione di parametri (**fitting**) | confronto modello vs dati |

Le ricette si combinano: un esercizio completo è quasi sempre la sequenza
*(1) + (2) + (3) + (4)*. Andiamo.


## Ricetta 1 — Dalla formula matematica alla funzione `rhs(t, x)`

**Schema mentale**: la matematica dice "lo stato evolve secondo
$\dot{\mathbf{x}} = \mathbf{f}(t,\mathbf{x})$". Python ti chiede *una funzione
con due argomenti `t` e `x`*, che ritorna un **array numpy** dei valori di
$\dot{\mathbf{x}}$ nello stesso ordine di $\mathbf{x}$.

Mantra:
1. Disimballa $\mathbf{x}$ nei tuoi nomi fisici (`V, n, m, h = x`).
2. Calcola i termini ausiliari (correnti, flussi, gradienti).
3. Costruisci ogni componente di $\dot{\mathbf{x}}$.
4. Ritorna `np.array([dV_dt, dn_dt, dm_dt, dh_dt])` *nello stesso ordine*.

**Errore tipico**: cambiare l'ordine tra `x` e il valore di ritorno
→ il modello "scambia" le variabili e produce risultati assurdi.


In [ ]:
# Esempio: equazione di membrana RC
# Matematica:  C_m dV/dt = I_ext(t) - g_L (V - E_L)
# In Python:

C_m = 1.0; g_L = 0.3; E_L = -65.0

def rhs_membrana(t, x):
    V = x[0]                              # 1) disimballa
    I_ext = 5.0 if t >= 2.0 else 0.0      # 2) calcolo ingresso
    dV_dt = (I_ext - g_L*(V - E_L)) / C_m # 3) costruisci derivata
    return np.array([dV_dt])              # 4) array numpy STESSO ORDINE

# Test della funzione
print("rhs in (t=0, V=-65):", rhs_membrana(0.0, np.array([-65.0])))
print("rhs in (t=5, V=-50):", rhs_membrana(5.0, np.array([-50.0])))


## Ricetta 2 — Parametri come dizionario (o `dataclass`)

Quando i parametri sono molti (oltre 4-5), scriverli come variabili globali
è ingestibile. Mettili in un **dizionario** e passalo alla funzione `rhs`.

**Vantaggi**:
- ogni parametro ha un nome, non un indice;
- li puoi salvare/caricare facilmente (es. in JSON);
- li puoi modificare in un loop di sensitività senza rinominare le variabili.

**Trucco con lambda**: `solve_ivp` vuole una funzione `f(t, x)` di due
soli argomenti. Per passare i parametri usiamo un *lambda di chiusura*:
`lambda t, x: rhs(t, x, p)`.


In [ ]:
# Pattern dizionario + lambda di chiusura
def rhs_HH(t, x, p):
    V, n, m, h = x
    I_Na = p['gNa'] * m**3 * h * (V - p['ENa'])
    I_K  = p['gK']  * n**4     * (V - p['EK'])
    I_L  = p['gL']             * (V - p['EL'])
    I_ext = p['I_ext'](t)         # nota: ingresso = FUNZIONE
    dV = (I_ext - I_Na - I_K - I_L) / p['Cm']
    # Le cinetiche di gating sono qui semplificate per chiarezza
    dn = 0.1*(1-n) - 1.0*n
    dm = 0.5*(1-m) - 2.0*m
    dh = 0.05*(1-h) - 0.5*h
    return np.array([dV, dn, dm, dh])

p = {
    'Cm':  1.0,
    'gNa': 120.0, 'gK': 36.0, 'gL': 0.3,
    'ENa': 50.0,  'EK': -77.0, 'EL': -54.4,
    'I_ext': lambda t: 10.0 if 2 <= t <= 2.5 else 0.0,
}

# Test
x0 = np.array([-65.0, 0.3, 0.05, 0.6])
print("rhs in x0 con I_ext(0)=0:", rhs_HH(0.0, x0, p))
print("rhs in x0 con I_ext(2.2)=10:", rhs_HH(2.2, x0, p))


## Ricetta 3 — Simulazione end-to-end con `solve_ivp`

Una volta scritto `rhs`, simulare è sempre lo **stesso giro di battute**:

```python
from scipy.integrate import solve_ivp

x0     = np.array([...])           # 1) condizione iniziale
t_span = (0.0, T_finale)           # 2) intervallo di tempo
t_eval = np.linspace(*t_span, 800) # 3) (opz) tempi a cui valutare l'output

sol = solve_ivp(
    fun=lambda t, x: rhs(t, x, p), # 4) funzione (con lambda per chiusura)
    t_span=t_span,                 # 5) intervallo
    y0=x0,                         # 6) condizione iniziale
    t_eval=t_eval,                 # 7) (opz) tempi richiesti
    method='RK45',                 # 8) 'RK45' default, 'BDF'/'Radau' per stiff
    rtol=1e-8, atol=1e-10,         # 9) tolleranze
    max_step=0.05,                 # 10) (opz) passo massimo per stiff fast
    dense_output=True,             # 11) (opz) per valutare a tempi arbitrari
)

t = sol.t        # array dei tempi
y = sol.y        # array 2D: y[i, k] = valore della i-esima componente al k-esimo istante
print(sol.success, sol.message, len(sol.t))
```

**3 errori tipici**:

1. **Passare `x0` come numero, non array**: `x0 = 1.0` rompe; usa `x0 = np.array([1.0])`.
2. **Dimensione di `rhs` ≠ `x0`**: la funzione deve tornare un array della *stessa lunghezza* di `x0`.
3. **Dimenticare il `lambda`** quando `rhs` ha più di 2 argomenti.


In [ ]:
# Simulazione end-to-end del modello di membrana RC con il pattern completo
from scipy.integrate import solve_ivp

# Parametri come dizionario
p = {'Cm': 1.0, 'gL': 0.3, 'EL': -65.0}

def rhs(t, x, p):
    V = x[0]
    I_ext = 5.0 if t >= 2.0 else 0.0
    dV = (I_ext - p['gL']*(V - p['EL'])) / p['Cm']
    return np.array([dV])

# Ricetta standard
x0 = np.array([-65.0])
t_span = (0.0, 25.0)
t_eval = np.linspace(*t_span, 600)

sol = solve_ivp(lambda t, x: rhs(t, x, p), t_span, x0,
                t_eval=t_eval, rtol=1e-8, atol=1e-10)

print(f"success={sol.success}, n_steps={len(sol.t)}")

# Plot
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(sol.t, sol.y[0], color='#1f4e79', lw=2)
ax.axvline(2, color='gray', ls=':', label='inizio stimolo')
ax.set_xlabel('t [ms]'); ax.set_ylabel('V [mV]')
ax.set_title('Ricetta 3: simulazione end-to-end di una membrana RC')
ax.legend(); plt.tight_layout(); plt.show()


## Ricetta 4 — Studio di sensitività con un `for` sui parametri

"Cosa succede se raddoppio $g_L$?" Si fa con un loop:

```python
for gL_val in [0.1, 0.3, 0.6, 1.0]:
    p['gL'] = gL_val
    sol = solve_ivp(lambda t, x: rhs(t, x, p), t_span, x0, t_eval=t_eval)
    ax.plot(sol.t, sol.y[0], label=f"gL={gL_val}")
```

**Importante**: dentro il loop, i parametri sono *catturati per riferimento*
dal lambda. Per evitare bug, ricrea il lambda dentro il loop (come sopra)
oppure passa i parametri *espliciti* via `args=` (vedi sotto).


In [ ]:
# Sensitivita' al variare di gL: pattern canonico
def rhs2(t, x, gL, EL=-65.0, Cm=1.0):
    V = x[0]
    I_ext = 5.0 if t >= 2.0 else 0.0
    return np.array([(I_ext - gL*(V - EL)) / Cm])

x0 = np.array([-65.0])
t_eval = np.linspace(0, 25, 600)

fig, ax = plt.subplots(figsize=(10, 5))
for gL, c in zip([0.1, 0.3, 0.6, 1.0],
                  ['#1f4e79', '#1e8449', '#d68910', '#c0392b']):
    sol = solve_ivp(lambda t, x: rhs2(t, x, gL), (0, 25), x0,
                    t_eval=t_eval, rtol=1e-8, atol=1e-10)
    V_inf = -65 + 5.0/gL                         # asintoto analitico
    tau   = 1.0/gL                                # tempo caratteristico
    ax.plot(sol.t, sol.y[0], color=c, lw=2,
            label=f"gL={gL}, tau={tau:.2f} ms, V_inf={V_inf:.1f} mV")
ax.set_xlabel('t [ms]'); ax.set_ylabel('V [mV]')
ax.set_title("Ricetta 4: sensitivita' al variare di $g_L$")
ax.legend(); plt.tight_layout(); plt.show()


## Ricetta 5 — Ingressi dipendenti dal tempo

Tre forme tipiche:

```python
# A) Gradino: 0 prima di t_on, A dopo
def step(t, t_on=2.0, A=10.0):
    return A if t >= t_on else 0.0

# B) Impulso rettangolare: A tra t_on e t_off, 0 altrove
def pulse(t, t_on=2.0, t_off=2.5, A=10.0):
    return A if t_on <= t <= t_off else 0.0

# C) Sinusoide: utile per studi in frequenza
def sinus(t, f_Hz=1.0, A=1.0): return A * np.sin(2*np.pi*f_Hz*t)
```

Puoi anche fare un **treno di impulsi** sommando più pulse con un loop.

**Trucco**: per studi più seri (eventi precisi), usa l'argomento
`events=` di `solve_ivp` con una funzione che cambia segno
all'istante di interesse — l'integratore si fermerà esattamente lì.


In [ ]:
# Ingressi tipo gradino, impulso, sinusoide -- esempi visivi
fig, axes = plt.subplots(3, 1, figsize=(10, 6.5), sharex=True)

t = np.linspace(0, 10, 1000)

# Gradino
axes[0].plot(t, np.array([10.0 if ti >= 2 else 0 for ti in t]),
             color='#1f4e79', lw=2)
axes[0].set_ylabel('I [µA]'); axes[0].set_title('Gradino')

# Impulso rettangolare
def pulse(ti, t_on, t_off, A): return A if t_on <= ti <= t_off else 0
axes[1].plot(t, [pulse(ti, 2, 2.5, 10) for ti in t],
             color='#c0392b', lw=2)
axes[1].set_ylabel('I [µA]'); axes[1].set_title('Impulso 2.0-2.5 ms')

# Treno di impulsi
def train(ti, period=2.0, dur=0.3, A=10.0):
    return A if (ti % period) <= dur else 0
axes[2].plot(t, [train(ti) for ti in t], color='#1e8449', lw=2)
axes[2].set_ylabel('I [µA]'); axes[2].set_xlabel('t [ms]')
axes[2].set_title('Treno di impulsi (period=2 ms, dur=0.3 ms)')

plt.tight_layout(); plt.show()


## Ricetta 6 — Calcolo dell'equilibrio numerico

Un **equilibrio** è uno $\mathbf{x}^*$ tale che $\mathbf{f}(\mathbf{x}^*)=\mathbf{0}$.
Si trova con `scipy.optimize.root`:

```python
from scipy.optimize import root

def F(x):  # ritorna l'rhs valutato in x (senza dipendenza dal tempo per LTI)
    return rhs(0.0, x, p)

sol_eq = root(F, x0=np.array([0.0, 0.0]), method='hybr')
print("Equilibrio:", sol_eq.x, " success:", sol_eq.success)
```

**Suggerimenti**:
- *Buon `x0`*: meglio partire vicino a una stima ragionevole (per il
  cardiovascolare: usa pressioni fisiologiche; per HH: $V_0=-65$).
- *Metodi*: `'hybr'` (default, Powell), `'lm'` (Levenberg–Marquardt) sono
  i più robusti per modelli ben condizionati.
- *Soluzioni multiple*: il sistema può avere più equilibri (bistabilità).
  Per trovarli tutti, riavvia da `x0` differenti.


In [ ]:
# Trova equilibri di un sistema bistabile: dx/dt = x - x^3
from scipy.optimize import root

def F(x): return np.array([x[0] - x[0]**3])

equilibria = []
for x0 in [-1.5, 0.5, 1.5]:
    res = root(F, x0=np.array([x0]))
    if res.success:
        # Stabilita' locale: J = 1 - 3 x^2
        J = 1 - 3*res.x[0]**2
        equilibria.append((res.x[0], J))
        print(f"  x0={x0:+.2f}  ->  x*={res.x[0]:+.4f}  J(x*)={J:+.3f}  "
              f"{'stabile' if J<0 else 'instabile'}")

# Visualizziamo
x = np.linspace(-1.6, 1.6, 400)
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(x, x - x**3, color='#1f4e79', lw=2, label='$f(x)=x-x^3$')
ax.axhline(0, color='gray', lw=0.7)
for xe, J in equilibria:
    c = '#1e8449' if J<0 else '#c0392b'
    ax.plot(xe, 0, 'o', ms=12, color='white', mec=c, mew=2.5,
            label=f"x*={xe:+.2f} ({'stabile' if J<0 else 'instabile'})")
ax.set_xlabel('x'); ax.set_ylabel("$\\dot x$")
ax.legend()
ax.set_title('Ricetta 6: equilibri trovati con scipy.optimize.root')
plt.tight_layout(); plt.show()


## Ricetta 7 — Jacobiano numerico + stabilità via autovalori

In modellistica **rarissimamente** si calcola il Jacobiano a mano: si fa
con differenze finite centrate e poi si guardano gli **autovalori**:

```python
def jacobian_fd(f, x, eps=1e-6):
    n = len(x)
    J = np.zeros((n, n))
    for j in range(n):
        ej = np.zeros(n); ej[j] = eps
        J[:, j] = (f(x + ej) - f(x - ej)) / (2 * eps)
    return J

J = jacobian_fd(lambda x: rhs(0.0, x, p), x_eq)
vals = np.linalg.eigvals(J)
print("Autovalori:", vals)
print("Stabile?", all(v.real < 0 for v in vals))
```

**Decisione di stabilità** (versione iperbolica):
- $\Re(\lambda_i) < 0$ per tutti gli $i$ → **asintoticamente stabile**;
- esiste $\Re(\lambda_i) > 0$ → **instabile**;
- esiste $\Re(\lambda_i) = 0$ → **non iperbolico**: serve un'analisi non
  lineare (vedi Hartman–Grobman).

Per sistemi 2D, è anche utile guardare:
- $\operatorname{tr} J = \sum \lambda_i$,
- $\det J = \prod \lambda_i$,

e applicare la classificazione trace–determinant del Parte 4.


In [ ]:
# Calcolo del Jacobiano numerico per un esempio non lineare
def jacobian_fd(f, x, eps=1e-6):
    n = len(x)
    J = np.zeros((n, n))
    for j in range(n):
        ej = np.zeros(n); ej[j] = eps
        J[:, j] = (f(x + ej) - f(x - ej)) / (2 * eps)
    return J

# Sistema preda-predatore (Lotka-Volterra)
def lv_rhs(x, a=1.0, b=0.5, c=0.75, d=0.25):
    return np.array([a*x[0] - b*x[0]*x[1],
                     -c*x[1] + d*x[0]*x[1]])

# Equilibrio non banale (c/d, a/b)
x_eq = np.array([0.75/0.25, 1.0/0.5])
print(f"Equilibrio: {x_eq}")
print(f"f(x_eq):    {lv_rhs(x_eq)}  (deve essere ~0)")

J = jacobian_fd(lv_rhs, x_eq)
print(f"\nJacobiano:\n{J}")

vals = np.linalg.eigvals(J)
print(f"\nAutovalori: {vals}")
print(f"tr(J)  = {J.trace():+.4f}")
print(f"det(J) = {np.linalg.det(J):+.4f}")
print(f"-> tr=0, det>0 -> CENTRO (autovalori immaginari puri, non iperbolico)")


## Ricetta 8 — Phase plane per sistemi 2D

In 2D si visualizzano: (a) **campo vettoriale**; (b) **nullcline**
($\dot x=0$ e $\dot y=0$); (c) **traiettorie**; (d) **equilibri**.

Pattern minimale:

```python
def f(t, z): x, y = z; return np.array([..., ...])

# Griglia
X, Y = np.meshgrid(np.linspace(*xlim, 20), np.linspace(*ylim, 20))
DX = np.zeros_like(X); DY = np.zeros_like(Y)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        DX[i,j], DY[i,j] = f(0, np.array([X[i,j], Y[i,j]]))

# Plot
fig, ax = plt.subplots()
M = np.hypot(DX, DY); M[M==0] = 1.0
ax.quiver(X, Y, DX/M, DY/M, M, cmap='Blues', pivot='mid', scale=25)

# Traiettorie
for x0 in [[1,0.5], [2,1.5], [4,1]]:
    sol = solve_ivp(f, (0,30), x0, dense_output=True, max_step=0.02)
    ax.plot(sol.y[0], sol.y[1], 'b', lw=1.5)
```

Per le **nullcline** si risolve $f_1(x,y)=0$ e $f_2(x,y)=0$
analiticamente, o si traccia il contorno `ax.contour(X, Y, DX, [0])`.


In [ ]:
# Phase plane completo con nullcline per un sistema 2D
def f(t, z):
    x, y = z
    return np.array([x*(1 - x) - 0.5*x*y,
                     -0.3*y + 0.4*x*y])

# Griglia per campo vettoriale e contour
X, Y = np.meshgrid(np.linspace(0, 2.5, 20), np.linspace(0, 2.5, 20))
DX = np.zeros_like(X); DY = np.zeros_like(Y)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        DX[i,j], DY[i,j] = f(0, np.array([X[i,j], Y[i,j]]))

# Griglia piu' fitta per le nullcline
Xf, Yf = np.meshgrid(np.linspace(0, 2.5, 200), np.linspace(0, 2.5, 200))
DXf = np.zeros_like(Xf); DYf = np.zeros_like(Yf)
for i in range(Xf.shape[0]):
    for j in range(Xf.shape[1]):
        DXf[i,j], DYf[i,j] = f(0, np.array([Xf[i,j], Yf[i,j]]))

fig, ax = plt.subplots(figsize=(8.5, 7))
M = np.hypot(DX, DY); M[M==0] = 1.0
ax.quiver(X, Y, DX/M, DY/M, M, cmap='Blues', pivot='mid', scale=25, alpha=0.7)
ax.contour(Xf, Yf, DXf, levels=[0], colors='#1f4e79', linewidths=1.5)   # nullcline x
ax.contour(Xf, Yf, DYf, levels=[0], colors='#c0392b', linewidths=1.5)   # nullcline y

# Traiettorie
for x0 in [[0.3, 0.3], [2.0, 0.3], [0.3, 2.0], [1.5, 1.0]]:
    sol = solve_ivp(f, (0, 40), x0, dense_output=True, max_step=0.02,
                    rtol=1e-8, atol=1e-10)
    ax.plot(sol.y[0], sol.y[1], color='black', lw=1.2)

ax.set_xlim(0, 2.5); ax.set_ylim(0, 2.5)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Ricetta 8: phase plane (quiver + nullcline blu/rossa + traiettorie nere)')
plt.tight_layout(); plt.show()


## Ricetta 9 — Bifurcation diagram numerico

Pattern: per ogni valore del parametro $r$ in una griglia, **simula a regime**
o **trova gli equilibri** e ne riporta l'ampiezza/posizione. Il risultato è
un diagramma di biforcazione "empirico".

Due varianti:

**A) Equilibri al variare di $r$** (per modelli 1D con `root`):

```python
r_vals = np.linspace(-1, 1, 50)
x_eq_branch = []
for r in r_vals:
    res = root(lambda x: r*x - x**3, x0=0.5)
    x_eq_branch.append(res.x[0])
```

**B) Ampiezza di oscillazioni al variare di $r$** (per cicli limite/Hopf):

```python
for r in r_vals:
    sol = solve_ivp(lambda t, x: rhs(t, x, r), (0, 300), x0, ...)
    # ampiezza misurata negli ultimi 30 secondi (a regime)
    x_late = sol.y[0, sol.t > 270]
    amplitudes.append((x_late.max() - x_late.min())/2)
```


In [ ]:
# Bifurcation diagram di una biforcazione pitchfork supercritica
# dx/dt = r x - x^3:  per r<0 un unico equilibrio stabile (x=0),
#                     per r>0 nasce un paio +-sqrt(r) stabile e (x=0) diventa instabile.

from scipy.optimize import root

r_vals = np.linspace(-1, 1, 100)
upper, lower, middle = [], [], []
for r in r_vals:
    # Sotto r=0 c'e' solo un equilibrio; sopra r=0 ce ne sono 3
    for x0_guess, lst in [(0.0, middle), (np.sqrt(max(r, 0.01)), upper),
                           (-np.sqrt(max(r, 0.01)), lower)]:
        res = root(lambda x: r*x[0] - x[0]**3, x0=np.array([x0_guess]))
        lst.append(res.x[0] if res.success else np.nan)

fig, ax = plt.subplots(figsize=(9, 5))
# Branch x=0: stabile per r<0, instabile per r>0
mid = np.array(middle)
ax.plot(r_vals[r_vals < 0], np.zeros((r_vals < 0).sum()),
        color='#1f4e79', lw=2.2, label='x*=0 stabile')
ax.plot(r_vals[r_vals >= 0], np.zeros((r_vals >= 0).sum()),
        color='#c0392b', lw=2.2, ls='--', label='x*=0 instabile')
ax.plot(r_vals[r_vals > 0], np.sqrt(r_vals[r_vals > 0]),
        color='#1e8449', lw=2.2, label='x*=±√r stabile')
ax.plot(r_vals[r_vals > 0], -np.sqrt(r_vals[r_vals > 0]),
        color='#1e8449', lw=2.2)
ax.axvline(0, color='gray', lw=0.7)
ax.axhline(0, color='gray', lw=0.7)
ax.set_xlabel('parametro r'); ax.set_ylabel('equilibrio x*')
ax.set_title('Ricetta 9: bifurcation diagram (pitchfork supercritica)')
ax.legend()
plt.tight_layout(); plt.show()


## Ricetta 10 — Feedback con uno stato di controllo lento

In molti modelli fisiologici il controllo *non* è istantaneo: la
"vasocostrizione" cambia in 5-10 s, l'aldosterone in ore. Pattern: aggiungi
una **variabile di stato in più** che insegue un *setpoint* dipendente da
qualche misura.

```python
def rhs(t, x, p):
    V, R = x                          # V = uscita, R = stato di controllo lento
    R_target = setpoint(V, p)          # quanto vuoi che valga
    dV = (...usa R...)                 # plant
    dR = (R_target - R) / p['tau_R']   # filtro passa-basso del primo ordine
    return np.array([dV, dR])
```

Questa è la struttura esatta di tutti i controlli del notebook: baroriflesso
(7.2), controllo ventilatorio (7.7), regolazione glicemica.

**Constante di tempo**: `tau_R` grande → controllo lento (può oscillare
in feedback chiuso); `tau_R` piccolo → controllo veloce (ma rumoroso se
ci sono misure imperfette).


In [ ]:
# Esempio minimale: una "pressione" V regolata da un controllore R(t) lento
def rhs(t, x, p):
    V, R = x
    # plant: dV/dt = -V/tau_V + R + ingresso esterno
    I_ext = p['I_ext'](t)
    dV = (-V/p['tau_V'] + R + I_ext)
    # controllore lento: R cerca di portare V al setpoint
    R_target = p['Gain'] * (p['setpoint'] - V)
    dR = (R_target - R) / p['tau_R']
    return np.array([dV, dR])

p = {'tau_V': 1.0, 'tau_R': 10.0,
     'Gain': 0.5, 'setpoint': 0.0,
     'I_ext': lambda t: -2.0 if 5 <= t <= 25 else 0.0}

x0 = np.array([0.0, 0.0])
sol = solve_ivp(lambda t, x: rhs(t, x, p), (0, 60), x0,
                dense_output=True, max_step=0.05, rtol=1e-8, atol=1e-10)

fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
axes[0].plot(sol.t, sol.y[0], color='#1f4e79', lw=2, label='uscita V(t)')
axes[0].axhline(p['setpoint'], color='gray', ls=':', label='setpoint')
axes[0].axvspan(5, 25, color='red', alpha=0.08, label='disturbo I_ext')
axes[0].set_ylabel('V'); axes[0].legend()
axes[0].set_title('Ricetta 10: il controllore R(t) compensa il disturbo')

axes[1].plot(sol.t, sol.y[1], color='#1e8449', lw=2)
axes[1].set_xlabel('t'); axes[1].set_ylabel('controllore R(t)')
plt.tight_layout(); plt.show()


## Ricetta 11 — Ritardi discreti con buffer circolare

Quando il modello ha un **ritardo** $\tau_d$ (es. tempo di transito del
sangue dai polmoni ai chemocettori), in teoria si dovrebbe usare una DDE
(Delay Differential Equation). In pratica didattica si fa così:

1. Discretizza il tempo con passo `dt`;
2. Mantieni un **buffer** delle ultime $N = \tau_d / dt$ misure;
3. Quando ti serve il valore *ritardato*, leggi `buffer[k - N]`.

Pattern:

```python
N = int(tau_d / dt)
buffer = np.full(N_total, x0)        # array dei valori storici
for k in range(N_total - 1):
    x_now = buffer[k]
    x_delayed = buffer[max(0, k - N)]   # valore ritardato
    # ...aggiorna lo stato...
    buffer[k+1] = x_new
```

È l'idea dietro al modello di Cheyne–Stokes nel Parte 7.7.


In [ ]:
# Esempio: x(t) inseguendo un setpoint con misura RITARDATA -> instabilita'
def simulate_delay(tau_d, T=40, dt=0.05, setpoint=1.0, gain=2.0):
    N_total = int(T/dt) + 1
    N_delay = max(1, int(tau_d/dt))
    t = np.linspace(0, T, N_total)
    x = np.zeros(N_total)
    u = np.zeros(N_total)
    for k in range(N_total - 1):
        # leggo x con ritardo
        x_meas = x[max(0, k - N_delay)]
        u[k] = gain * (setpoint - x_meas)
        # plant del primo ordine: dx/dt = u - x
        x[k+1] = x[k] + dt * (u[k] - x[k])
    return t, x, u

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for tau, c in zip([0.1, 0.5, 1.5, 3.0],
                   ['#1e8449', '#1f4e79', '#d68910', '#c0392b']):
    t, x, u = simulate_delay(tau)
    axes[0].plot(t, x, color=c, lw=1.6, label=f"$\\tau_d={tau}$")
    axes[1].plot(t, u, color=c, lw=1.6)
axes[0].axhline(1.0, color='gray', ls=':')
axes[0].set_ylabel('x(t)'); axes[0].legend()
axes[0].set_title("Ricetta 11: controllo con ritardo -> Hopf-instabilita")
axes[1].set_xlabel('t'); axes[1].set_ylabel('u(t) = controllo')
plt.tight_layout(); plt.show()


## Ricetta 12 — Calibrazione di parametri (fitting)

Hai dati sperimentali e vuoi stimare $\boldsymbol{\theta}$? Pattern:

```python
from scipy.optimize import curve_fit, least_squares, minimize

# 1) funzione modello: parametri -> uscita simulata sui tempi dei dati
def model_output(t_data, *theta):
    p = {'k1': theta[0], 'k2': theta[1]}
    sol = solve_ivp(lambda t, x: rhs(t, x, p), (0, t_data[-1]), x0,
                    t_eval=t_data, rtol=1e-8, atol=1e-10)
    return sol.y[0]   # variabile osservata

# 2) curve_fit minimizza |model_output(t_data, *theta) - y_data|^2
popt, pcov = curve_fit(model_output, t_data, y_data,
                       p0=[1.0, 1.0])

# 3) intervalli di confidenza approssimati dalla covarianza
perr = np.sqrt(np.diag(pcov))
print(f"k1 = {popt[0]:.3f} +- {perr[0]:.3f}")
```

**Suggerimenti operativi**:
- *normalizza i parametri* se hanno scale molto diverse (es. uno è 1e-6,
  l'altro è 1e3);
- *usa `least_squares` con `bounds=`* se sai i parametri devono essere positivi;
- *ricorda l'identificabilità*: se $\sigma(\theta_1)$ è enorme, il dato
  non basta a fissare $\theta_1$ (vedi Esercizio G);
- per modelli stiff, passa `method='BDF'` dentro `solve_ivp`.


In [ ]:
# Esempio: stima di k da dati di decadimento esponenziale rumorosi
from scipy.optimize import curve_fit

# Genero dati "veri"
np.random.seed(0)
t_data = np.linspace(0, 5, 30)
k_true, A_true = 0.7, 3.0
y_clean = A_true * np.exp(-k_true * t_data)
y_data  = y_clean + 0.15 * np.random.randn(len(t_data))

# Modello a 2 parametri
def model(t, A, k): return A * np.exp(-k*t)

popt, pcov = curve_fit(model, t_data, y_data, p0=[1.0, 1.0])
A_hat, k_hat = popt
A_err, k_err = np.sqrt(np.diag(pcov))

print(f"VERO:  A={A_true:.3f}, k={k_true:.3f}")
print(f"STIMA: A={A_hat:.3f} +- {A_err:.3f}")
print(f"       k={k_hat:.3f} +- {k_err:.3f}")

# Plot
t_fine = np.linspace(0, 5, 200)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(t_data, y_data, 'o', color='#c0392b', label='dati rumorosi')
ax.plot(t_fine, A_true*np.exp(-k_true*t_fine), color='black', ls=':',
        label=f'vero: A={A_true}, k={k_true}')
ax.plot(t_fine, A_hat*np.exp(-k_hat*t_fine), color='#1f4e79', lw=2,
        label=f'fit: A={A_hat:.2f}, k={k_hat:.2f}')
ax.set_xlabel('t'); ax.set_ylabel('y')
ax.set_title('Ricetta 12: calibrazione di 2 parametri con curve_fit')
ax.legend()
plt.tight_layout(); plt.show()


## Riepilogo: il "**workflow di un modellista**"

Mettendo insieme le 12 ricette, un esercizio tipico segue questa
sequenza, *quasi sempre identica*:

```
1) definisci `rhs(t, x, p)`                              [Ricetta 1, 2]
2) definisci parametri `p` (dizionario)                  [Ricetta 2]
3) definisci ingressi tempo-varianti `p['I_ext'] = ...`  [Ricetta 5]
4) scegli condizione iniziale `x0`                       [Ricetta 3]
5) simula con solve_ivp                                  [Ricetta 3]
6) (opz) per molti parametri: loop                       [Ricetta 4]
7) trova equilibri con root                              [Ricetta 6]
8) calcola Jacobiano + autovalori per stabilità          [Ricetta 7]
9) (2D) phase plane + nullcline                          [Ricetta 8]
10) (parametri) bifurcation diagram                      [Ricetta 9]
11) (controlli) stato lento di feedback                  [Ricetta 10]
12) (ritardi) buffer circolare                           [Ricetta 11]
13) (dati) curve_fit o least_squares                     [Ricetta 12]
14) plot finale con annotazioni
```

Tutto il notebook usa **solo** queste mosse. Adesso che le hai viste
tutte in fila, ogni modello successivo è solo "un riempire i punti".


# Parte 1 — Recap matematico totale

In questa parte costruiamo, **da zero**, tutti gli oggetti matematici che
useremo dopo per i modelli fisiologici:

- scalari, vettori, matrici, norme;
- autovalori e autovettori (cuore della stabilità lineare);
- esponenziale di matrice $e^{At}$ (soluzione esatta dei sistemi LTI);
- equazioni differenziali del primo ordine, scalari e vettoriali;
- linearizzazione: serie di Taylor, Jacobiano, teorema di Hartman–Grobman;
- trasformata di Laplace, funzioni di trasferimento, poli/zeri;
- metodi numerici: Euler esplicito, Euler implicito, Runge–Kutta;
- analisi dimensionale e nondimensionalizzazione.

Ognuno di questi oggetti viene usato esplicitamente nelle parti successive,
quindi vale la pena costruirseli "in pancia" qui.


## 1.1 Scalari, vettori, matrici — il minimo indispensabile

Uno **scalare** è un singolo numero reale: una pressione $p_{sa}=100\,\text{mmHg}$
o un volume $V=70\,\text{mL}$.

Un **vettore** è una colonna ordinata di scalari che, in modellistica,
rappresenta lo *stato* del sistema:

$$
\mathbf{X}(t)=\begin{bmatrix} x_1(t)\\ x_2(t)\\ \vdots\\ x_n(t)\end{bmatrix}\in\mathbb{R}^n .
$$

Una **matrice** $A\in\mathbb{R}^{m\times n}$ è una tabella rettangolare di
scalari; in modellistica codifica *quanto ogni variabile influenza la
derivata di ogni altra*:

$$
A=\begin{bmatrix}
a_{11}&a_{12}&\cdots&a_{1n}\\
a_{21}&a_{22}&\cdots&a_{2n}\\
\vdots & & \ddots & \vdots\\
a_{m1}&a_{m2}&\cdots&a_{mn}
\end{bmatrix},\qquad a_{ij}=[A]_{ij} .
$$

### Prodotto matrice–vettore "esploso"

L'operazione fondamentale che incontri ovunque è

$$
\mathbf{y}=A\mathbf{x},\qquad
\begin{bmatrix} y_1\\ y_2\\ y_3\end{bmatrix}
=
\begin{bmatrix}
a_{11}&a_{12}&a_{13}\\
a_{21}&a_{22}&a_{23}\\
a_{31}&a_{32}&a_{33}
\end{bmatrix}
\begin{bmatrix} x_1\\ x_2\\ x_3\end{bmatrix}
=
\begin{bmatrix}
a_{11}x_1+a_{12}x_2+a_{13}x_3\\
a_{21}x_1+a_{22}x_2+a_{23}x_3\\
a_{31}x_1+a_{32}x_2+a_{33}x_3
\end{bmatrix}.
$$

**Non è magia**: ogni componente di $\mathbf{y}$ è una *combinazione lineare* delle
componenti di $\mathbf{x}$ con pesi presi dalla riga corrispondente di $A$.
Tutta la teoria dei sistemi lineari sta in questa frase.

### Riga vs colonna: due letture utili

- **Per righe**: la riga $i$-esima di $A$ dice *come la variabile $i$ è
  influenzata* da tutte le altre.
- **Per colonne**: la colonna $j$-esima di $A$ dice *quanto la variabile
  $j$ "spinge" tutte le altre*. Se $\mathbf{e}_j$ è il versore con $1$
  nella posizione $j$, allora $A\mathbf{e}_j$ è proprio la colonna $j$ di $A$.

Tenere a mente entrambe le letture aiuta moltissimo quando, ad esempio,
si guarda una matrice di Jacobiano e si vuole capire chi inibisce chi.


In [ ]:
# Prodotto matrice-vettore: numerico ed "esploso" simbolicamente
A = np.array([[ 2.0, -1.0,  0.5],
              [ 0.0,  3.0, -2.0],
              [-1.0,  1.0,  4.0]])
x = np.array([1.0, 2.0, -1.0])

y = A @ x
print("A =")
print(A)
print("x =", x)
print("y = A x =", y)

# Versione "esplosa" riga per riga - utile per capire cosa succede
print("\nForma esplosa:")
for i, row in enumerate(A):
    pezzi = [f"({row[j]:+.2f})*x{j+1}" for j in range(len(x))]
    print(f"  y{i+1} = " + " + ".join(pezzi)
          + f" = {sum(row[j]*x[j] for j in range(len(x))):+.4f}")


## 1.2 Norme e geometria dei vettori

Una **norma** misura la "lunghezza" di un vettore. Le più usate sono:

- **Norma 2 (euclidea)**: $\|\mathbf{x}\|_2=\sqrt{\sum_i x_i^2}$;
- **Norma 1**: $\|\mathbf{x}\|_1=\sum_i |x_i|$;
- **Norma infinito**: $\|\mathbf{x}\|_\infty=\max_i |x_i|$.

In modellistica usiamo le norme per:

1. misurare quanto due simulazioni differiscono (errore numerico);
2. studiare la stabilità ("se la norma cresce, la traiettoria esplode");
3. confrontare l'effetto di perturbazioni.

Il **prodotto scalare** $\mathbf{x}\cdot\mathbf{y}=\sum_i x_i y_i$ permette di
definire l'**angolo** tra vettori:

$$
\cos\theta=\frac{\mathbf{x}\cdot\mathbf{y}}{\|\mathbf{x}\|_2\,\|\mathbf{y}\|_2}.
$$

Due vettori sono **ortogonali** se $\mathbf{x}\cdot\mathbf{y}=0$. Questa
relazione è fondamentale per la decomposizione spettrale (autovettori
ortogonali per matrici simmetriche) che useremo nei modelli respiratori
e cardiovascolari linearizzati.


In [ ]:
# Norme di un vettore e angoli
x = np.array([3.0, 4.0, 0.0])
y = np.array([1.0, 0.0, 2.0])

print(f"x      = {x}")
print(f"||x||_2 = {np.linalg.norm(x, 2):.4f}")
print(f"||x||_1 = {np.linalg.norm(x, 1):.4f}")
print(f"||x||_inf = {np.linalg.norm(x, np.inf):.4f}")
print(f"x . y  = {x @ y:.4f}")
cos_theta = (x @ y) / (np.linalg.norm(x) * np.linalg.norm(y))
print(f"cos(angolo) = {cos_theta:.4f}  ->  angolo = {np.degrees(np.arccos(cos_theta)):.2f} deg")


## 1.3 Autovalori e autovettori: il cuore della stabilità

Una matrice quadrata $A\in\mathbb{R}^{n\times n}$ può, in alcune direzioni
speciali $\mathbf{v}$, comportarsi come uno **scalare** $\lambda$:

$$
A\mathbf{v}=\lambda \mathbf{v},\qquad \mathbf{v}\ne\mathbf{0}.
$$

- $\lambda$ è un **autovalore**: lo scalare di "stretching/contrazione" lungo $\mathbf{v}$;
- $\mathbf{v}$ è il corrispondente **autovettore**: la direzione invariante.

Geometricamente: applicare $A$ a $\mathbf{v}$ non lo *ruota*, lo *scala*.

### Come si trovano

Riscriviamo l'equazione come $(A-\lambda I)\mathbf{v}=\mathbf{0}$. Affinché
esista $\mathbf{v}\ne\mathbf{0}$ in soluzione, la matrice $A-\lambda I$
deve essere **singolare**:

$$
\det(A-\lambda I)=0.
$$

Per $A$ generica $n\times n$ questa equazione si chiama **equazione
caratteristica** ed è un polinomio di grado $n$ in $\lambda$.

### Esempio 2×2 esplicito

$$
A=\begin{bmatrix} a&b\\ c&d\end{bmatrix},\qquad
\det(A-\lambda I)=(a-\lambda)(d-\lambda)-bc
=\lambda^2 - \underbrace{(a+d)}_{=\operatorname{tr}A}\lambda + \underbrace{(ad-bc)}_{=\det A}.
$$

Quindi gli autovalori 2×2 si calcolano con la formula della quadratica:

$$
\boxed{\;\lambda_{1,2}=\frac{\operatorname{tr}A \pm \sqrt{(\operatorname{tr}A)^2-4\det A}}{2}\;}
$$

**Regole d'oro**:

- $\operatorname{tr}A=\lambda_1+\lambda_2$,
- $\det A=\lambda_1\,\lambda_2$,
- se $(\operatorname{tr}A)^2<4\det A$ gli autovalori sono **complessi coniugati**
  → oscillazioni;
- se $\operatorname{tr}A<0$ e $\det A>0$: equilibrio **stabile**.

Useremo questa classificazione costantemente nelle sezioni successive.


In [ ]:
# Esempio 2x2: calcolo manuale e con numpy
A = np.array([[ 0.0,  1.0],
              [-2.0, -3.0]])

tr  = A.trace()
det = np.linalg.det(A)
disc = tr**2 - 4*det

print(f"tr(A) = {tr}")
print(f"det(A) = {det}")
print(f"discriminante = {disc}")

lam1 = (tr + np.sqrt(disc + 0j)) / 2
lam2 = (tr - np.sqrt(disc + 0j)) / 2
print(f"\nAutovalori a mano: lambda1 = {lam1}, lambda2 = {lam2}")

vals, vecs = np.linalg.eig(A)
print("\nAutovalori (numpy):", vals)
print("Autovettori (colonne):")
print(vecs)

# Verifica: A @ v_i ?= lambda_i * v_i
for i, l in enumerate(vals):
    v = vecs[:, i]
    print(f"  A v{i+1} = {A @ v},  l{i+1} v{i+1} = {l * v}")


### Esempio 3×3: calcolo del polinomio caratteristico

Per $A\in\mathbb{R}^{3\times 3}$ l'equazione caratteristica è un polinomio
cubico:

$$
\det(A-\lambda I)=-\lambda^3 + (\operatorname{tr}A)\lambda^2
 -\Big(\sum_{i<j}M_{ij}\Big)\lambda + \det A,
$$

dove $M_{ij}$ sono i minori principali $2\times 2$. Le radici sono tre — reali
o una reale + una coppia complessa coniugata. Il **segno della parte reale**
di ciascuna determina la stabilità della corrispondente componente di stato.


In [ ]:
# Esempio 3x3
A = np.array([
    [-1.0,  0.2,  0.0],
    [ 0.3, -2.0,  1.5],
    [ 0.0,  0.4, -0.8],
])

vals, vecs = np.linalg.eig(A)
print("Autovalori:", vals)
print("Parti reali:", vals.real)
print("Parti immag.:", vals.imag)
print("\nStabile? (Re(lambda) < 0 per tutti):",
      np.all(vals.real < 0))

# Polinomio caratteristico esplicito
p = np.poly(A)  # coefficienti a partire da lambda^n
print("\nCoefficienti polinomio caratteristico (alto -> basso grado):", p)
print("Radici del polinomio (devono coincidere):", np.roots(p))


### Diagonalizzazione

Se $A$ ha $n$ autovettori linearmente indipendenti, raccolti come colonne
in $V$, e $\Lambda=\operatorname{diag}(\lambda_1,\dots,\lambda_n)$, allora

$$
A V = V\Lambda\quad\Longrightarrow\quad A = V\Lambda V^{-1}.
$$

Questa fattorizzazione disaccoppia il sistema: nella base degli autovettori
ciascuna componente evolve **indipendentemente** come $\dot z_i=\lambda_i z_i$.

**Casi non diagonalizzabili**: quando un autovalore ha molteplicità algebrica
$>1$ ma molteplicità geometrica più piccola (cioè non ci sono abbastanza
autovettori indipendenti). Si parla di **autovalori difettivi**. In questo
caso esiste la **forma di Jordan**:

$$
A=V\,J\,V^{-1},\qquad J=\begin{bmatrix}\lambda&1&0\\0&\lambda&1\\0&0&\lambda\end{bmatrix}
$$

con dei "1" sopra la diagonale. La conseguenza dinamica più importante:
in presenza di blocchi di Jordan compaiono termini del tipo $t^{k}e^{\lambda t}$
nella soluzione.


In [ ]:
# Diagonalizzazione e verifica numerica
A = np.array([[ 2.0, 1.0],
              [ 0.0, 3.0]])
vals, V = np.linalg.eig(A)
Lambda = np.diag(vals)
Vinv = np.linalg.inv(V)

print("A reconstructed:")
print(V @ Lambda @ Vinv)
print("\nA originale:")
print(A)

# Esempio difettivo: blocco di Jordan 2x2 con autovalore doppio
J = np.array([[3.0, 1.0],
              [0.0, 3.0]])
vals_J, V_J = np.linalg.eig(J)
print("\nJordan 2x2: autovalori =", vals_J)
print("autovettori (le 2 colonne sarebbero quasi paralleli):")
print(V_J)
print("Determinante della matrice degli autovettori (vicino a 0 = difettiva):",
      np.linalg.det(V_J))


## 1.4 Esponenziale di matrice $e^{At}$

Per uno scalare la soluzione di $\dot x=ax$ con $x(0)=x_0$ è $x(t)=e^{at}x_0$.
Per un vettore con $\dot{\mathbf{x}}=A\mathbf{x}$, $\mathbf{x}(0)=\mathbf{x}_0$,
**la stessa formula** funziona:

$$
\boxed{\;\mathbf{x}(t)=e^{At}\,\mathbf{x}_0\;}
$$

dove $e^{At}$ è la **matrice esponenziale**, definita dalla serie

$$
e^{At}=\sum_{k=0}^{\infty}\frac{(At)^k}{k!}=I+At+\frac{(At)^2}{2!}+\cdots
$$

### Derivazione passo-passo

Cerchiamo $\mathbf{x}(t)$ come serie di potenze: $\mathbf{x}(t)=\sum_k \mathbf{c}_k t^k$.
Derivando termine a termine:

$$
\dot{\mathbf{x}}(t)=\sum_k k\,\mathbf{c}_k t^{k-1},
$$

e imponendo $\dot{\mathbf{x}}=A\mathbf{x}$ si ricava
$\mathbf{c}_{k+1}=\frac{1}{k+1}A\mathbf{c}_k$. Con $\mathbf{c}_0=\mathbf{x}_0$
si ottiene $\mathbf{c}_k=\frac{1}{k!}A^k\mathbf{x}_0$, da cui

$$
\mathbf{x}(t)=\Big(\sum_k \tfrac{(At)^k}{k!}\Big)\mathbf{x}_0=e^{At}\mathbf{x}_0.
$$

### Calcolo via diagonalizzazione

Se $A=V\Lambda V^{-1}$ allora $A^k=V\Lambda^k V^{-1}$ e

$$
e^{At}=V\,e^{\Lambda t}\,V^{-1}=V\operatorname{diag}(e^{\lambda_1 t},\dots,e^{\lambda_n t})V^{-1}.
$$

È **la** ragione per cui gli autovalori contano: ciascun $\lambda_i$ controlla
una "modalità" temporale. Se tutti hanno parte reale negativa, $e^{At}\to 0$ e
il sistema è asintoticamente stabile.

In Python lo calcoliamo con `scipy.linalg.expm`, che usa algoritmi
robusti (scaling-and-squaring + approssimazione di Padé) anche per matrici
non diagonalizzabili.


In [ ]:
# Soluzione esatta tramite e^{At} confrontata con solve_ivp
A = np.array([[-1.0,  2.0],
              [-3.0, -4.0]])

x0 = np.array([1.0, 0.0])

t_grid = np.linspace(0, 5, 400)
# Soluzione esatta x(t) = e^{At} x0
X_exact = np.array([expm(A * t) @ x0 for t in t_grid]).T

# Soluzione numerica con solve_ivp (Runge-Kutta 4/5)
sol = solve_ivp(lambda t, x: A @ x, (0, 5), x0,
                t_eval=t_grid, rtol=1e-9, atol=1e-12)
X_num = sol.y

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(t_grid, X_exact[0], lw=2.2, color=COL["main"],   label="$x_1$ esatta (expm)")
ax.plot(t_grid, X_exact[1], lw=2.2, color=COL["accent"], label="$x_2$ esatta (expm)")
ax.plot(t_grid, X_num[0],  "--", color="white", lw=1.0)
ax.plot(t_grid, X_num[1],  "--", color="white", lw=1.0)
ax.plot(t_grid, X_num[0],  ":", color="black", lw=1.2, label="$x_1$ solve_ivp")
ax.plot(t_grid, X_num[1],  ":", color="black", lw=1.2, label="$x_2$ solve_ivp")
ax.set_xlabel("tempo $t$")
ax.set_ylabel("stato")
ax.set_title("Confronto soluzione esatta $e^{At}x_0$ vs integratore numerico")
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

# Errore massimo
print("Errore massimo |expm - solve_ivp| =", np.max(np.abs(X_exact - X_num)))


**Cosa mostra il grafico** — le due soluzioni (esatta via $e^{At}$ e
numerica via Runge–Kutta adattivo) coincidono visivamente. L'errore massimo
è dell'ordine di $10^{-9}$, dominato dalla tolleranza dell'integratore.

**Come si interpreta** — gli autovalori di $A$ in questo esempio sono
complessi coniugati con parte reale negativa, quindi vediamo un'**oscillazione
smorzata**: il sistema converge a zero con una forma a "spirale".

**Cosa cambia se varia $A$** — autovalori a parte reale positiva → divergenza
esponenziale; autovalori reali negativi → decadimento senza oscillazione;
autovalori immaginari puri → oscillazione perfetta (centro).

**Perché è importante** — questa è esattamente la matematica che useremo
nei modelli LTI cardiovascolare e respiratorio: la "forma" della risposta
si legge negli autovalori della matrice di sistema.


### Dimostrazione di convergenza della serie $e^{At}$

La serie $e^{At}=\sum_{k=0}^{\infty}\frac{(At)^k}{k!}$ **converge per
ogni** $A\in\mathbb{R}^{n\times n}$ e per ogni $t\in\mathbb{R}$. Dimostrazione
in 3 righe:

1. Fissata una norma matriciale sub-moltiplicativa $\|\cdot\|$ (es. norma
   operatoriale 2 o norma di Frobenius), vale $\|A^k\|\le \|A\|^k$.
2. Allora
   $$
   \Big\|\sum_{k=N}^{M}\frac{(At)^k}{k!}\Big\| \le \sum_{k=N}^{M}\frac{(\|A\|\,|t|)^k}{k!}.
   $$
3. Il membro a destra è la coda di una serie esponenziale **scalare**, che
   converge → la serie di matrici è **Cauchy** nello spazio (di Banach)
   delle matrici → converge.

Quindi $e^{At}$ è **sempre ben definita**: nessuna ipotesi su
diagonalizzabilità è necessaria.

### Calcolo esplicito di $e^{At}$ per un esempio $2\times 2$ diagonalizzabile

Sia $A=V\Lambda V^{-1}$ con $\Lambda=\operatorname{diag}(\lambda_1,\lambda_2)$.
Allora $A^k=V\Lambda^k V^{-1}$ e

$$
e^{At}=V\,e^{\Lambda t}\,V^{-1}=V\begin{bmatrix}e^{\lambda_1 t}&0\\0&e^{\lambda_2 t}\end{bmatrix}V^{-1}.
$$

**Esempio concreto**: $A=\begin{bmatrix}-1&2\\0&-3\end{bmatrix}$.

- Autovalori: $\det(A-\lambda I)=(\lambda+1)(\lambda+3)=0$ → $\lambda_1=-1, \lambda_2=-3$.
- Autovettori: per $\lambda_1=-1$: $(A+I)v=0 \Rightarrow v_1=(1,0)^T$. Per $\lambda_2=-3$: $(A+3I)v=0 \Rightarrow v_2=(1,-1)^T$.
- $V=\begin{bmatrix}1&1\\0&-1\end{bmatrix}$, $V^{-1}=\begin{bmatrix}1&1\\0&-1\end{bmatrix}$ (in questo esempio coincide!).
- $e^{At}=V\,\operatorname{diag}(e^{-t},e^{-3t})\,V^{-1}=\begin{bmatrix}e^{-t}&e^{-t}-e^{-3t}\\0&e^{-3t}\end{bmatrix}$.

Verifichiamo numericamente con `scipy.linalg.expm`.


In [ ]:
# Verifica numerica della formula esplicita di e^{At}
A = np.array([[-1.0,  2.0],
              [ 0.0, -3.0]])

for t in [0.0, 0.5, 1.0, 2.0]:
    E_num = expm(A * t)
    E_ana = np.array([[np.exp(-t),             np.exp(-t) - np.exp(-3*t)],
                      [0.0,                    np.exp(-3*t)]])
    err = np.max(np.abs(E_num - E_ana))
    print(f"t={t:.1f}:  ||numerico - analitico||_inf = {err:.2e}")

print("\ne^{At} per t=1 (analitico):")
t = 1.0
print(np.array([[np.exp(-t),             np.exp(-t) - np.exp(-3*t)],
                [0.0,                    np.exp(-3*t)]]))


## 1.5 Equazioni differenziali ordinarie (ODE)

Un'**ODE del primo ordine** è una relazione tra una funzione incognita
$x(t)$ e la sua derivata $\dot x(t)$:

$$
\dot x(t)=f\big(t,x(t)\big),\qquad x(t_0)=x_0.
$$

Il dato $x(t_0)=x_0$ è la **condizione iniziale**. Insieme, equazione +
condizione iniziale = **problema di Cauchy**.

### Caso lineare scalare $\dot x = a x + b u(t)$

Per il caso più semplice $\dot x=a x$, $x(0)=x_0$, la soluzione si trova
per separazione di variabili:

$$
\frac{dx}{x}=a\,dt\;\;\Longrightarrow\;\;\ln|x|=at+C\;\;\Longrightarrow\;\;x(t)=x_0 e^{at}.
$$

Per il caso forzato $\dot x = ax + b u(t)$ moltiplichiamo per il **fattore
integrante** $e^{-at}$:

$$
\frac{d}{dt}\big(e^{-at}x\big)=e^{-at}b u(t),
$$

e integrando da $0$ a $t$:

$$
\boxed{\;x(t)=e^{at}x_0+\int_0^t e^{a(t-\tau)}b\,u(\tau)\,d\tau.\;}
$$

Il primo termine è la **risposta libera** (dipende solo da $x_0$). Il secondo
è la **risposta forzata** (convoluzione tra l'ingresso $u$ e la risposta
impulsiva $h(t)=e^{at}b\,\mathbb{1}(t\ge 0)$).

**Generalizzazione vettoriale** (la dimostreremo nella Parte 4):

$$
\mathbf{x}(t)=e^{At}\mathbf{x}_0+\int_0^t e^{A(t-\tau)}B\,\mathbf{u}(\tau)\,d\tau.
$$


In [ ]:
# Decadimento esponenziale: soluzione analitica vs Eulero esplicito vs solve_ivp
a = -0.7
x0 = 4.0
T = 6.0

# Analitica
t_grid = np.linspace(0, T, 400)
x_exact = x0 * np.exp(a * t_grid)

# Eulero esplicito a passo dt
def euler_explicit(a, x0, T, dt):
    n = int(T / dt) + 1
    t = np.linspace(0, T, n)
    x = np.zeros(n); x[0] = x0
    for k in range(n - 1):
        x[k+1] = x[k] + dt * (a * x[k])
    return t, x

t_e, x_e = euler_explicit(a, x0, T, dt=0.5)

# solve_ivp (alta precisione)
sol = solve_ivp(lambda t, x: a*x, (0, T), [x0], t_eval=t_grid,
                rtol=1e-9, atol=1e-12)

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(t_grid, x_exact, lw=2.2, color=COL["main"], label=r"esatta $x_0 e^{at}$")
ax.plot(sol.t, sol.y[0], "--", lw=1.6, color=COL["ok"], label="solve_ivp")
ax.plot(t_e, x_e, "o-", color=COL["accent"], lw=1.4, ms=5,
        label=f"Eulero esplicito, $\\Delta t=0.5$")
ax.set_xlabel("tempo $t$ [s]")
ax.set_ylabel("$x(t)$")
ax.set_title(r"$\dot{x}=ax$ con $a=$"+f"{a}"+r", $x_0=$"+f"{x0}")
ax.legend()
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — l'esponenziale esatta è la curva continua.
`solve_ivp` la riproduce con errore impercettibile. Eulero esplicito a passo
grande ($\Delta t=0.5$) la *sotto-stima* perché tira sempre la "tangente in
avanti" perdendo curvatura.

**Cosa cambia se varia $a$** — con $a=0$ la soluzione è costante; con $a>0$ la
soluzione esplode esponenzialmente. Più $|a|$ è grande, più Eulero esplicito
ha bisogno di passi piccoli per restare stabile.

**Perché è importante** — molti modelli fisiologici (decadimento di una
concentrazione, scarica di un capacitore di membrana, rilassamento muscolare)
sono varianti di questa singola equazione.


### Conversione di un'ODE di ordine $n$ in un sistema di primo ordine

Una qualunque ODE di ordine $n$ del tipo

$$
x^{(n)}=g\big(t,x,\dot x,\ddot x,\dots,x^{(n-1)}\big)
$$

si trasforma in un **sistema** di $n$ equazioni del primo ordine
introducendo le variabili ausiliarie $z_1=x$, $z_2=\dot x$, ...,
$z_n=x^{(n-1)}$:

$$
\begin{aligned}
\dot z_1 &= z_2,\\
\dot z_2 &= z_3,\\
&\;\vdots\\
\dot z_{n-1} &= z_n,\\
\dot z_n &= g(t,z_1,\dots,z_n).
\end{aligned}
$$

In forma matriciale per il caso lineare $\ddot x + 2\zeta\omega_n \dot x +
\omega_n^2 x = u(t)$:

$$
\frac{d}{dt}\begin{bmatrix} z_1\\ z_2\end{bmatrix}
=\begin{bmatrix} 0 & 1\\ -\omega_n^2 & -2\zeta\omega_n\end{bmatrix}
\begin{bmatrix} z_1\\ z_2\end{bmatrix}
+\begin{bmatrix} 0\\ 1\end{bmatrix}u(t).
$$

Questa è la **forma canonica controllabile**: ogni libro e ogni libreria
di controllo la usano. La useremo per il modello massa–molla–smorzatore e
per i circuiti RLC, ed è esattamente la struttura che ha la meccanica
respiratoria.


## 1.6 Serie di Taylor, Jacobiano e linearizzazione

La **serie di Taylor** di una funzione scalare attorno a $x^*$:

$$
f(x)=f(x^*)+f'(x^*)(x-x^*)+\tfrac{1}{2}f''(x^*)(x-x^*)^2+\cdots
$$

Per una funzione vettoriale $\mathbf{f}:\mathbb{R}^n\to\mathbb{R}^n$ il
ruolo della derivata lo prende la matrice **Jacobiana**:

$$
J(\mathbf{x})=\frac{\partial \mathbf{f}}{\partial \mathbf{x}}=
\begin{bmatrix}
\partial_{x_1} f_1 & \partial_{x_2} f_1 & \cdots & \partial_{x_n} f_1\\
\partial_{x_1} f_2 & \partial_{x_2} f_2 & \cdots & \partial_{x_n} f_2\\
\vdots & & \ddots & \vdots\\
\partial_{x_1} f_n & \partial_{x_2} f_n & \cdots & \partial_{x_n} f_n
\end{bmatrix}.
$$

Lo sviluppo del primo ordine attorno a $\mathbf{x}^*$ è

$$
\mathbf{f}(\mathbf{x})\approx \mathbf{f}(\mathbf{x}^*) +
J(\mathbf{x}^*)(\mathbf{x}-\mathbf{x}^*).
$$

### Linearizzazione di un sistema non lineare

Sia $\dot{\mathbf{x}}=\mathbf{f}(\mathbf{x})$ e $\mathbf{x}^*$ un equilibrio
(cioè $\mathbf{f}(\mathbf{x}^*)=\mathbf{0}$). Definiamo
$\boldsymbol{\xi}=\mathbf{x}-\mathbf{x}^*$. Allora:

$$
\dot{\boldsymbol{\xi}}=\dot{\mathbf{x}}=\mathbf{f}(\mathbf{x}^*+\boldsymbol{\xi})
\approx \mathbf{f}(\mathbf{x}^*)+J(\mathbf{x}^*)\boldsymbol{\xi}=J(\mathbf{x}^*)\boldsymbol{\xi}.
$$

Quindi vicino all'equilibrio il sistema si comporta come il sistema
**lineare** $\dot{\boldsymbol{\xi}}=J(\mathbf{x}^*)\boldsymbol{\xi}$.

### Teorema di Hartman–Grobman (versione "amichevole")

Se tutti gli autovalori di $J(\mathbf{x}^*)$ hanno parte reale **diversa da
zero** (equilibrio *iperbolico*), allora esiste un intorno di $\mathbf{x}^*$
nel quale le traiettorie del sistema non lineare e quelle del sistema
linearizzato sono *topologicamente equivalenti*: c'è una corrispondenza
continua biunivoca tra di esse.

In pratica: per equilibri iperbolici **la linearizzazione racconta la
verità locale**. Se invece un autovalore ha $\Re(\lambda)=0$ il teorema
non si applica e serve altro (centro, Hopf, ...).


In [ ]:
# Jacobiano numerico via differenze finite centrate, confronto con quello analitico
def jacobian_fd(f, x, eps=1e-6):
    """Calcola J(x) di f: R^n -> R^n con differenze centrate. Robusto e generico."""
    n = len(x)
    J = np.zeros((n, n))
    for j in range(n):
        ej = np.zeros(n); ej[j] = eps
        J[:, j] = (f(x + ej) - f(x - ej)) / (2 * eps)
    return J

# Esempio: Lotka-Volterra
def f_lv(z, a=1.0, b=0.5, c=0.75, d=0.25):
    x, y = z
    return np.array([a*x - b*x*y,
                     -c*y + d*x*y])

x_eq = np.array([3.0, 2.0])  # equilibrio non banale: c/d, a/b
print("f(x_eq) =", f_lv(x_eq))  # deve essere ~0

J_num = jacobian_fd(f_lv, x_eq)
print("Jacobiano numerico:\n", J_num)

# Jacobiano analitico per Lotka-Volterra
a, b, c, d = 1.0, 0.5, 0.75, 0.25
x, y = x_eq
J_ana = np.array([[a - b*y, -b*x],
                  [d*y,     -c + d*x]])
print("Jacobiano analitico:\n", J_ana)
print("Differenza max:", np.max(np.abs(J_num - J_ana)))

# Stabilita' locale: autovalori del Jacobiano
print("Autovalori in equilibrio:", np.linalg.eigvals(J_num))


## 1.7 Trasformata di Laplace e funzioni di trasferimento

Per i sistemi lineari **autonomi** (LTI) un cambio di prospettiva potentissimo
è la **trasformata di Laplace**. Per una funzione $f(t)$ definita su $t\ge 0$:

$$
F(s)=\mathcal{L}\{f(t)\}=\int_0^{\infty} f(t)\,e^{-st}\,dt,
$$

dove $s=\sigma+j\omega\in\mathbb{C}$. Le proprietà chiave per la modellistica:

| Proprietà | Espressione |
|---|---|
| linearità | $\mathcal{L}\{af+bg\}=aF+bG$ |
| derivata | $\mathcal{L}\{\dot f\}=sF(s)-f(0)$ |
| integrale | $\mathcal{L}\{\int_0^t f\}=F(s)/s$ |
| convoluzione | $\mathcal{L}\{f*g\}=F(s)G(s)$ |
| ritardo | $\mathcal{L}\{f(t-T)\}=e^{-sT}F(s)$ |
| esponenziale | $\mathcal{L}\{e^{at}\}=1/(s-a)$ |

### Dalla rappresentazione di stato alla funzione di trasferimento

Dato $\dot{\mathbf{x}}=A\mathbf{x}+B\mathbf{u}$, $\mathbf{y}=C\mathbf{x}+D\mathbf{u}$, con
condizioni iniziali nulle, trasformando:

$$
sX(s)=AX(s)+BU(s)\;\Longrightarrow\;X(s)=(sI-A)^{-1}B\,U(s).
$$

Quindi

$$
\boxed{\;G(s)=\frac{Y(s)}{U(s)}=C(sI-A)^{-1}B+D.\;}
$$

I **poli** di $G(s)$ sono gli autovalori di $A$ (almeno per i casi
"strutturalmente sensati"): leggere i poli equivale a leggere gli autovalori.

### Funzione di trasferimento di un sistema del secondo ordine

Per $\ddot y+2\zeta\omega_n \dot y+\omega_n^2 y=u(t)$:

$$
G(s)=\frac{1}{s^2+2\zeta\omega_n s+\omega_n^2}.
$$

I poli sono $-\zeta\omega_n\pm \omega_n\sqrt{\zeta^2-1}$. Per $\zeta<1$
oscillazioni smorzate; $\zeta=1$ smorzamento critico; $\zeta>1$ sovra-smorzato.
Lo useremo per la meccanica respiratoria semplificata.


In [ ]:
# Costruzione della G(s) di un secondo ordine e visualizzazione poli
omega_n = 2.0
for zeta in [0.2, 0.7, 1.0, 1.5]:
    poles = np.roots([1.0, 2*zeta*omega_n, omega_n**2])
    print(f"zeta={zeta:.2f} -> poli = {poles}")

# Risposta al gradino calcolata con solve_ivp (sistema di stato 2D)
def step_response(zeta, omega_n=2.0, T=10.0):
    A = np.array([[0.0, 1.0],
                  [-omega_n**2, -2*zeta*omega_n]])
    B = np.array([0.0, 1.0])
    def rhs(t, x): return A @ x + B * 1.0  # gradino u=1
    sol = solve_ivp(rhs, (0, T), [0.0, 0.0], dense_output=True, max_step=0.01)
    return sol

fig, ax = plt.subplots(figsize=(9, 4.8))
for zeta, c in zip([0.2, 0.5, 0.7, 1.0, 1.5],
                    [COL["accent"], COL["warn"], COL["ok"], COL["main"], COL["extra"]]):
    sol = step_response(zeta)
    ax.plot(sol.t, sol.y[0], label=fr"$\zeta={zeta}$", color=c)
ax.axhline(1/omega_n**2, color="black", lw=0.8, ls=":", label="valore a regime")
ax.set_xlabel("tempo $t$")
ax.set_ylabel("uscita $y(t)$")
ax.set_title(r"Risposta al gradino di $G(s)=\frac{1}{s^2+2\zeta\omega_n s+\omega_n^2}$, "
             rf"$\omega_n={omega_n}$")
ax.legend(); plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — al variare di $\zeta$:

- $\zeta=0.2$: oscillazioni marcate e lentamente smorzate ("sotto-smorzato");
- $\zeta=0.7$: compromesso tipico controlli (≈ 5% overshoot, salita rapida);
- $\zeta=1$: smorzamento critico, niente overshoot;
- $\zeta=1.5$: sovra-smorzato, lento.

**Come si interpreta** — $\zeta$ è il **rapporto di smorzamento**; controlla
quanto "energicamente" il sistema rallenta. $\omega_n$ è la **pulsazione naturale**:
controlla *quanto velocemente* il sistema risponde.

**Perché è importante** — la stessa equazione descrive un sistema massa-molla,
un circuito RLC e — con resistenza e compliance — un compartimento
respiratorio. La stessa intuizione si trasferisce ovunque.


## 1.8 Metodi numerici per ODE: Eulero, Runge–Kutta, stiffness

In quasi tutti i modelli fisiologici interessanti non esiste soluzione
analitica. Si integra numericamente. Le tre famiglie di metodi più
importanti:

### Eulero esplicito (forward Euler)

$$
\mathbf{x}_{k+1}=\mathbf{x}_k+\Delta t\,\mathbf{f}(t_k,\mathbf{x}_k).
$$

**Pro**: semplicissimo. **Contro**: instabile se $\Delta t$ è troppo grande.
Per $\dot x=\lambda x$ richiede $|1+\lambda \Delta t|\le 1$.

### Eulero implicito (backward Euler)

$$
\mathbf{x}_{k+1}=\mathbf{x}_k+\Delta t\,\mathbf{f}(t_{k+1},\mathbf{x}_{k+1}).
$$

**Pro**: stabile per qualsiasi $\Delta t$ su sistemi dissipativi. **Contro**:
richiede di risolvere un'equazione non lineare a ogni passo (`scipy.optimize.root`).

### Runge–Kutta del 4° ordine (RK4)

$$
\begin{aligned}
\mathbf{k}_1&=\mathbf{f}(t_k,\mathbf{x}_k),\\
\mathbf{k}_2&=\mathbf{f}(t_k+\tfrac{\Delta t}{2},\mathbf{x}_k+\tfrac{\Delta t}{2}\mathbf{k}_1),\\
\mathbf{k}_3&=\mathbf{f}(t_k+\tfrac{\Delta t}{2},\mathbf{x}_k+\tfrac{\Delta t}{2}\mathbf{k}_2),\\
\mathbf{k}_4&=\mathbf{f}(t_k+\Delta t,\mathbf{x}_k+\Delta t\mathbf{k}_3),\\
\mathbf{x}_{k+1}&=\mathbf{x}_k+\tfrac{\Delta t}{6}(\mathbf{k}_1+2\mathbf{k}_2+2\mathbf{k}_3+\mathbf{k}_4).
\end{aligned}
$$

**Pro**: ordine 4 di accuratezza, sufficiente per la maggior parte dei modelli.
**Contro**: non robusto su sistemi *stiff*.

### Stiffness

Un sistema è **stiff** se accoppia dinamiche su scale temporali molto diverse
(es. una capacità di membrana che scarica in 1 ms e un trend di concentrazione
che cambia in 100 s). Un esplicito è obbligato a usare passi piccolissimi
imposti dalla scala più rapida → diventa lentissimo. Servono i metodi
**impliciti** (Eulero implicito, BDF, Radau...).

`solve_ivp` ha varie modalità: `RK45` (default, esplicito adattivo),
`BDF`, `Radau` (impliciti, robusti su stiff).


In [ ]:
# Confronto: Eulero esplicito, Eulero implicito, RK4 "a mano", solve_ivp(RK45)
def euler_explicit(f, x0, T, dt):
    n = int(np.round(T / dt)) + 1
    t = np.linspace(0, T, n)
    x = np.zeros((len(x0), n)); x[:, 0] = x0
    for k in range(n - 1):
        x[:, k+1] = x[:, k] + dt * f(t[k], x[:, k])
    return t, x

def euler_implicit(f, x0, T, dt):
    """x_{k+1} risolvendo x_{k+1} - x_k - dt f(t_{k+1}, x_{k+1}) = 0."""
    n = int(np.round(T / dt)) + 1
    t = np.linspace(0, T, n)
    x = np.zeros((len(x0), n)); x[:, 0] = x0
    for k in range(n - 1):
        x_prev = x[:, k]; t_next = t[k+1]
        residual = lambda xnext: xnext - x_prev - dt * f(t_next, xnext)
        sol = root(residual, x_prev, method="hybr")
        x[:, k+1] = sol.x
    return t, x

def rk4(f, x0, T, dt):
    n = int(np.round(T / dt)) + 1
    t = np.linspace(0, T, n)
    x = np.zeros((len(x0), n)); x[:, 0] = x0
    for k in range(n - 1):
        k1 = f(t[k],          x[:, k])
        k2 = f(t[k]+dt/2,      x[:, k] + dt/2 * k1)
        k3 = f(t[k]+dt/2,      x[:, k] + dt/2 * k2)
        k4 = f(t[k]+dt,        x[:, k] + dt   * k3)
        x[:, k+1] = x[:, k] + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
    return t, x

# Test: scalare stiff dot x = -50 x, T=1, vari dt
lam = -50.0
def f(t, x): return lam * x

T = 1.0
dt_big = 0.05   # critico per esplicito: 1 + lam*dt = 1 - 2.5 < -1 -> esplode
dt_small = 0.005

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
t_ref = np.linspace(0, T, 400); x_ref = np.exp(lam * t_ref)

for ax, dt in zip(axes, [dt_big, dt_small]):
    t1, x1 = euler_explicit(f, np.array([1.0]), T, dt)
    t2, x2 = euler_implicit(f, np.array([1.0]), T, dt)
    t3, x3 = rk4(f, np.array([1.0]), T, dt)
    ax.plot(t_ref, x_ref, color="black", lw=2, label="esatta $e^{-50t}$")
    ax.plot(t1, x1[0], "o-", color=COL["accent"], ms=4, label="Eulero esplicito")
    ax.plot(t2, x2[0], "s-", color=COL["main"], ms=4, label="Eulero implicito")
    ax.plot(t3, x3[0], "^-", color=COL["ok"], ms=4, label="RK4")
    ax.set_title(rf"$\dot x=-50x$, $\Delta t={dt}$")
    ax.set_xlabel("t"); ax.set_ylabel("x")
    ax.set_ylim(-0.4, 1.1)
    ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — a sinistra ($\Delta t=0.05$): Eulero esplicito
oscilla esplodendo (il passo viola la condizione di stabilità $|1+\lambda\Delta t|<1$),
mentre Eulero implicito resta stabile e RK4 è quasi esatto. A destra
($\Delta t=0.005$): tutti i metodi danno la risposta giusta.

**Lezione operativa** — per un sistema *stiff* (autovalori grandi in modulo)
il passo dell'esplicito è bloccato dalla scala più rapida, anche se la
soluzione varia lentamente. Gli impliciti rompono questo vincolo a costo
di risolvere un'equazione non lineare a ogni passo.


## 1.9 Analisi dimensionale e nondimensionalizzazione

Ogni grandezza fisica ha **dimensioni**. Un'equazione dinamica ben posta
deve avere dimensioni coerenti su entrambi i lati. Esempio: nell'equazione
RC di membrana

$$
C_m \frac{dV}{dt} = -g_L (V - E_L) + I_{ext},
$$

verifichiamo le unità:

- $C_m\,dV/dt$: $\;\frac{\text{F}\cdot\text{V}}{\text{s}}=\text{A}$ ✓;
- $g_L(V-E_L)$: $\;\text{S}\cdot\text{V}=\text{A}$ ✓;
- $I_{ext}$: $\;\text{A}$ ✓.

Tutto coerente: un'**equazione di corrente**.

### Perché nondimensionalizzare

Definire variabili adimensionali $x=X/X_{ref}$, $\tau=t/T_{ref}$ riduce il
numero di parametri "veri" del problema e fa emergere i **rapporti
caratteristici** (numeri adimensionali). Lo facciamo esplicitamente per:

- l'equazione logistica $\dot N = r N(1-N/K) \;\to\; \dot n=n(1-n)$,
- l'equazione di Van der Pol,
- l'equazione di Hodgkin–Huxley (versione adimensionata).

Risultato: dimostriamo che modelli diversi *con gli stessi parametri
adimensionali* hanno dinamica qualitativamente identica.

### Esempio: nondimensionalizzazione della logistica

Partiamo da $\dot N=rN(1-N/K)$. Definiamo $n=N/K$, $\tau=rt$. Allora
$dN/dt=K\,dn/dt\,r$ (regola della catena) e

$$
K r \frac{dn}{d\tau}=rKn(1-n) \;\Longrightarrow\; \frac{dn}{d\tau}=n(1-n).
$$

Nessun parametro residuo: l'unica scala temporale è $1/r$, l'unica scala
di popolazione è $K$.


# Parte 2 — Elettrotecnica essenziale per la fisiologia

I modelli fisiologici dinamici sono **quasi tutti** equivalenti a circuiti
elettrici grazie a due osservazioni:

1. Le membrane biologiche si comportano da **capacitori** in parallelo con
   **conduttanze ioniche**;
2. I sistemi cardiovascolare e respiratorio sono reti idrauliche con
   **resistenze** (perdite per attrito) e **complianze** (deformabilità dei vasi
   e degli alveoli), cioè di nuovo $R$ e $C$.

Quindi una solida intuizione di elettrotecnica = una solida intuizione di
fisiologia. In questa parte ricostruiamo, con schemi disegnati direttamente
da Python, tutti i mattoni elementari.

## 2.1 Grandezze fondamentali

| Grandezza elettrica | Simbolo | Unità | Analogo fisiologico |
|---|---|---|---|
| tensione (potenziale) | $V$ | V | pressione (mmHg, cmH$_2$O), conc. mEq/L |
| corrente | $I$ | A | flusso volumetrico (mL/s), corrente di soluto |
| resistenza | $R$ | $\Omega$ | resistenza vascolare/aerea (mmHg·s/mL) |
| conduttanza | $G=1/R$ | S | conduttanza (più naturale in elettrofisiologia) |
| capacità | $C$ | F | compliance ($\Delta V/\Delta P$) |
| carica | $Q$ | C | volume immagazzinato |
| flusso di carica | $I=dQ/dt$ | A | flusso volumetrico |

**Leggi di base**:

- **Ohm**: $V=R\,I=I/G$;
- **Capacitore**: $I_C=C\,dV/dt$, cioè la corrente è proporzionale alla
  *rapidità di variazione* della tensione;
- **Kirchhoff delle correnti (KCL)**: la somma algebrica delle correnti
  entranti in un nodo è zero;
- **Kirchhoff delle tensioni (KVL)**: la somma algebrica delle tensioni
  lungo una maglia chiusa è zero.

Tutte le derivazioni di modelli RC, RLC, membrana, ventricolo seguiranno
da queste quattro regole.


## 2.0 — Elettromagnetismo essenziale: dai campi ai potenziali

Prima di mettere insieme resistenze, condensatori e fili in un *circuito*,
serve avere chiaro **da dove vengono** quelle quantità. La risposta sta
nell'elettromagnetismo: cariche, campi, potenziali. Spieghiamo solo ciò
che serve per i modelli del notebook (membrane, canali ionici, doppio
strato lipidico), in modo intuitivo ma rigoroso.

> **Filo conduttore**: tutta la fisiologia bioelettrica ruota attorno a
> *cariche ioniche separate da una sottile membrana isolante*. Una volta
> capita questa geometria, l'origine fisica di $C_m\approx 1\,\mu\text{F/cm}^2$,
> della legge di Ohm a livello locale e dell'equazione di Nernst diventa
> automatica.

### 2.0.1 Carica elettrica come grandezza fisica fondamentale

L'universo macroscopico è fatto di due tipi di carica:

- **positiva** (protoni, ioni $Na^+, K^+, Ca^{2+}$);
- **negativa** (elettroni, ioni $Cl^-$, anioni proteici, fosfati).

La **carica elementare** vale $e \approx 1.602 \cdot 10^{-19}$ C. La
carica di una popolazione di N ioni monovalenti è $Q = N\,e$ (con segno).

**Legge di Coulomb** — due cariche puntiformi $q_1, q_2$ separate da una
distanza $r$ esercitano una forza

$$
\vec F = \frac{1}{4\pi\varepsilon_0}\,\frac{q_1 q_2}{r^2}\,\hat r,
$$

con $\varepsilon_0 \approx 8.854\cdot 10^{-12}$ F/m (**permittività del vuoto**).
La forza è **repulsiva** se le cariche hanno lo stesso segno, **attrattiva**
se hanno segno opposto, e *cala come* $1/r^2$.

In presenza di un materiale (acqua, lipide), $\varepsilon_0$ si rimpiazza
con $\varepsilon = \varepsilon_0 \varepsilon_r$ dove $\varepsilon_r$ è la
**costante dielettrica relativa**: $\varepsilon_r \approx 80$ per
l'acqua, $\approx 2$ per il doppio strato lipidico delle membrane.


### 2.0.2 Campo elettrico $\vec E$ — la "forza per unità di carica"

Il campo elettrico in un punto è la forza che una carica di prova
unitaria sentirebbe se messa lì:

$$
\vec E(\vec r) = \frac{\vec F}{q_{\text{prova}}}, \quad [\vec E] = \text{V/m}.
$$

Per una carica puntiforme $q$ nell'origine:

$$
\vec E(\vec r) = \frac{1}{4\pi\varepsilon}\,\frac{q}{r^2}\,\hat r.
$$

**Linee di campo**: linee immaginarie tangenti a $\vec E$ in ogni punto.
Escono dalle cariche positive, entrano in quelle negative. La densità
delle linee misura l'intensità.

### 2.0.3 Potenziale elettrico $V$ — l'"altezza" da cui deriva il campo

Il potenziale in un punto è l'**energia potenziale per carica unitaria**:

$$
V(\vec r) = \frac{U_e(\vec r)}{q}, \quad [V] = \text{J/C} = \text{volt}.
$$

Legame fondamentale con $\vec E$:

$$
\vec E = -\nabla V,\qquad V(B) - V(A) = -\int_A^B \vec E \cdot d\vec\ell.
$$

In parole: il campo elettrico "scende" sempre dalle alte tensioni alle
basse, e la **tensione tra due punti** è l'integrale del campo lungo un
qualunque cammino che li unisce.

> *Mantra*: $V$ sta a "altezza" come $\vec E$ sta a "pendenza".

**Conseguenze immediate**:

- In un **conduttore** in equilibrio elettrostatico $\vec E_{\text{interno}}=0$
  (altrimenti le cariche libere si muoverebbero), quindi *tutto il conduttore
  è equipotenziale*.
- In un **isolante** (es. doppio strato lipidico) un campo può sopravvivere:
  è la base del condensatore.


In [ ]:
# Visualizzazione: linee di campo di un dipolo elettrico (analogo della membrana)
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-2, 2, 30)
y = np.linspace(-2, 2, 30)
X, Y = np.meshgrid(x, y)

# Dipolo: carica +1 in (0, 0.4), carica -1 in (0, -0.4)
def field(X, Y, charges):
    Ex = np.zeros_like(X); Ey = np.zeros_like(Y)
    for q, (cx, cy) in charges:
        dx, dy = X - cx, Y - cy
        r2 = dx**2 + dy**2 + 1e-6
        r3 = r2**1.5
        Ex += q * dx / r3
        Ey += q * dy / r3
    return Ex, Ey

Ex, Ey = field(X, Y, [(+1, (0, 0.4)), (-1, (0, -0.4))])
M = np.hypot(Ex, Ey)

fig, ax = plt.subplots(figsize=(7, 7))
ax.streamplot(X, Y, Ex, Ey, color=np.log10(M+1e-9), cmap='RdYlBu_r',
              density=1.6, linewidth=1)
ax.plot(0, +0.4, 'o', color='red', ms=18, markeredgecolor='black')
ax.plot(0, -0.4, 'o', color='blue', ms=18, markeredgecolor='black')
ax.text(0, +0.4, '+', ha='center', va='center', color='white', fontsize=18, fontweight='bold')
ax.text(0, -0.4, '−', ha='center', va='center', color='white', fontsize=18, fontweight='bold')
ax.set_xlim(-2, 2); ax.set_ylim(-2, 2); ax.set_aspect('equal')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Linee di campo elettrico di un dipolo\n(analogo a un piccolo elemento di membrana polarizzata)')
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — le linee escono dalla carica positiva
ed entrano in quella negativa. Il colore indica l'intensità del campo
(più rosso = più forte). La membrana cellulare, vista da vicino, è
**l'analogo macroscopico di un dipolo distribuito**: c'è separazione di
carica tra le due facce, e quindi un campo elettrico forte attraverso
i 5 nm dello spessore lipidico.

### 2.0.4 Legge di Gauss — il "flusso del campo elettrico"

La **legge di Gauss** è uno dei quattro postulati di Maxwell. Dice che il
**flusso del campo elettrico attraverso una superficie chiusa** è proporzionale
alla **carica racchiusa**:

$$
\oint_{\partial\mathcal{V}} \vec E \cdot d\vec A = \frac{Q_{\text{enc}}}{\varepsilon}.
$$

In parole: "se *dentro* la scatola c'è una carica netta, le linee di campo
devono *uscirne* (o *entrarci*) per un totale proporzionale a $Q$".

Per noi conta soprattutto un caso particolare: **due piastre piane parallele**
caricate con $+\sigma$ e $-\sigma$ (densità di carica superficiale,
$\sigma$ in C/m²). Applicando Gauss a una "scatolina" attraverso una piastra:

$$
E_{\text{tra le piastre}} = \frac{\sigma}{\varepsilon}.
$$

Il campo è **uniforme e perpendicolare** alle piastre. Fuori è $\approx 0$.


### 2.0.5 Capacitore a piastre piane — perché $C = \varepsilon A / d$

Date due piastre conduttrici di area $A$ separate da una distanza $d$ con
un isolante di permittività $\varepsilon$:

- carica $Q$ → densità $\sigma = Q/A$;
- campo (Gauss) $E = \sigma/\varepsilon = Q/(\varepsilon A)$;
- tensione (integrazione) $V = E \cdot d = Q d/(\varepsilon A)$;
- **capacità**: $C = Q/V = \varepsilon A / d$.

$$
\boxed{\;C = \varepsilon_0 \varepsilon_r \,\frac{A}{d}\;}
$$

**Tre conseguenze**:

1. $C$ è grande quando $A$ è grande (più area = più carica per stesso $V$);
2. $C$ è grande quando $d$ è piccolo (campo più intenso a parità di $V$);
3. $C$ è grande quando $\varepsilon_r$ è grande (acqua = 80, lipide = 2).

### 2.0.6 Calcolo di $C_m$ del doppio strato lipidico

Una membrana cellulare è approssimabile a due "piastre conduttrici"
(le soluzioni elettrolitiche intra/extra) separate da un isolante
lipidico. Parametri tipici:

- $\varepsilon_r \approx 2$ (lipidi);
- $d \approx 5\;\text{nm} = 5\cdot 10^{-9}$ m (spessore doppio strato);
- $A = 1\;\text{cm}^2 = 10^{-4}$ m².

Allora

$$
C_m \approx \varepsilon_0 \varepsilon_r \frac{A}{d}
\approx 8.85\cdot 10^{-12}\cdot 2 \cdot \frac{10^{-4}}{5\cdot 10^{-9}}
\approx 0.35\,\mu\text{F/cm}^2.
$$

Il valore misurato è $\approx 1\,\mu\text{F/cm}^2$ — l'ordine di grandezza
è giusto, e la differenza si spiega con valori effettivi di $\varepsilon_r$
fra 2 e 6 (per il bilipide idratato + proteine). Il fatto che $C_m$ sia
una **costante universale** in tutte le cellule animali è proprio perché
$d$ è fissato dalla biochimica del fosfolipide.


In [ ]:
# Calcolo numerico esplicito di C_m
eps_0 = 8.854e-12   # F/m
A_cm2 = 1.0          # cm^2
A_m2 = A_cm2 * 1e-4 # m^2

for eps_r, d_nm in [(2.0, 5.0), (3.0, 4.0), (6.0, 3.0)]:
    d_m = d_nm * 1e-9
    C = eps_0 * eps_r * A_m2 / d_m
    print(f"eps_r={eps_r:.1f}, d={d_nm:.1f} nm -> C/A = {C*1e6:.2f} uF/cm^2")

# Conferma: il valore "biologico" tipico e' ~1 uF/cm^2


### 2.0.7 Energia immagazzinata in un capacitore

L'energia per caricare un capacitore da 0 a $V$ è

$$
U = \int_0^V V'\,dQ = \int_0^V V'\,C\,dV' = \tfrac{1}{2}CV^2.
$$

**In fisiologia**: l'energia elettrostatica di una membrana di area $A=1$
cm² polarizzata a $V_m = -70$ mV vale

$$
U \approx \tfrac{1}{2}\cdot 10^{-6}\cdot(0.07)^2 \approx 2.4\,\text{nJ/cm}^2.
$$

È piccolissima rispetto all'energia metabolica disponibile, ma il segreto
è la **densità di potenza dinamica**: durante un potenziale d'azione
$V_m$ cambia di 100 mV in 1 ms, il che corrisponde a flussi di carica
$Q = C_m \Delta V \approx 10^{-7}$ C/cm², cioè *milioni* di ioni per μm²,
e a costanti di tempo $\tau$ molto rapide.

### 2.0.8 Conducibilità e legge di Ohm in forma locale

Quando in un materiale conduttivo c'è un campo elettrico, le cariche
libere si muovono. In regime lineare la **densità di corrente**
$\vec J$ (A/m²) è proporzionale a $\vec E$:

$$
\boxed{\;\vec J = \sigma\,\vec E\;}
$$

dove $\sigma$ è la **conducibilità** (in S/m). Questa è la "legge di Ohm
in forma puntuale".

Per un cilindro di sezione $A$, lunghezza $L$, materiale di conducibilità $\sigma$:

- corrente $I = J\,A = \sigma\,E\,A$;
- tensione lungo il cilindro $V = E\,L$;
- quindi $V/I = L/(\sigma A) = \rho L/A = R$ con $\rho = 1/\sigma$ (resistività).

Si riottiene così la **forma macroscopica** $V = R I$ a partire dai campi.

> In fisiologia, $\sigma$ del citoplasma è $\sim 1$ S/m; quella dei lipidi
> è $\sim 10^{-15}$ S/m (sostanzialmente isolante). Questo enorme rapporto
> è ciò che rende la membrana un "buon" capacitore.

### 2.0.9 Equazione di continuità — la conservazione della carica

In un volume $\mathcal{V}$ con densità di carica $\rho(\vec r, t)$
(C/m³), la carica si conserva: ciò che entra/esce attraverso il bordo
cambia $Q_{\mathcal V}$:

$$
\frac{\partial \rho}{\partial t} + \nabla\cdot \vec J = 0.
$$

**Forma integrale**: $\frac{dQ_{\mathcal V}}{dt} = -\oint \vec J\cdot d\vec A$
— la carica nella scatola cala se la corrente fluisce fuori.

**Per i circuiti**: in un nodo non c'è "volume" che accumuli carica, quindi
$\sum I_{in} = \sum I_{out}$. **Questa è esattamente KCL**. KCL è una
versione discreta dell'equazione di continuità per le reti.


### 2.0.10 Diffusione di Fick — perché gli ioni si muovono anche senza campo

Anche in assenza di campo elettrico, una sostanza con **gradiente di
concentrazione** si diffonde: gli ioni saltano per agitazione termica,
dalle zone affollate verso quelle vuote.

**Legge di Fick** (1855):

$$
\boxed{\;\vec J_{\text{diff}} = -D\,\nabla c\;}
$$

con $c$ concentrazione (mol/m³) e $D$ **coefficiente di diffusione** (m²/s,
tipicamente $10^{-9}$ m²/s per ioni in acqua).

**Effetto sulla bioelettricità**: un gradiente di K (alto dentro, basso
fuori) genera un flusso diffusivo *uscente* anche se $V_m = 0$. È metà
della storia: l'altra metà è il **drift** elettrico, che spinge il K
verso la regione più negativa.

### 2.0.11 Equazione di Nernst–Planck (drift + diffusione)

Mettendo insieme drift elettrico e diffusione, il flusso totale di uno
ione $X$ di carica $z$ vale:

$$
\boxed{\;\vec J_X = -D_X\nabla c_X - \frac{z F D_X}{RT}\,c_X\,\nabla V\;}
$$

(con $F$ Faraday, $R$ costante dei gas, $T$ temperatura). All'equilibrio
elettrochimico $\vec J_X = 0$, e si ottiene esattamente l'**equazione di
Nernst** (sezione 7.8):

$$
\frac{dc_X}{c_X} = -\frac{zF}{RT}\,dV \;\Longleftrightarrow\;
V = E_X = \frac{RT}{zF}\ln\frac{[X]_o}{[X]_i}.
$$

> Quindi: la "batteria di Nernst" del canale ionico **è proprio** la
> tensione che si stabilirebbe se diffusione e drift si bilanciassero
> esattamente. Non è un artificio matematico ma una conseguenza diretta
> dell'equazione di Nernst-Planck.

### 2.0.12 Doppio strato di Debye — la "lunghezza di schermatura"

In una soluzione elettrolitica, una carica fissa è circondata da una nube
di ioni di carica opposta. Lo spessore tipico di questa "nube" è la
**lunghezza di Debye**:

$$
\lambda_D = \sqrt{\frac{\varepsilon_r\varepsilon_0 RT}{2 F^2 I_s}},
$$

con $I_s$ forza ionica della soluzione. Per fisiologia ($I_s \approx 0.15$ M)
$\lambda_D \approx 1$ nm.

**Conseguenza importante**: il campo elettrico nelle soluzioni intra/extracellulari
*si esaurisce* in pochi nanometri dalla membrana. Quindi quasi tutta la
differenza di potenziale $V_m$ "cade" attraverso il doppio strato
lipidico (5 nm), che si comporta come un capacitore con un piccolo
margine di schermatura ionica.


In [ ]:
# Visualizziamo l'andamento di V(x) attraverso una membrana semplificata
import numpy as np, matplotlib.pyplot as plt

# Coordinate: x in nm, x in [-15, +15], membrana centrata in 0 con spessore 5 nm
x = np.linspace(-15, 15, 600)
V_in, V_out = -70, 0       # mV
d_mem = 5                   # spessore membrana
lam_D = 1                   # lunghezza di Debye

def V_profile(x):
    out = np.zeros_like(x)
    for i, xi in enumerate(x):
        if xi <= -d_mem/2:
            # zona intracellulare: schermatura esponenziale verso V_in
            out[i] = V_in - V_in * np.exp(-(xi + d_mem/2) / lam_D) * 0  # quasi piatto a V_in
            out[i] = V_in
        elif xi >= d_mem/2:
            out[i] = V_out
        else:
            # zona membrana: salita lineare (capacitore a piastre)
            out[i] = V_in + (V_out - V_in) * (xi + d_mem/2) / d_mem
    return out

V = V_profile(x)

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_betweenx([-90, +10], -d_mem/2, d_mem/2, color='#fdebd0', alpha=0.7,
                  label='membrana lipidica (5 nm)')
ax.axvspan(-15, -d_mem/2, color='#aed6f1', alpha=0.3, label='intracellulare')
ax.axvspan(d_mem/2, 15, color='#d4efdf', alpha=0.3, label='extracellulare')
ax.plot(x, V, color='#1f4e79', lw=2.5)
ax.axhline(V_in, color='gray', lw=0.6, ls=':')
ax.axhline(V_out, color='gray', lw=0.6, ls=':')
ax.set_xlabel('x [nm]'); ax.set_ylabel('V(x) [mV]')
ax.set_title("Profilo di potenziale attraverso una membrana cellulare\n"
             "Quasi tutta la differenza V_m cade sui 5 nm del doppio strato lipidico")
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

print("Campo medio nella membrana (5 nm, 70 mV):")
E_field = 70e-3 / 5e-9
print(f"  E ~ {E_field:.2e} V/m = {E_field/1e6:.1f} MV/m")
print("  -> ENORME! Un dielettrico macroscopico si romperebbe.")
print("  La membrana resiste perche' e' solo 5 nm spessa (e quindi V finito).")


**Cosa mostra il grafico** — il potenziale è praticamente piatto
nelle soluzioni intra- ed extracellulari (lì le cariche sono libere di
muoversi e schermano), e **tutto il salto** $V_m$ avviene attraverso lo
strato lipidico isolante. Il campo elettrico medio dentro la membrana è
dell'ordine di **decine di MV/m** — enorme per uno standard macroscopico,
ma sopportabile perché lo strato è solo 5 nm.

### 2.0.13 Equazioni di Maxwell: cosa ci serve, cosa no

Le 4 equazioni di Maxwell sono:

1. Gauss: $\nabla \cdot \vec E = \rho/\varepsilon$ — sorgenti del campo elettrico.
2. Gauss magnetico: $\nabla \cdot \vec B = 0$ — nessun "mono-polo magnetico".
3. Faraday: $\nabla \times \vec E = -\partial \vec B/\partial t$ — un campo magnetico variabile *induce* un campo elettrico.
4. Ampère-Maxwell: $\nabla \times \vec B = \mu_0(\vec J + \varepsilon_0 \partial \vec E/\partial t)$.

Per la **modellistica fisiologica delle membrane**:

- gli effetti magnetici sono trascurabili (correnti piccole, frequenze basse → $\vec B \approx 0$);
- i tempi caratteristici sono > 0.1 ms → l'**approssimazione quasi-statica** è eccellente: si usa solo Gauss + continuità, e si dimenticano gli effetti d'induzione e di propagazione delle onde EM.

Questo è il motivo per cui i nostri modelli sono ODE in $V_m(t)$, e non
PDE complete in $(t, \vec r)$ con onde EM. La fisiologia di membrana
*vive* nella regione "quasi-statica" dell'elettromagnetismo.

> Eccezioni: nella **stimolazione magnetica transcranica (TMS)** un campo
> $\vec B$ variabile induce $\vec E$ nel tessuto secondo Faraday; lì
> serve l'EM dinamico. Ma è fuori dallo scope di questo notebook.

### 2.0.14 Riepilogo: da $\vec E$ a $V_m$ in 5 passi

| Passo | Concetto EM | Conseguenza fisiologica |
|---|---|---|
| 1 | Cariche generano $\vec E$ (Coulomb) | gli ioni separati dalla membrana creano un campo elettrico |
| 2 | $\vec E = -\nabla V$ | esiste una "tensione" $V_m$ ben definita |
| 3 | Gauss + piastre piane → $C = \varepsilon A/d$ | la membrana è un capacitore con $C_m \approx 1\,\mu\text{F/cm}^2$ |
| 4 | Continuità $\dot\rho + \nabla\cdot\vec J = 0$ | nei nodi vale KCL: bilancio di corrente |
| 5 | Nernst-Planck (drift + Fick) | la "batteria" $E_X$ è il potenziale di equilibrio |

A questo punto hai tutti i prerequisiti fisici per i modelli circuitali
delle membrane. Continuiamo con il vocabolario circuitale nella sezione
seguente.


## 2.0bis — Elettrotecnica circuitale da zero

Se non hai mai studiato elettrotecnica, questa sezione è il salvavita.
Spieghiamo da **zero**, con analogie idrauliche, esempi numerici, e
disegni: tutto ciò che serve per leggere ogni modello del notebook (RC di
membrana, circolazione, respiratoria, Hodgkin–Huxley).

> Idea guida: **un circuito non è altro che acqua che scorre fra serbatoi a
> diverse "altezze"**. Carica = acqua, corrente = portata, tensione = altezza,
> resistenza = strozzatura, capacitore = serbatoio elastico. Tienitela vicina.

### 2.0bis.1 Carica elettrica $Q$ — la "quantità di elettricità"

La materia è fatta di particelle cariche: elettroni (carica negativa,
$-1.6\cdot 10^{-19}$ C) e protoni/ioni positivi. La **carica** $Q$ si
misura in **coulomb (C)**: 1 C = 6.24 × 10¹⁸ elettroni.

In fisiologia gli ioni $Na^+, K^+, Cl^-, Ca^{2+}$ trasportano la carica
attraverso le membrane: sono *loro* la "corrente biologica".

**Conservazione**: in un sistema chiuso la carica totale non cambia. È
la base di tutto.

### 2.0bis.2 Corrente elettrica $I$ — "quanti coulomb al secondo"

$$
I = \frac{dQ}{dt}, \quad [I]=\frac{\text{C}}{\text{s}} = \text{ampere (A)}.
$$

In parole: **la corrente è la *portata* di carica**. Se in un filo passa
1 C ogni secondo, la corrente vale 1 A. Per gli ioni cellulari si lavora
in microampere o pico-ampere.

**Analogia idraulica**: il flusso d'acqua in un tubo, in litri/secondo,
è l'equivalente della corrente in A.

**Verso**: per convenzione, $I$ ha lo stesso *verso* del moto delle
**cariche positive**. Se sono gli elettroni a muoversi a destra,
la corrente convenzionale va a sinistra. (È un'antica scelta storica;
oggi nessuno la cambia.)


### 2.0bis.3 Tensione $V$ — "l'altezza energetica"

La **tensione** (o differenza di potenziale) tra due punti $A$ e $B$ è
**l'energia necessaria per spostare 1 coulomb da $A$ a $B$**:

$$
V_{AB} = \frac{E_{AB}}{Q}, \quad [V] = \frac{\text{J}}{\text{C}} = \text{volt (V)}.
$$

**Analogia**: tensione = **dislivello d'acqua** tra due serbatoi. Un'acqua
che è 1 m più in alto ha più "voglia" di scorrere verso il basso; analogamente,
le cariche positive sono spinte dai punti ad alta tensione verso quelli a
bassa tensione (e viceversa per le negative).

In fisiologia il **potenziale di membrana** $V_m$ (in millivolt) è
proprio la differenza di tensione tra l'interno e l'esterno della cellula.
A riposo è $\approx -70$ mV (interno negativo rispetto all'esterno).

### 2.0bis.4 La "terra" o ground: il riferimento zero

Le tensioni sono sempre **differenze**: per assegnare a un nodo un valore
"assoluto" serve scegliere un punto di riferimento, detto **ground** o **massa**.
Per convenzione il ground ha tensione 0. Nelle membrane il "fuori cellula"
è preso come ground, quindi $V_m = V_{in} - V_{out} = V_{in}$.

> **Mantra**: una tensione "assoluta" non esiste; è sempre rispetto a qualcosa.

### 2.0bis.5 Legge di Ohm — il legame fra $V$, $I$, $R$

In molti materiali (e in molte porzioni di membrana, in prima
approssimazione) vale la **legge di Ohm**:

$$
\boxed{\;V = R\,I\;}
$$

con $R$ = **resistenza** (in ohm, Ω). Letta in tre modi diversi:

- *fissato $V$, una $R$ più grande riduce $I$*: la resistenza "frena" la corrente;
- *fissata $R$, $V$ e $I$ sono proporzionali*: applichi più tensione → più corrente;
- *fissata $I$, $V$ aumenta linearmente con $R$*: la "caduta di tensione" è $RI$.

Analogia: $R$ è la **strozzatura del tubo**; a parità di dislivello
($V$), un tubo stretto lascia passare meno acqua.

**Conduttanza** = inverso della resistenza: $G = 1/R$, in **siemens (S)**.
Più conduttanza = più "facile" passare la corrente. In fisiologia
celebriamo $G$ perché corrisponde a "*quanti canali ionici sono aperti*".


In [ ]:
# Esempio numerico: legge di Ohm su una membrana semplificata
# Dato V (potenziale di driving force) e g_L (conduttanza di leak), trovo I.
V_driv = 10.0e-3   # 10 mV (rispetto al riposo)
g_L = 30e-9        # 30 nS (nanosiemens)
I = g_L * V_driv   # corrente di leak
print(f"V driving = {V_driv*1e3:.1f} mV")
print(f"g_L       = {g_L*1e9:.1f} nS = 1/{1/g_L:.2e} Ohm")
print(f"I = g_L * V = {I*1e9:.2f} nA = {I*1e12:.0f} pA")

# Inversamente: per un canale singolo con g_single = 20 pS, quanti aperti servono
# per ottenere 1 nA con driving 50 mV?
g_single = 20e-12
V_d = 50e-3
I_target = 1e-9
N_open = I_target / (g_single * V_d)
print(f"\nPer I = 1 nA con V_d = 50 mV servono ~{N_open:.0f} canali da 20 pS aperti.")


### 2.0bis.6 Potenza dissipata sui resistori

Quando una corrente $I$ attraversa una resistenza $R$ sotto tensione $V$,
viene dissipata **potenza** (in **watt**, W):

$$
P = V\,I = R\,I^2 = \frac{V^2}{R}, \quad [P] = \text{V}\cdot\text{A}=\text{W}.
$$

In fisiologia non parliamo di "watt" sui canali ionici, ma il principio è
identico: il flusso di ioni *dissipa* energia chimica (gradiente
elettrochimico) → la cellula deve spendere ATP per rifare il gradiente
con le pompe Na/K. Senza le pompe, i gradienti si appiattirebbero in
qualche minuto.

### 2.0bis.7 Il capacitore — il "serbatoio di carica"

Un **capacitore** è due conduttori (piastre) separati da un isolante. Se
applichi una tensione $V$ fra le piastre, una si carica positivamente di
$+Q$ e l'altra negativamente di $-Q$, con la relazione lineare

$$
\boxed{\;Q = C\,V\;}
$$

con $C$ = **capacità** (in **farad**, F). $C$ dipende dall'area, dallo
spessore, dal materiale isolante.

**In fisiologia**: la membrana cellulare è uno strato lipidico isolante
spesso ~5 nm fra due "soluzioni conduttrici" (intra-/extracellulare). È
*letteralmente* un capacitore. La sua capacità per cm² vale circa
$C_m \approx 1\,\mu\text{F/cm}^2$, valore notoriamente costante in tutte
le cellule del regno animale (perché il doppio strato lipidico ha sempre
le stesse proprietà).

**Equazione fondamentale del capacitore** — derivando $Q=CV$ rispetto al tempo:

$$
\frac{dQ}{dt} = C\,\frac{dV}{dt}, \quad\text{ma } \frac{dQ}{dt} = I,
$$

quindi

$$
\boxed{\;I = C\,\frac{dV}{dt}\;}
$$

**Lettura intuitiva**:

- per **cambiare** la tensione su un capacitore *serve* corrente;
- se la tensione è *costante*, la corrente nel capacitore è zero;
- più $C$ è grande, più "carica serve" per cambiare $V$ di un dato $\Delta V$.

**Analogia**: il capacitore è un secchio elastico. $V$ è il livello
d'acqua, $C$ è l'area di base (capienza), $Q$ è il volume d'acqua
contenuto, $I=dQ/dt$ è la portata che entra/esce.


In [ ]:
# Esempio numerico: quanto tempo per caricare un capacitore
C = 1e-12     # 1 pF (tipico di un piccolo segmento di membrana)
I = 1e-9      # 1 nA di corrente costante iniettata
dV_dt = I/C   # quanto cambia V al secondo
print(f"Capacitore C = {C*1e12:.1f} pF caricato con I = {I*1e9:.1f} nA:")
print(f"  dV/dt = I/C = {dV_dt:.3e} V/s = {dV_dt*1e3:.1f} mV/ms")

# Quanto tempo per andare da V=0 a V=20 mV?
dV_target = 20e-3
dt = dV_target / dV_dt
print(f"  Per arrivare a V = 20 mV servono {dt*1e3:.2f} ms.")
print()

# Anche un capacitore "fisiologico" tipico (membrana 100 um di lato = 1e-4 cm^2):
C_real = 1.0e-6 * 1e-4   # F/cm^2 * area in cm^2
print(f"Membrana di 0.0001 cm^2 -> C = {C_real*1e12:.1f} pF")


### 2.0bis.8 Nodi, rami, maglie: il vocabolario dei circuiti

Un circuito è un *grafo*:

- **ramo** (o branca) = un singolo componente (resistenza, capacitore, generatore...);
- **nodo** = punto di giunzione di due o più rami;
- **maglia** = qualunque cammino chiuso.

Esempi pratici (vedi disegno sotto):

- circuito serie di $R_1$ e $R_2$: stesso ramo, 2 nodi (estremi);
- circuito parallelo di $R_1$ e $R_2$: 2 rami in parallelo, 2 nodi.

### 2.0bis.9 Convenzioni di segno (la "trappola" da capire una volta)

Per ogni componente serve fissare:

1. **Verso positivo della corrente** (di solito una freccia stilizzata sul filo);
2. **Polarità del riferimento di tensione** (di solito "+" da un lato, "−" dall'altro).

**Convenzione passiva** (la più usata): se la corrente entra dal lato "+",
allora $V$ e $I$ hanno lo stesso segno per i componenti passivi (R, L, C).
Per i generatori si usa la convenzione **attiva** (corrente esce dal lato "+").

Nei modelli di membrana usiamo:

- $V_m = V_{in} - V_{out}$;
- **corrente entrante in cellula** = positiva, perché aumenta la $Q$ intracellulare;
- corrente di canale: $I_X = g_X (V_m - E_X)$ è *positiva* (uscente) quando $V_m > E_X$.

### 2.0bis.10 Le leggi di Kirchhoff — bilanci elementari

**KCL (Kirchhoff Current Law)** — conservazione della carica nei nodi:

$$
\sum_{k\text{ entranti}} I_k = \sum_{k\text{ uscenti}} I_k.
$$

In ogni istante, **la corrente che entra in un nodo = la corrente che esce**.
Banalmente: la carica non si accumula nei fili.

**KVL (Kirchhoff Voltage Law)** — la tensione è una funzione di stato:

$$
\sum_{k\text{ lungo una maglia chiusa}} V_k = 0.
$$

Sommando le cadute di tensione lungo una maglia (rispettando i segni) si
torna a zero. Equivale a dire che il "potenziale" è univoco.

> **In modellistica fisiologica, KCL al nodo intracellulare *è*
> l'equazione del modello**. È un bilancio: tutto quello che entra
> (corrente iniettata) è uguale a tutto quello che esce (correnti attraverso
> i canali) più ciò che si accumula sul capacitore di membrana.


In [ ]:
# Esempio numerico esplicito di KCL e KVL su un circuito a 2 maglie
# Schema:
#
#        + V1=10 V -      R1 = 2 Ohm     R2 = 4 Ohm
#       [Generatore] ---/\/\/\/\/\---+---/\/\/\/\/\--- (terra)
#                                    |
#                                    R3 = 6 Ohm
#                                    |
#                                  (terra)
#
# Domanda: I_1, I_2, I_3 e V al nodo centrale?

import numpy as np

V1 = 10.0
R1, R2, R3 = 2.0, 4.0, 6.0

# Metodo del nodo: scrivo KCL al nodo centrale V (con V_terra = 0)
# (V1 - V)/R1 = V/R2 + V/R3
# -> V * (1/R1 + 1/R2 + 1/R3) = V1/R1
G_tot = 1/R1 + 1/R2 + 1/R3
V_nodo = (V1/R1) / G_tot

I1 = (V1 - V_nodo) / R1   # entrante nel nodo
I2 = V_nodo / R2           # uscente nel nodo
I3 = V_nodo / R3           # uscente nel nodo

print(f"V al nodo centrale: {V_nodo:.4f} V")
print(f"I1 (da V1)  : {I1:.4f} A")
print(f"I2 (verso R2): {I2:.4f} A")
print(f"I3 (verso R3): {I3:.4f} A")
print(f"KCL al nodo:  I1 = I2 + I3  ?  {I1:.4f} = {I2+I3:.4f}  -> {abs(I1 - (I2+I3)) < 1e-9}")

# Verifica KVL sulla maglia esterna (R1 + R2):
print(f"KVL maglia 1: V1 - R1*I1 - R2*I2 = {V1 - R1*I1 - R2*I2:.6f} (deve essere 0)")


### 2.0bis.11 Resistenze in serie e in parallelo (intuizione)

**In serie** ($R_1$ e $R_2$ uno dopo l'altro, stessa corrente attraverso entrambi):

$$
R_{eq} = R_1 + R_2.
$$

Intuizione: due strozzature in fila → più strozzato. La conduttanza
combinata: $1/G_{eq} = 1/G_1 + 1/G_2$.

**In parallelo** ($R_1$ e $R_2$ tra gli stessi due nodi, stessa tensione):

$$
\frac{1}{R_{eq}} = \frac{1}{R_1} + \frac{1}{R_2}, \quad
G_{eq} = G_1 + G_2.
$$

Intuizione: più strade in parallelo → la corrente trova più "vie" → si
divide e passa più facilmente. **Le conduttanze in parallelo si sommano**.

**Conseguenza fisiologica**: la membrana ha *molti canali in parallelo*.
La conduttanza totale è la **somma** delle conduttanze dei singoli tipi
di canale:

$$
g_{tot}(V, t) = g_{Na}(V, t) + g_K(V, t) + g_L.
$$

Quando un canale si apre, la sua conduttanza aumenta e $g_{tot}$ pure.
Quando si chiude, la conduttanza scompare dal totale. È letteralmente
"aggiungere/togliere un tubo in parallelo".

### 2.0bis.12 Sorgenti di tensione e di corrente

Una **sorgente di tensione ideale** mantiene $V$ costante (o $V(t)$ noto)
*qualunque* corrente passi.
Una **sorgente di corrente ideale** mantiene $I$ costante (o $I(t)$ noto)
*qualunque* tensione si stabilisca.

**In fisiologia**:

- una "batteria di Nernst" $E_X$ è una sorgente di tensione: rappresenta il
  potenziale di equilibrio dello ione $X$;
- un'iniezione di corrente da un elettrodo è una sorgente di corrente;
- un *patch clamp* può funzionare sia in modalità "voltage clamp" (sorgente di tensione)
  sia in modalità "current clamp" (sorgente di corrente).


### 2.0bis.13 Il canale ionico = batteria + conduttanza (LA chiave per HH)

Questa è **l'idea centrale** dell'elettrofisiologia. Un canale ionico
selettivo per lo ione $X$ è equivalente a:

1. una **batteria** ideale di tensione $E_X$ (il potenziale di Nernst);
2. una **conduttanza** $g_X$ in serie (eventualmente variabile in $V$ e
   $t$, come per i canali voltaggio-dipendenti).

La corrente attraverso questo "ramo" è

$$
\boxed{\;I_X = g_X\,(V_m - E_X)\;}
$$

che si chiama "**forma a driving force**": l'**driving force** è
$(V_m - E_X)$ ed è zero quando il potenziale di membrana è esattamente al
potenziale di Nernst dello ione → equilibrio elettrochimico, nessun flusso netto.

Pictogrammaticamente:

```
   V_m  o---[ g_X ]---[+E_X-]---o  V_out
         <-- I_X  (positiva se entrante)
```

**Perché questa forma è universale**:

- se $V_m > E_X$ → driving force > 0 → corrente *uscente* (ioni positivi che
  vanno *fuori* dalla cellula);
- se $V_m < E_X$ → driving force < 0 → corrente *entrante*;
- se $V_m = E_X$ → corrente zero.

**Esempio**: $E_{Na} \approx +60$ mV. A riposo $V_m \approx -70$ mV,
quindi driving force = $-70 - 60 = -130$ mV → corrente Na *fortemente entrante*
quando i canali si aprono → depolarizzazione rapida (potenziale d'azione).

**Esempio**: $E_K \approx -90$ mV. A riposo $V_m \approx -70$ mV → driving
force = $-70 - (-90) = +20$ mV → corrente K *uscente* quando i canali si aprono
→ iperpolarizzazione/ripolarizzazione.

> **Tutta** la matematica di Hodgkin–Huxley si riduce ad applicare KCL
> alla membrana, scrivendo *ogni* canale come "batteria + conduttanza" in
> parallelo a un capacitore $C_m$. La sezione 7.10 fa esattamente questo.


In [ ]:
# Esempio numerico: corrente di K in una membrana per diversi V_m
import matplotlib.pyplot as plt
import numpy as np

E_K = -90e-3        # potenziale di Nernst K
g_K = 100e-9        # conduttanza K aperta (100 nS, idealmente)

V_m_range = np.linspace(-120e-3, +40e-3, 200)
I_K = g_K * (V_m_range - E_K)   # legge di Ohm con E_K come riferimento

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(V_m_range*1e3, I_K*1e9, color='#1f4e79', lw=2.2)
ax.axhline(0, color='gray', lw=0.7)
ax.axvline(E_K*1e3, color='red', ls='--', label=f"$E_K = {E_K*1e3:.0f}$ mV (no corrente)")
ax.fill_betweenx([I_K.min()*1e9, 0], -120, E_K*1e3, alpha=0.1, color='green')
ax.fill_betweenx([0, I_K.max()*1e9], E_K*1e3, 40, alpha=0.1, color='red')
ax.text(-105, -7, "I_K entrante\n(V_m < E_K)", color='green', fontsize=10)
ax.text(-30, 7, "I_K uscente\n(V_m > E_K)", color='red', fontsize=10)
ax.set_xlabel('$V_m$ [mV]'); ax.set_ylabel('$I_K$ [nA]')
ax.set_title("Driving force: $I_K = g_K \\, (V_m - E_K)$")
ax.legend(); plt.tight_layout(); plt.show()


### 2.0bis.14 Impedenza e reattanza — accenni per la frequenza

Quando l'ingresso è sinusoidale, $V(t) = V_0\cos(\omega t)$, su un capacitore
non vale più $V = R I$ ma una versione "complessa":

$$
V(t) = \frac{1}{j\omega C}\,I(t),
$$

dove $1/(j\omega C)$ è la **reattanza capacitiva**. La quantità complessa
$Z(j\omega) = R + 1/(j\omega C)$ si chiama **impedenza** ed estende
la legge di Ohm al regime sinusoidale: $V = Z\,I$.

**Conseguenza pratica**: a basse frequenze il capacitore si comporta come
un "circuito aperto" ($|Z|\to\infty$), ad alte frequenze come un
"corto circuito" ($|Z|\to 0$). È il motivo per cui un RC è un **filtro
passa-basso**: smorza le frequenze alte (perché passano attraverso $C$
con bassa impedenza, "perdendo" energia).

Questo concetto torna in Parte 4 con i diagrammi di Bode e di Nyquist.

### 2.0bis.15 Dalla rete circuitale all'equazione differenziale — la "ricetta"

In tutti i modelli del notebook seguiamo lo **stesso giro**:

1. **Disegna** la rete equivalente: nodi, rami, generatori, capacitori, resistenze.
2. **Scegli un riferimento** (terra) e dai nomi alle **tensioni dei nodi non-terra**.
3. Per ogni capacitore: la corrente che lo attraversa è $C\,\dot V_{nodo}$.
4. Per ogni resistenza o conduttanza tra nodi $A$ e $B$: $I = (V_A - V_B)/R = g(V_A - V_B)$.
5. **Applica KCL ad ogni nodo non-terra**: somma algebrica delle correnti = 0.
6. Riarrangia per isolare $\dot V_{nodo}$ → ottieni un'**equazione differenziale**
   nelle tensioni dei nodi.

Questa ricetta produce:

- l'equazione di membrana RC (sezione 2.3);
- le 3 equazioni del modello cardiocircolatorio (sezione 7.1);
- le equazioni della meccanica respiratoria (sezione 7.5);
- l'equazione di Hodgkin–Huxley (sezione 7.10).

**È sempre la stessa procedura**. Una volta padroneggiata, costruisci un
modello fisiologico in pochi minuti partendo dallo schema circuitale.


### 2.0bis.16 Riepilogo intuitivo (cheat sheet a 5 righe)

| Grandezza | Cosa è | Analogia idraulica | In fisiologia |
|---|---|---|---|
| Carica $Q$ | "quantità di elettricità" | volume d'acqua | ioni accumulati al di qua/là della membrana |
| Corrente $I = \dot Q$ | flusso di cariche | portata d'acqua | flusso ionico transmembrana |
| Tensione $V$ | dislivello energetico | dislivello d'acqua | $V_m$ = potenziale di membrana |
| Resistenza $R$ | "ostacolo" al flusso | strozzatura | resistenza al passaggio ionico |
| Conduttanza $g=1/R$ | "facilità" di flusso | apertura | quanti canali aperti |
| Capacitore $C$ | serbatoio di carica | secchio elastico | doppio strato lipidico |

| Legge | Formula | Significato |
|---|---|---|
| Ohm | $V = RI$ | tensione = resistenza × corrente |
| Capacitore | $I = C\,\dot V$ | la corrente cambia la tensione |
| KCL | $\sum I_{in} = \sum I_{out}$ | la carica non si accumula nei nodi |
| KVL | $\sum V_{maglia} = 0$ | la tensione è una funzione di stato |
| Driving force | $I_X = g_X(V_m - E_X)$ | corrente di un canale ionico |

Con queste **5 righe + 5 formule** sei pronto/a per qualunque modello
elettrico del notebook.


### Schemi pedagogici dei componenti base

Lavoriamo costruendoci i **simboli circuitali** direttamente in matplotlib,
così non dipendiamo da figure esterne e ogni schema è riproducibile.


In [ ]:
# =============================================================================
# DISEGNATORI DI SIMBOLI ELETTROTECNICI ELEMENTARI (matplotlib only)
# =============================================================================
# Una libreria minima per disegnare resistori, capacitori, generatori, terra,
# fili e annotazioni: la useremo per tutti i circuiti del notebook.
# =============================================================================

def _line(ax, p0, p1, **kw):
    ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color=kw.get("color", "black"),
            lw=kw.get("lw", 1.6))

def wire(ax, *points, color="black", lw=1.6):
    for a, b in zip(points[:-1], points[1:]):
        _line(ax, a, b, color=color, lw=lw)

def resistor(ax, p0, p1, label=None, sub=None):
    """Resistore zigzag tra p0 e p1, con etichetta sopra."""
    p0, p1 = np.array(p0, float), np.array(p1, float)
    d = p1 - p0; L = np.linalg.norm(d); u = d / L; n = np.array([-u[1], u[0]])
    # Pad iniziale/finale
    pad = 0.18 * L
    a, b = p0 + u*pad, p1 - u*pad
    # 6 spigoli zigzag
    pts = [a]
    seg = (b - a)
    for k in range(1, 7):
        s = a + seg * k/7
        s = s + (0.10*L) * n * (1 if k % 2 == 1 else -1)
        pts.append(s)
    pts.append(b)
    pts = np.array(pts)
    ax.plot(pts[:, 0], pts[:, 1], color="black", lw=1.6)
    _line(ax, p0, a); _line(ax, b, p1)
    if label is not None:
        mid = (p0 + p1) / 2 + 0.18 * L * n
        ax.text(mid[0], mid[1], label, ha="center", va="center", fontsize=11)
    if sub is not None:
        mid = (p0 + p1) / 2 - 0.20 * L * n
        ax.text(mid[0], mid[1], sub, ha="center", va="center", fontsize=9, color="#555")

def capacitor(ax, p0, p1, label=None, sub=None, gap=0.10):
    """Due piastre parallele tra p0 e p1."""
    p0, p1 = np.array(p0, float), np.array(p1, float)
    d = p1 - p0; L = np.linalg.norm(d); u = d / L; n = np.array([-u[1], u[0]])
    a = p0 + u * (L/2 - gap*L/2)
    b = p1 - u * (L/2 - gap*L/2)
    _line(ax, p0, a); _line(ax, b, p1)
    plate = 0.18 * L
    _line(ax, a - n*plate, a + n*plate)
    _line(ax, b - n*plate, b + n*plate)
    if label is not None:
        mid = (p0 + p1) / 2 + 0.30 * L * n
        ax.text(mid[0], mid[1], label, ha="center", va="center", fontsize=11)
    if sub is not None:
        mid = (p0 + p1) / 2 - 0.30 * L * n
        ax.text(mid[0], mid[1], sub, ha="center", va="center", fontsize=9, color="#555")

def vsource(ax, p0, p1, label="$V$", plus_up=True):
    """Generatore di tensione: cerchio con + e -."""
    p0, p1 = np.array(p0, float), np.array(p1, float)
    mid = (p0 + p1) / 2
    r = np.linalg.norm(p1 - p0) * 0.18
    ax.add_patch(Circle(mid, r, fill=False, lw=1.6))
    d = (p1 - p0) / np.linalg.norm(p1 - p0)
    _line(ax, p0, mid - d*r); _line(ax, mid + d*r, p1)
    sign_plus = mid + d*r*0.55
    sign_minus = mid - d*r*0.55
    if not plus_up:
        sign_plus, sign_minus = sign_minus, sign_plus
    ax.text(*sign_plus,  "+", ha="center", va="center", fontsize=10)
    ax.text(*sign_minus, "−", ha="center", va="center", fontsize=10)
    n = np.array([-d[1], d[0]])
    ax.text(*(mid + n*r*2.2), label, ha="center", va="center", fontsize=11)

def isource(ax, p0, p1, label="$I$"):
    """Generatore di corrente: cerchio con freccia interna."""
    p0, p1 = np.array(p0, float), np.array(p1, float)
    mid = (p0 + p1) / 2
    r = np.linalg.norm(p1 - p0) * 0.18
    ax.add_patch(Circle(mid, r, fill=False, lw=1.6))
    d = (p1 - p0) / np.linalg.norm(p1 - p0)
    _line(ax, p0, mid - d*r); _line(ax, mid + d*r, p1)
    arr = FancyArrowPatch(mid - d*r*0.7, mid + d*r*0.7,
                          arrowstyle="->", mutation_scale=14, lw=1.4)
    ax.add_patch(arr)
    n = np.array([-d[1], d[0]])
    ax.text(*(mid + n*r*2.2), label, ha="center", va="center", fontsize=11)

def ground(ax, p, size=0.10):
    """Simbolo di terra."""
    p = np.array(p, float)
    _line(ax, p, p + np.array([0, -size]))
    for k, w in enumerate([1.0, 0.7, 0.45]):
        y = p[1] - size - 0.04 * k
        _line(ax, [p[0] - size*w*0.7, y], [p[0] + size*w*0.7, y])

def node_label(ax, p, label, offset=(0.05, 0.05), color="#1f4e79"):
    p = np.array(p)
    ax.plot(*p, "o", color=color, markersize=4.5, zorder=4)
    ax.text(p[0] + offset[0], p[1] + offset[1], label, fontsize=10, color=color)

print("Libreria di simboli circuitali caricata: wire, resistor, capacitor, vsource, isource, ground, node_label.")


### Catalogo dei componenti base

Disegniamo in una sola figura i mattoni elementari che ricomporremo nei
circuiti più complessi.


In [ ]:
# Pannello con i componenti base: resistore, capacitore, V-source, I-source, ground
fig, ax = plt.subplots(figsize=(11.0, 5.5))
ax.set_xlim(0, 11); ax.set_ylim(0, 5.5); ax.set_aspect("equal"); ax.axis("off")

# Resistore
resistor(ax, (0.5, 4.5), (3.0, 4.5), label="$R$", sub="resistenza")
ax.text(1.75, 5.05, "Resistore", ha="center", fontsize=11, fontweight="bold")
ax.text(1.75, 3.95, "$V=R\\,I$", ha="center", fontsize=10, color=COL["main"])

# Capacitore
capacitor(ax, (4.0, 4.5), (6.5, 4.5), label="$C$", sub="capacità / compliance")
ax.text(5.25, 5.05, "Capacitore", ha="center", fontsize=11, fontweight="bold")
ax.text(5.25, 3.95, "$I=C\\,\\dot V$", ha="center", fontsize=10, color=COL["main"])

# Generatore di tensione
vsource(ax, (7.0, 4.5), (10.0, 4.5), label="$V_s$")
ax.text(8.5, 5.05, "Generatore di tensione", ha="center", fontsize=11, fontweight="bold")
ax.text(8.5, 3.65, "tensione imposta", ha="center", fontsize=10, color=COL["main"])

# Generatore di corrente
isource(ax, (0.5, 1.5), (3.0, 1.5), label="$I_s$")
ax.text(1.75, 2.05, "Generatore di corrente", ha="center", fontsize=11, fontweight="bold")
ax.text(1.75, 0.85, "corrente imposta", ha="center", fontsize=10, color=COL["main"])

# Filo + terra
wire(ax, (4.0, 1.5), (6.5, 1.5))
ground(ax, (5.25, 1.5))
ax.text(5.25, 2.05, "Filo + terra", ha="center", fontsize=11, fontweight="bold")
ax.text(5.25, 0.85, "nodo a potenziale 0", ha="center", fontsize=10, color=COL["main"])

# Nodo
ax.plot(8.0, 1.5, "o", color="black", markersize=8)
ax.text(8.0, 2.05, "Nodo", ha="center", fontsize=11, fontweight="bold")
ax.text(8.0, 0.85, "punto di giunzione", ha="center", fontsize=10, color=COL["main"])

ax.set_title("Componenti elettrotecnici elementari", fontsize=13)
plt.tight_layout(); plt.show()


## 2.2 Le leggi di Kirchhoff

Le leggi di Kirchhoff sono semplici conservazioni:

- **Kirchhoff delle correnti (KCL)** = conservazione della carica nei nodi:
  $$\sum_{k\text{ entranti}} I_k = \sum_{k\text{ uscenti}} I_k.$$
- **Kirchhoff delle tensioni (KVL)** = il potenziale è una funzione di stato:
  $$\sum_{k\text{ lungo una maglia}} V_k = 0.$$

In modellistica fisiologica, KCL **è** un bilancio: di corrente di membrana,
di flusso di sangue, di flusso d'aria. Disegniamole esplicitamente.


In [ ]:
# KCL e KVL: due schemi pedagogici
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
ax.set_xlim(-1.5, 4); ax.set_ylim(-1.5, 2); ax.set_aspect("equal"); ax.axis("off")

# Nodo centrale, 3 entranti, 1 uscente
node = (1.5, 0.5)
ax.plot(*node, "o", color="black", markersize=10, zorder=5)
draw_arrow(ax, (-1.2, 1.5), (node[0]-0.3, node[1]+0.18), label="$I_1$",
           color=COL["main"], offset=(-0.2, 0.1))
draw_arrow(ax, (-1.2, -1.0), (node[0]-0.3, node[1]-0.18), label="$I_2$",
           color=COL["main"], offset=(-0.2, -0.2))
draw_arrow(ax, (1.5, 2.0), (node[0], node[1]+0.25), label="$I_3$",
           color=COL["main"], offset=(0.15, 0))
draw_arrow(ax, (node[0]+0.25, node[1]), (3.8, 0.4), label="$I_4$",
           color=COL["accent"], offset=(0.15, 0.1))
ax.set_title("KCL: $I_1+I_2+I_3 = I_4$", fontsize=12)
ax.text(1.5, -1.3, "Le correnti entranti pareggiano quelle uscenti.",
        ha="center", fontsize=10, color="#444")

ax = axes[1]
ax.set_xlim(-0.5, 4); ax.set_ylim(-0.5, 3); ax.set_aspect("equal"); ax.axis("off")
# Maglia rettangolare
wire(ax, (0.5, 0.5), (3.5, 0.5), (3.5, 2.5), (0.5, 2.5), (0.5, 0.5))
# Componenti sui lati
vsource(ax, (0.5, 1.0), (0.5, 2.0), label=r"$V_s$")
resistor(ax, (1.0, 2.5), (2.5, 2.5), label=r"$R_1$")
resistor(ax, (3.0, 2.5), (3.5, 1.5), label=r"$R_2$")  # diagonale stilizzata? No
# Replico in orizzontale piu' chiaro
ax.clear()
ax.set_xlim(-0.5, 5); ax.set_ylim(-0.5, 3); ax.set_aspect("equal"); ax.axis("off")
# Maglia
wire(ax, (0.5, 0.5), (4.5, 0.5), (4.5, 2.5), (0.5, 2.5), (0.5, 0.5))
vsource(ax, (0.5, 1.0), (0.5, 2.0), label=r"$V_s$")
resistor(ax, (1.0, 2.5), (2.3, 2.5), label=r"$R_1$", sub=r"$V_1=R_1 I$")
resistor(ax, (2.7, 2.5), (4.0, 2.5), label=r"$R_2$", sub=r"$V_2=R_2 I$")
# Frecce di corrente: senso orario
draw_arrow(ax, (1.0, 1.5), (1.0, 2.45), color=COL["accent"], mutation_scale=10)
draw_arrow(ax, (4.0, 2.5), (4.5, 2.5), color=COL["accent"], mutation_scale=10)
draw_arrow(ax, (4.5, 1.5), (4.5, 0.55), color=COL["accent"], mutation_scale=10)
draw_arrow(ax, (3.0, 0.5), (1.0, 0.5), color=COL["accent"], mutation_scale=10)
ax.text(2.5, 1.5, "$I$ (senso orario)", ha="center", color=COL["accent"], fontsize=10)
ax.set_title("KVL: $V_s = V_1 + V_2 = (R_1+R_2)\\,I$", fontsize=12)

plt.tight_layout(); plt.show()


## 2.3 Il circuito RC: il "mattone" della fisiologia

Il **circuito RC** è la chiave per leggere quasi tutti i modelli del notebook:

- **membrana cellulare passiva** = $C_m$ in parallelo a $g_L$ (leak);
- **compartimento vascolare** = $C$ (compliance) in parallelo a $1/R$
  (resistenza vascolare);
- **alveolo** = $C$ (compliance polmonare) in serie a $R$ (resistenza
  delle vie aeree).

### Derivazione dal bilancio di carica/corrente

Consideriamo un resistore $R$ in **parallelo** con un capacitore $C$,
con un ingresso di corrente esterna $I_{ext}(t)$ iniettato al nodo. Sia
$V$ la tensione del nodo rispetto a terra. KCL al nodo:

$$
\underbrace{I_{ext}(t)}_{\text{iniettata}}
=\underbrace{\frac{V}{R}}_{\text{nel resistore}}
+\underbrace{C\frac{dV}{dt}}_{\text{nel capacitore}}.
$$

Riarrangiando:

$$
\boxed{\;C\frac{dV}{dt} = -\frac{V}{R} + I_{ext}(t)\;}
$$

Soluzione del caso $I_{ext}=$ costante = $I_0$ con $V(0)=V_0$:

$$
V(t)=V_\infty + (V_0 - V_\infty)\,e^{-t/\tau},\qquad
V_\infty = R I_0,\qquad \tau=RC.
$$

- $V_\infty$ è il valore **a regime** (asintoto);
- $\tau$ è la **costante di tempo**: tempo per coprire $\approx 63\%$ del salto.

### Significato delle costanti

| Quantità | Definizione | In membrana passiva |
|---|---|---|
| $\tau$ | $RC$ | $C_m/g_L$ = costante di tempo di membrana |
| $V_\infty$ | $R\,I_{ext}$ | $E_L + I_{ext}/g_L$ |

Quindi se conosci $R$ e $C$ (o equivalentemente $g_L$ e $C_m$), conosci sia
**quanto velocemente** sia **a quale livello** il sistema risponde a una
forzante costante.


In [ ]:
# Schema del circuito RC di membrana, fatto a mano
fig, ax = plt.subplots(figsize=(9.5, 5.0))
ax.set_xlim(-0.5, 7.5); ax.set_ylim(-0.5, 4.5); ax.set_aspect("equal"); ax.axis("off")

# Linea superiore (nodo "interno", V_m)
wire(ax, (1.0, 3.5), (6.5, 3.5))
# Linea inferiore (riferimento extracellulare, terra)
wire(ax, (1.0, 0.5), (6.5, 0.5))
ground(ax, (3.75, 0.5))

# Generatore di corrente esterna a sinistra (iniezione)
wire(ax, (1.0, 3.5), (1.0, 2.5))
isource(ax, (1.0, 2.5), (1.0, 1.5), label=r"$I_{ext}$")
wire(ax, (1.0, 1.5), (1.0, 0.5))

# Conduttanza di leak g_L in serie con il reversal E_L (batteria)
wire(ax, (3.0, 3.5), (3.0, 3.0))
resistor(ax, (3.0, 3.0), (3.0, 2.0), label="$1/g_L$", sub="leak")
wire(ax, (3.0, 2.0), (3.0, 1.6))
vsource(ax, (3.0, 1.6), (3.0, 1.0), label="$E_L$", plus_up=False)
wire(ax, (3.0, 1.0), (3.0, 0.5))

# Capacitore Cm di membrana
wire(ax, (5.5, 3.5), (5.5, 2.5))
capacitor(ax, (5.5, 2.5), (5.5, 1.5), label="$C_m$")
wire(ax, (5.5, 1.5), (5.5, 0.5))

# Etichette nodi: dentro / fuori
ax.text(6.7, 3.5, "intracell. ($V_m$)", ha="left", va="center", fontsize=11, color=COL["main"])
ax.text(6.7, 0.5, "extracell. (0)", ha="left", va="center", fontsize=11, color=COL["grey"])

# Equazione esplicita
ax.text(3.5, 4.2,
        r"KCL al nodo $V_m$:  "
        r"$\;I_{ext} = g_L (V_m - E_L) + C_m \dot V_m$",
        ha="center", fontsize=12, color=COL["main"])

ax.set_title("Circuito equivalente di una membrana passiva (RC + reversal)",
             fontsize=12)
plt.tight_layout(); plt.show()


**Come leggere lo schema** — l'interno della cellula (linea superiore) è
collegato all'esterno (terra) attraverso due percorsi paralleli:

1. una **conduttanza** $g_L$ (resistenza $1/g_L$) in serie a una **batteria**
   $E_L$: rappresenta il flusso ionico passivo verso il potenziale di
   inversione del leak;
2. un **capacitore** $C_m$: rappresenta la separazione di carica attraverso
   il doppio strato lipidico.

L'**iniezione esterna** $I_{ext}$ (elettrodo dello sperimentatore o sinapsi)
entra al nodo intracellulare.

**Dallo schema all'equazione** — KCL al nodo $V_m$ con segni: corrente entrante
$I_{ext}$ uguale alla somma delle correnti uscenti attraverso il leak
$g_L(V_m-E_L)$ e attraverso la capacità $C_m\dot V_m$.

**Variabili di stato**: $V_m$.
**Ingresso**: $I_{ext}$.
**Parametri**: $C_m$, $g_L$, $E_L$.

**Ipotesi implicite**: linearità del leak (no rettifica), assenza di
correnti voltaggio-dipendenti, geometria a parametri concentrati ("punto"
elettrico, non cavo).


In [ ]:
# Simulazione del circuito RC di membrana: 3 metodi a confronto
C_m = 1.0     # uF/cm^2
g_L = 0.3    # mS/cm^2 -> tau = C_m / g_L
E_L = -65.0   # mV
I_ext_amp = 5.0  # uA/cm^2, gradino accesso a t=2 ms
T = 30.0      # ms

def I_ext(t):
    return I_ext_amp if t >= 2.0 else 0.0

def f(t, V):
    return np.array([(I_ext(t) - g_L * (V[0] - E_L)) / C_m])

# Soluzione "a mano" lineare con esatta della risposta a tratti
t_grid = np.linspace(0, T, 600)
V_exact = np.zeros_like(t_grid)
V_exact[t_grid < 2.0] = E_L                 # prima del gradino: equilibrio E_L
tau = C_m / g_L
V_inf = E_L + I_ext_amp / g_L
mask = t_grid >= 2.0
V_exact[mask] = V_inf + (E_L - V_inf) * np.exp(-(t_grid[mask] - 2.0) / tau)

# solve_ivp
sol = solve_ivp(f, (0, T), [E_L], t_eval=t_grid, rtol=1e-8, atol=1e-10)
V_num = sol.y[0]

# Eulero esplicito
def euler_explicit_scalar(f, x0, T, dt):
    n = int(np.round(T/dt)) + 1
    t = np.linspace(0, T, n)
    x = np.zeros(n); x[0] = x0
    for k in range(n-1):
        x[k+1] = x[k] + dt * f(t[k], np.array([x[k]]))[0]
    return t, x
t_e, V_e = euler_explicit_scalar(f, E_L, T, dt=0.3)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_grid, V_exact, lw=2.4, color=COL["main"], label="soluzione esatta")
ax.plot(sol.t, V_num, "--", lw=1.2, color=COL["ok"], label="solve_ivp")
ax.plot(t_e, V_e, "o", color=COL["accent"], ms=4.5, label="Eulero esplicito $\\Delta t=0.3$ ms")
ax.axhline(E_L, color="gray", ls=":", label="$E_L$")
ax.axhline(V_inf, color="black", ls=":", label="$V_\\infty=E_L+I_{ext}/g_L$")
ax.axvline(2.0, color="gray", ls="--", alpha=0.5)
ax.annotate("inizio gradino", xy=(2.0, -64), xytext=(4.0, -55),
            arrowprops=dict(arrowstyle="->", color="gray"))
# Tau in evidenza
ax.annotate(r"$\tau=C_m/g_L\approx$" + f"{tau:.2f} ms",
            xy=(2.0 + tau, V_inf + (E_L - V_inf)*np.exp(-1)),
            xytext=(2.0 + tau + 2, V_inf - 5),
            arrowprops=dict(arrowstyle="->", color="black"))
ax.set_xlabel("tempo [ms]"); ax.set_ylabel("$V_m$ [mV]")
ax.set_title("Risposta di membrana passiva a un gradino di corrente")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

print(f"tau = {tau:.3f} ms,  V_infty = {V_inf:.2f} mV")


**Cosa mostra il grafico** — la membrana parte a $E_L=-65$ mV.
Quando a $t=2$ ms entra una corrente di $5\;\mu\text{A/cm}^2$, il potenziale
**non salta**: cresce esponenzialmente verso $V_\infty=E_L+I_{ext}/g_L\approx-48.3$ mV
con costante di tempo $\tau=C_m/g_L\approx 3.33$ ms.

**Come si interpreta** — il capacitore *frena*: il potenziale non può cambiare
istantaneamente perché serve corrente per caricare il doppio strato. La
conduttanza di leak *tira* sempre verso $E_L$.

**Cosa cambia se varia $g_L$** — aumentando $g_L$ diminuiscono $\tau$ **e**
$V_\infty$: la membrana risponde più velocemente ma resta più vicina al
potenziale di riposo. Diminuendo $g_L$ la membrana diventa lenta e si
allontana di più dal riposo per la stessa corrente iniettata.

**Perché è importante** — la stessa equazione descrive un neurone passivo,
un compartimento vascolare con singola compliance, un alveolo con compliance
e resistenza. Il "RC" è il mattone universale.


### Stabilità numerica di Eulero esplicito sul circuito RC

Per $\dot V = -V/\tau$, Eulero esplicito dà
$V_{k+1}=V_k(1-\Delta t/\tau)$. Stabile se $|1-\Delta t/\tau|<1$, cioè
$\Delta t < 2\tau$. **Oltre** quel limite la soluzione numerica oscilla e
diverge, anche se quella esatta decade tranquillamente.


In [ ]:
# Verifica sperimentale del limite di stabilita' di Eulero esplicito
tau = 3.0  # ms
T = 30.0
fig, ax = plt.subplots(figsize=(10, 5))
t_ref = np.linspace(0, T, 400); V_ref = np.exp(-t_ref/tau)
ax.plot(t_ref, V_ref, "k-", lw=2.2, label=r"esatta $e^{-t/\tau}$")

for dt, c in zip([1.0, 2.5, 5.0, 7.0],
                 [COL["ok"], COL["warn"], COL["accent"], COL["extra"]]):
    n = int(T/dt) + 1
    t = np.linspace(0, T, n); V = np.zeros(n); V[0] = 1.0
    for k in range(n-1):
        V[k+1] = V[k] * (1 - dt/tau)
    label = (f"$\\Delta t={dt}$ "
             + ("(stabile)" if dt < 2*tau else "(instabile)"))
    ax.plot(t, V, "o-", color=c, ms=4, lw=1.2, label=label)

ax.axhline(0, color="gray", lw=0.6)
ax.set_xlabel("tempo [ms]"); ax.set_ylabel("V (normalizzato)")
ax.set_title(r"Eulero esplicito su $\dot V=-V/\tau$, $\tau=3$ ms: stabile sse $\Delta t<2\tau=6$ ms")
ax.legend(); plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — fino a $\Delta t < 2\tau$ Eulero esplicito è
stabile; per $\Delta t > 2\tau$ inizia a oscillare con segno alternato e
diverge. La soglia $2\tau$ esce esatta dalla condizione $|1-\Delta t/\tau|<1$.

**Perché è importante** — anche per **modelli stabili** un cattivo $\Delta t$
fa esplodere la simulazione. È esattamente quello che succede a Hodgkin–Huxley
se si usa Eulero esplicito con $\Delta t > 0.025$ ms.


### Visualizzazione "ricca" della carica e scarica di un capacitore

Aggiungiamo una figura che mostra esplicitamente, sullo stesso pannello,
*tensione*, *corrente* e *potenza istantanea*. Aiuta a internalizzare il
legame "$I=C\,dV/dt$" e a vedere che durante la carica la potenza
istantanea $P=VI$ è prima alta e poi va a zero quando $V=V_\infty$.


In [ ]:
# Carica e scarica di un capacitore RC, viste in 3 grandezze
R, C = 1.0e3, 100e-6    # 1 kOhm, 100 uF -> tau = 0.1 s
V_in = 5.0              # tensione di sorgente
tau = R*C
T = 0.5

# Carica: V(t) = V_in (1 - e^{-t/tau})
t1 = np.linspace(0, T, 400)
V_chg = V_in * (1 - np.exp(-t1/tau))
I_chg = (V_in - V_chg) / R
P_chg = V_chg * I_chg

# Scarica: dopo aver caricato, V(t) = V_in e^{-(t-T)/tau}
t2 = np.linspace(T, 2*T, 400)
V_dsc = V_in * np.exp(-(t2-T)/tau)
I_dsc = -V_dsc / R
P_dsc = V_dsc * (-I_dsc)  # potenza dissipata sul resistore

t = np.concatenate([t1, t2])
V = np.concatenate([V_chg, V_dsc])
I = np.concatenate([I_chg, I_dsc])
P = np.concatenate([P_chg, P_dsc])

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
axes[0].plot(t*1000, V, color=COL["main"], lw=2)
axes[0].axhline(V_in, color="gray", ls=":")
axes[0].axvline(T*1000, color="gray", ls=":")
axes[0].set_ylabel("V [V]")
axes[0].set_title("Carica (0–0.5 s) e scarica (0.5–1.0 s) di un RC")
axes[0].annotate(r"$V_\infty = V_{in}$", xy=(50, V_in), xytext=(80, 4.0),
                 arrowprops=dict(arrowstyle="->"))

axes[1].plot(t*1000, I*1e3, color=COL["accent"], lw=2)
axes[1].axhline(0, color="gray", lw=0.6)
axes[1].axvline(T*1000, color="gray", ls=":")
axes[1].set_ylabel("I [mA]")
axes[1].set_title("Corrente: in carica positiva, in scarica negativa")

axes[2].plot(t*1000, P*1e3, color=COL["ok"], lw=2)
axes[2].axhline(0, color="gray", lw=0.6)
axes[2].axvline(T*1000, color="gray", ls=":")
axes[2].set_xlabel("tempo [ms]"); axes[2].set_ylabel("P [mW]")
axes[2].set_title("Potenza istantanea $P=VI$ dissipata sul resistore")
plt.tight_layout(); plt.show()

print(f"tau = R*C = {tau*1000:.1f} ms")
print(f"In carica, V raggiunge il 63% di V_in in tau = {tau*1000:.1f} ms")
print(f"Dopo 5 tau = {5*tau*1000:.1f} ms, V è al 99.3% del valore finale")


**Cosa mostra il grafico** — V cresce e poi cala in modo esponenziale.
La corrente è massima nei primi istanti e tende a zero quando V raggiunge
$V_\infty$ (perché allora $I = (V_{in}-V)/R=0$). La potenza dissipata sul
resistore è alta all'inizio, va a zero quando il sistema raggiunge
l'equilibrio. **Stessa identica matematica** descrive il riempimento di
un alveolo o il riempimento di un compartimento vascolare.

## 2.4 Analogia idraulico–elettrica e compartimenti fisiologici

L'analogia che sta dietro a tutta la fisiologia dei sistemi è:

| Idraulico/fluidico | Elettrico | Membrana |
|---|---|---|
| pressione $P$ | tensione $V$ | potenziale $V_m$ |
| flusso $Q$ | corrente $I$ | corrente ionica |
| resistenza $R$: $P=R\,Q$ | $V=R\,I$ | $V=I/g$ |
| compliance $C=\Delta V/\Delta P$ | capacità $C=Q/V$ | $C_m$ |
| volume $V_{vol}$ | carica $Q$ | carica accumulata |
| pompa cardiaca | generatore | "patch clamp" come $I$/$V$ source |
| valvole | diodi | canali rettificanti |

Un **compartimento vascolare** è quindi nient'altro che un RC: pressione $P$
funziona come tensione, flusso $Q$ come corrente, e la compliance $C$ del
vaso accumula volume come un capacitore accumula carica.


In [ ]:
# Schema dell'analogia: compartimento vascolare singolo (RC fisiologico)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# Sinistra: schema "idraulico"
ax = axes[0]; ax.set_xlim(0, 8); ax.set_ylim(-0.5, 5); ax.set_aspect("equal"); ax.axis("off")
# Disegno stilizzato di un serbatoio (compliance) con tubo in entrata e in uscita
tank = Rectangle((3, 1.0), 2.0, 2.4, fill=False, lw=1.8, ec=COL["main"])
ax.add_patch(tank)
ax.text(4.0, 3.7, r"compartimento ($C$)", ha="center", fontsize=11, color=COL["main"])
# Tubo in: simbolo "rubinetto" stilizzato
wire(ax, (0.5, 2.5), (3.0, 2.5), color=COL["accent"])
draw_arrow(ax, (1.0, 2.7), (2.0, 2.7), color=COL["accent"])
ax.text(1.5, 3.1, r"$Q_{in}$", color=COL["accent"], fontsize=11)
# Tubo out con strozzatura (resistenza)
wire(ax, (5.0, 2.5), (7.5, 2.5), color=COL["accent"])
# strozzatura
ax.plot([6.0, 6.5], [2.5, 2.5], color=COL["accent"], lw=4)
ax.text(6.25, 2.95, "$R$ (resistenza)", ha="center", fontsize=10, color=COL["accent"])
draw_arrow(ax, (6.8, 2.5), (7.4, 2.5), color=COL["accent"])
ax.text(7.0, 2.95, r"$Q_{out}=P/R$", color=COL["accent"], fontsize=10)
# Pressione interna
ax.text(4.0, 2.0, r"$P$", fontsize=14, color=COL["main"], ha="center")
ax.set_title("Analogia idraulica: compliance + resistenza")

# Destra: equivalente RC
ax = axes[1]; ax.set_xlim(-0.5, 7); ax.set_ylim(-0.5, 4.5); ax.set_aspect("equal"); ax.axis("off")
# Linee superiore/inferiore
wire(ax, (1.0, 3.5), (6.0, 3.5))
wire(ax, (1.0, 0.5), (6.0, 0.5))
ground(ax, (3.5, 0.5))
# Iniezione di corrente Q_in
wire(ax, (1.0, 3.5), (1.0, 2.5))
isource(ax, (1.0, 2.5), (1.0, 1.5), label=r"$Q_{in}$")
wire(ax, (1.0, 1.5), (1.0, 0.5))
# Capacitore C in parallelo
wire(ax, (3.0, 3.5), (3.0, 2.5))
capacitor(ax, (3.0, 2.5), (3.0, 1.5), label="$C$")
wire(ax, (3.0, 1.5), (3.0, 0.5))
# Resistenza R verso terra
wire(ax, (5.5, 3.5), (5.5, 2.7))
resistor(ax, (5.5, 2.7), (5.5, 1.3), label="$R$")
wire(ax, (5.5, 1.3), (5.5, 0.5))
# Etichetta nodo P
ax.text(6.2, 3.5, "$P$", color=COL["main"], fontsize=14)
ax.text(2.0, 4.1, "$C\\,\\dot P = Q_{in} - P/R$", ha="center", fontsize=12, color=COL["main"])
ax.set_title("Equivalente elettrico (RC)")

plt.tight_layout(); plt.show()


**Come leggere lo schema** — a sinistra il modello idraulico: un
serbatoio elastico (compliance $C$) viene riempito con flusso $Q_{in}$ e
si svuota attraverso una resistenza $R$. La pressione interna $P$ è
analoga al potenziale, $Q$ al flusso di corrente.

A destra l'**equivalente circuitale**: la compliance $C$ è il capacitore;
la resistenza $R$ è il resistore; il flusso entrante $Q_{in}$ è una sorgente
di corrente; la pressione $P$ è la tensione del nodo.

**Dall'analogia all'equazione** — KCL al nodo: $Q_{in} = C\dot P + P/R$,
identica all'equazione del circuito RC con corrente esterna.

**Importante**: useremo *esattamente questa* struttura nei modelli
cardiovascolare (compartimenti arterioso, venoso, atriale) e respiratorio.


## 2.5 Combinazioni serie/parallelo e divisori

**Resistenze in serie**: $R_{eq}=R_1+R_2$. **Conduttanze in serie**:
$1/G_{eq}=1/G_1+1/G_2$.

**Resistenze in parallelo**: $1/R_{eq}=1/R_1+1/R_2$. **Conduttanze in
parallelo**: $G_{eq}=G_1+G_2$.

**Capacitori in parallelo**: $C_{eq}=C_1+C_2$ (le superfici si sommano).
**Capacitori in serie**: $1/C_{eq}=1/C_1+1/C_2$.

### Divisori

- **Divisore di tensione** (serie): $V_2 = V \cdot R_2/(R_1+R_2)$;
- **Divisore di corrente** (parallelo): $I_2 = I \cdot R_1/(R_1+R_2)$
  (la corrente preferisce la via meno resistiva).

In fisiologia ad esempio la distribuzione del **flusso aereo** tra bronchi
in parallelo segue esattamente il divisore di corrente.


In [ ]:
# Schema dei divisori di tensione e di corrente
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Divisore di tensione
ax = axes[0]; ax.set_xlim(-0.5, 6); ax.set_ylim(-0.5, 5); ax.set_aspect("equal"); ax.axis("off")
wire(ax, (0.5, 0.5), (0.5, 4.5))
wire(ax, (0.5, 4.5), (4.0, 4.5))
wire(ax, (4.0, 4.5), (4.0, 0.5))
wire(ax, (4.0, 0.5), (0.5, 0.5))
ground(ax, (0.5, 0.5))
vsource(ax, (0.5, 1.5), (0.5, 3.5), label=r"$V$")
resistor(ax, (4.0, 4.2), (4.0, 2.5), label=r"$R_1$")
resistor(ax, (4.0, 2.0), (4.0, 0.8), label=r"$R_2$")
ax.plot(4.0, 2.25, "o", color="black", ms=6)
ax.annotate(r"$V_2 = V\,\dfrac{R_2}{R_1+R_2}$", xy=(4.0, 2.25),
            xytext=(4.4, 2.5), fontsize=11, color=COL["main"],
            arrowprops=dict(arrowstyle="->", color=COL["main"]))
ax.set_title("Divisore di tensione (serie)")

# Divisore di corrente
ax = axes[1]; ax.set_xlim(-0.5, 7); ax.set_ylim(-0.5, 5); ax.set_aspect("equal"); ax.axis("off")
wire(ax, (0.5, 3.0), (1.5, 3.0))
isource(ax, (0.5, 1.0), (0.5, 3.0), label=r"$I$")
wire(ax, (1.5, 3.0), (5.5, 3.0))
wire(ax, (1.5, 0.5), (5.5, 0.5))
wire(ax, (0.5, 0.5), (1.5, 0.5))
wire(ax, (0.5, 1.0), (0.5, 0.5))
ground(ax, (3.0, 0.5))
# Due rami paralleli
resistor(ax, (3.0, 3.0), (3.0, 0.5), label=r"$R_1$")
resistor(ax, (5.0, 3.0), (5.0, 0.5), label=r"$R_2$")
draw_arrow(ax, (3.0, 3.4), (3.0, 2.0), color=COL["accent"], mutation_scale=10)
draw_arrow(ax, (5.0, 3.4), (5.0, 2.0), color=COL["accent"], mutation_scale=10)
ax.text(3.0, 3.6, r"$I_1$", color=COL["accent"], ha="center")
ax.text(5.0, 3.6, r"$I_2$", color=COL["accent"], ha="center")
ax.text(4.0, -0.2,
        r"$I_1 = I\,\dfrac{R_2}{R_1+R_2}$,   $I_2 = I\,\dfrac{R_1}{R_1+R_2}$",
        ha="center", fontsize=11, color=COL["main"])
ax.set_title("Divisore di corrente (parallelo)")
plt.tight_layout(); plt.show()


## 2.6 Perché in elettrofisiologia si preferisce la conduttanza

In elettrotecnica generale è frequente parlare di **resistenza** $R$. In
elettrofisiologia delle membrane si preferisce la **conduttanza**
$g=1/R$. La ragione è fisica:

1. La conduttanza è proporzionale al *numero di canali aperti* nella
   membrana. Quindi $g$ ha un significato microscopico diretto: più canali
   aperti = più $g$.
2. Le correnti ioniche si scrivono come **driving force** moltiplicata per
   la conduttanza:
   $$I_i = g_i\,(V_m - E_i),$$
   dove $E_i$ è il potenziale di inversione (Nernst). Questa forma rende
   naturale dire "se $V_m > E_i$ la corrente entra/esce con segno definito".
3. Più canali in parallelo → conduttanze che si sommano (additività diretta).
4. Una conduttanza che si "apre o chiude" $g(V_m,t)$ permette di modellare i
   canali voltaggio-dipendenti come Hodgkin–Huxley.

In tutto il notebook per le membrane useremo $g$ e per i sistemi
cardiovascolare/respiratorio useremo $R$, coerentemente con la letteratura.


# Parte 3 — Teoria dei modelli matematici

Un **modello matematico** di un sistema biologico è un insieme di equazioni
che descrivono in modo *parsimonioso* l'evoluzione di una collezione di
**variabili di stato** in risposta a **ingressi** noti, e che produce delle
**uscite** misurabili. Più precisamente:

$$
\dot{\mathbf{x}}=\mathbf{f}(\mathbf{x},\mathbf{u},\boldsymbol{\theta}),\qquad
\mathbf{y}=\mathbf{h}(\mathbf{x},\mathbf{u},\boldsymbol{\theta}).
$$

| Simbolo | Significato |
|---|---|
| $\mathbf{x}\in\mathbb{R}^n$ | **stato** (variabili di memoria del sistema) |
| $\mathbf{u}\in\mathbb{R}^m$ | **ingressi** (controlli, perturbazioni) |
| $\mathbf{y}\in\mathbb{R}^p$ | **uscite** (ciò che misuriamo) |
| $\boldsymbol{\theta}\in\mathbb{R}^q$ | **parametri** (caratteristiche fisse del sistema) |

## 3.1 Costruzione del modello: i 7 passi

1. **Domanda scientifica chiara** — cosa vuoi spiegare? (es. "perché un
   pasto ipotonico abbassa la natremia").
2. **Scelta delle variabili di stato** — il minimo insieme di variabili la
   cui conoscenza al tempo $t$ + l'ingresso futuro determina il futuro.
3. **Bilanci** — di massa, di carica, di volume, di momento, di energia.
   Quasi sempre il modello viene da $\frac{d(\text{contenuto})}{dt}=
   \text{ingresso}-\text{uscita}$.
4. **Leggi costitutive** — come dipendono i flussi dagli stati (Ohm,
   Fick, Michaelis–Menten, Hill, Nernst, Hodgkin–Huxley).
5. **Ipotesi semplificative** — esplicitate (es. cellula puntiforme, gas
   ideali, miscelazione perfetta, geometria 1D).
6. **Scrittura in forma normale** — sistema del primo ordine
   $\dot{\mathbf{x}}=\mathbf{f}(\mathbf{x},\mathbf{u},\boldsymbol{\theta})$.
7. **Analisi**: equilibri, stabilità, sensitività; **validazione** contro dati;
   **uso** per predire scenari.

## 3.2 Identificabilità

Un modello è **strutturalmente identificabile** se i parametri si possono
distinguere a partire da dati di ingresso/uscita di **alta qualità** e
*lunghezza adeguata*. Esempi pratici:

- Se $y(t)=A\,e^{-kt}$ e misuri solo $y$ a tempi noti, $A$ e $k$ sono
  identificabili.
- Se invece $y(t)=A e^{-k_1 t}+A e^{-k_2 t}$ e $k_1, k_2$ sono molto vicini,
  l'**identificabilità pratica** (con rumore) si perde.

In modellistica fisiologica si dovrebbe sempre porre la domanda *"i miei
dati possono in linea di principio distinguere fra parametri diversi?"*
prima di calibrare.

## 3.3 Calibrazione e validazione

- **Calibrazione**: scegliere $\boldsymbol{\theta}$ che minimizza una funzione
  costo $J(\boldsymbol{\theta})=\sum_k \big(y_{\text{model}}(t_k)-y_{\text{data}}(t_k)\big)^2$.
  Strumento: `scipy.optimize.least_squares`, `scipy.optimize.minimize`.
- **Validazione**: usare *dati indipendenti* (non usati in calibrazione) per
  testare il modello. Il modello che si adatta perfettamente ai dati di
  calibrazione ma fallisce sui dati indipendenti è in **overfitting**.

## 3.4 Analisi di sensitività

La **sensitività locale** del modello a un parametro $\theta_j$ si misura
con la derivata $\partial \mathbf{x}/\partial \theta_j$ (oppure
$\partial \mathbf{y}/\partial \theta_j$):

$$
S_{ij}(t)=\frac{\partial x_i(t)}{\partial \theta_j}.
$$

Soddisfa essa stessa un sistema di ODE (le *equazioni di sensitività*),
ma per modelli piccoli basta perturbare numericamente
$\theta_j \to \theta_j(1+\epsilon)$ e osservare $\mathbf{x}$.

In analisi **globale** si campiona $\boldsymbol{\theta}$ in una scatola
e si studia statisticamente la variabilità dell'uscita (Sobol, Morris).


In [ ]:
# Esempio "demo" di calibrazione: stima di un decadimento esponenziale
from scipy.optimize import curve_fit

# "Dati sperimentali" simulati
np.random.seed(7)
t_data = np.linspace(0, 5, 25)
A_true, k_true = 3.2, 0.55
y_data = A_true * np.exp(-k_true * t_data) + 0.1 * np.random.randn(len(t_data))

# Modello
def model(t, A, k): return A * np.exp(-k * t)

# Calibrazione (least squares non lineare)
popt, pcov = curve_fit(model, t_data, y_data, p0=[1.0, 1.0])
A_hat, k_hat = popt
print(f"Veri:  A={A_true}, k={k_true}")
print(f"Stima: A={A_hat:.3f}, k={k_hat:.3f}")
print("Matrice di covarianza dei parametri stimati:\n", pcov)

# Validazione visiva
fig, ax = plt.subplots(figsize=(9, 4.6))
t_fine = np.linspace(0, 5, 200)
ax.plot(t_data, y_data, "o", color=COL["accent"], label="dati (con rumore)")
ax.plot(t_fine, model(t_fine, A_true, k_true), color="black", ls=":", lw=1.8, label="vero")
ax.plot(t_fine, model(t_fine, *popt), color=COL["main"], lw=2.0, label="modello calibrato")
ax.set_xlabel("t"); ax.set_ylabel("y")
ax.set_title("Calibrazione di un modello a 2 parametri vs dati rumorosi")
ax.legend(); plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — i pallini sono i dati rumorosi, la linea
puntinata è la curva "vera", la linea continua è quella stimata. La
calibrazione recupera $A,k$ con errore $\sim 5\%$.

**Come interpretare la matrice di covarianza** — la diagonale sono le
varianze stimate dei parametri. Termini fuori-diagonale: se sono grandi i
parametri sono correlati (un cambiamento in $A$ può essere compensato da un
cambiamento in $k$), il che è un segnale di possibile non identificabilità
pratica.

## 3.5 Limiti dei modelli in silico

Anche un modello calibrato bene resta una *semplificazione*. Limiti tipici:

- **assenza di eterogeneità**: il modello a 1 compartimento ignora che la
  cellula ha citosol + organelli, il vaso ha laminar/turbolento, ecc.;
- **linearizzazione attorno a un punto**: valida solo in un intorno;
- **parametri "lumped"**: assorbono molte non linearità;
- **costanti di tempo molto diverse**: rischio di stiffness;
- **identificabilità pratica**: con dati limitati la covarianza esplode.

Un modello è *utile* nella misura in cui spiega bene i fenomeni nel suo
**dominio di validità**, e questo dominio va sempre esplicitato.


# Parte 4 — Sistemi lineari LTI

Un sistema **lineare tempo-invariante (LTI)** ha la forma di stato

$$
\dot{\mathbf{x}}(t)=A\mathbf{x}(t)+B\mathbf{u}(t),\qquad
\mathbf{y}(t)=C\mathbf{x}(t)+D\mathbf{u}(t),
$$

con $A\in\mathbb{R}^{n\times n}$, $B\in\mathbb{R}^{n\times m}$,
$C\in\mathbb{R}^{p\times n}$, $D\in\mathbb{R}^{p\times m}$ **costanti**.

Anche se molti modelli fisiologici reali **non sono** lineari, due ragioni
li rendono cruciali:

1. **vicino agli equilibri** ogni sistema non lineare regolare *è* lineare
   (Hartman–Grobman);
2. esistono tecniche analitiche (autovalori, $e^{At}$, funzione di
   trasferimento, Nyquist) che danno **risposte chiuse** e intuizioni
   immediate.

Per ogni concetto di questa parte: **definizione**, **derivazione**,
**esempio scalare/matriciale**, **simulazione Python**, **grafico**.

## 4.1 Forma canonica di stato

| Matrice | Dimensione | Significato |
|---|---|---|
| $A$ | $n\times n$ | dinamica intrinseca dello stato |
| $B$ | $n\times m$ | come gli ingressi influenzano lo stato |
| $C$ | $p\times n$ | quali combinazioni di stato si misurano |
| $D$ | $p\times m$ | accoppiamento istantaneo ingresso→uscita |

**Espansione scalare** del termine $A\mathbf{x}+B\mathbf{u}$ per $n=3$, $m=2$:

$$
\begin{aligned}
\dot x_1 &= a_{11}x_1+a_{12}x_2+a_{13}x_3 + b_{11}u_1+b_{12}u_2,\\
\dot x_2 &= a_{21}x_1+a_{22}x_2+a_{23}x_3 + b_{21}u_1+b_{22}u_2,\\
\dot x_3 &= a_{31}x_1+a_{32}x_2+a_{33}x_3 + b_{31}u_1+b_{32}u_2.
\end{aligned}
$$

## 4.2 Moto libero e moto forzato

Il sistema LTI ammette **separazione**:

$$
\mathbf{x}(t)=\underbrace{e^{At}\mathbf{x}_0}_{\text{moto libero}}
+\underbrace{\int_0^t e^{A(t-\tau)}B\,\mathbf{u}(\tau)\,d\tau}_{\text{moto forzato}}.
$$

### Derivazione completa via fattore integrante

Dato $\dot{\mathbf{x}}=A\mathbf{x}+B\mathbf{u}$, moltiplichiamo a sinistra
per $e^{-At}$:

$$
e^{-At}\dot{\mathbf{x}}-e^{-At}A\mathbf{x}=e^{-At}B\mathbf{u}.
$$

Notiamo che $\frac{d}{dt}\big(e^{-At}\mathbf{x}\big)=e^{-At}\dot{\mathbf{x}}-
A e^{-At}\mathbf{x}=e^{-At}\dot{\mathbf{x}}-e^{-At}A\mathbf{x}$ (le matrici
$A$ ed $e^{-At}$ commutano). Quindi

$$
\frac{d}{dt}\big(e^{-At}\mathbf{x}\big)=e^{-At}B\mathbf{u}(t).
$$

Integrando da $0$ a $t$:

$$
e^{-At}\mathbf{x}(t)-\mathbf{x}(0)=\int_0^t e^{-A\tau}B\mathbf{u}(\tau)\,d\tau,
$$

e moltiplicando a sinistra per $e^{At}$ otteniamo la **formula della
variazione delle costanti**:

$$
\boxed{\;\mathbf{x}(t)=e^{At}\mathbf{x}_0+\int_0^t e^{A(t-\tau)}B\mathbf{u}(\tau)\,d\tau.\;}
$$

La matrice $\Phi(t)=e^{At}$ si chiama **matrice di transizione**: trasporta
lo stato da $0$ a $t$.


In [ ]:
# Massa-molla-smorzatore: moto libero + forzato, e^{At} vs solve_ivp
m, k, c = 1.0, 4.0, 0.6   # massa, rigidita', smorzamento
A = np.array([[0.0, 1.0],
              [-k/m, -c/m]])
B = np.array([0.0, 1.0/m])
x0 = np.array([1.0, 0.0])   # parto spostato di 1, fermo

# Moto libero
T = 10.0
t_grid = np.linspace(0, T, 600)
X_free = np.array([expm(A * t) @ x0 for t in t_grid]).T

# Moto forzato: u(t) = gradino di ampiezza F0
F0 = 2.0
def force(t): return F0 * (t >= 0.0)

def rhs(t, x): return A @ x + B * force(t)

sol_total = solve_ivp(rhs, (0, T), x0, t_eval=t_grid, rtol=1e-9, atol=1e-11)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax = axes[0]
ax.plot(t_grid, X_free[0], color=COL["main"], lw=2, label="$x_1$ libero ($e^{At}x_0$)")
ax.plot(t_grid, X_free[1], color=COL["accent"], lw=2, label="$x_2$ libero")
ax.set_ylabel("stato")
ax.legend(); ax.set_title("Moto LIBERO: solo condizione iniziale, $\\mathbf{u}=0$")

ax = axes[1]
ax.plot(t_grid, sol_total.y[0], color=COL["main"], lw=2, label="$x_1$ totale")
ax.plot(t_grid, sol_total.y[1], color=COL["accent"], lw=2, label="$x_2$ totale")
ax.axhline(F0/k, color="gray", ls=":", label="equilibrio forzato $F_0/k$")
ax.set_xlabel("tempo [s]"); ax.set_ylabel("stato")
ax.legend(); ax.set_title("Moto TOTALE: libero + forzato (ingresso a gradino $F_0$)")

plt.tight_layout(); plt.show()

print("Autovalori di A (parte reale = decadimento, parte imm. = pulsazione):",
      np.linalg.eigvals(A))


## 4.3 Funzione di trasferimento

Dalla formula $\mathbf{x}(t)=e^{At}\mathbf{x}_0+\int e^{A(t-\tau)}B\mathbf{u}\,d\tau$
e da $\mathbf{y}=C\mathbf{x}+D\mathbf{u}$, **trasformando di Laplace** con
$\mathbf{x}_0=\mathbf{0}$:

$$
sX(s)=AX(s)+BU(s),\qquad X(s)=(sI-A)^{-1}B\,U(s),
$$

quindi

$$
\boxed{\;G(s)=\frac{Y(s)}{U(s)}=C(sI-A)^{-1}B+D.\;}
$$

I **poli** di $G$ sono gli zeri di $\det(sI-A)$, cioè gli autovalori di $A$.
Gli **zeri** sono gli $s$ per cui $G(s)=0$ (per SISO).

### Stabilità BIBO

Un sistema LTI è **BIBO-stabile** (Bounded Input → Bounded Output) se e solo
se *tutti i suoi poli hanno parte reale negativa*. È la traduzione di
"tutti gli autovalori di $A$ a parte reale $<0$".


In [ ]:
# Funzione di trasferimento massa-molla-smorzatore tramite formula matriciale
m, k, c = 1.0, 4.0, 0.6
A = np.array([[0.0, 1.0],
              [-k/m, -c/m]])
B = np.array([[0.0], [1.0/m]])
C_mat = np.array([[1.0, 0.0]])
D_mat = np.array([[0.0]])

# G(s) campionata in piu' s
def G(s):
    return (C_mat @ np.linalg.inv(s*np.eye(2) - A) @ B + D_mat)[0, 0]

# Diagramma di Bode "manuale"
omega = np.logspace(-1, 1.5, 400)
H = np.array([G(1j*w) for w in omega])
mag_db = 20 * np.log10(np.abs(H))
phase_deg = np.angle(H, deg=True)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].semilogx(omega, mag_db, color=COL["main"], lw=2)
axes[0].set_ylabel("|G| [dB]"); axes[0].set_title("Diagramma di Bode")
axes[0].axvline(np.sqrt(k/m), color="gray", ls=":", label=r"$\omega_n=\sqrt{k/m}$")
axes[0].legend()
axes[1].semilogx(omega, phase_deg, color=COL["accent"], lw=2)
axes[1].set_ylabel("fase [°]"); axes[1].set_xlabel(r"$\omega$ [rad/s]")
axes[1].axhline(-90, color="gray", ls=":")
plt.tight_layout(); plt.show()

# Poli (autovalori di A)
print("Poli =", np.linalg.eigvals(A))
print("BIBO-stabile?", np.all(np.linalg.eigvals(A).real < 0))


**Cosa mostra il grafico** — ampiezza e fase del trasferimento
$G(j\omega)$ in funzione di $\omega$: si vede chiaramente la **risonanza**
attorno a $\omega_n=\sqrt{k/m}=2$ rad/s, dove il modulo ha un massimo
locale. A frequenze basse il sistema risponde come una compliance pura
($G(0)=1/k$); a frequenze alte il sistema attenua (massa che fa da filtro
inerziale).

## 4.4 Classificazione degli equilibri 2×2

Per $\dot{\mathbf{x}}=A\mathbf{x}$ con $A$ $2\times 2$, l'unico equilibrio
non banale (se $A$ invertibile) è $\mathbf{x}^*=\mathbf{0}$. La sua natura
si legge dal **piano traccia–determinante**:

- $\det A<0$ → **sella** (autovalori reali di segno opposto), instabile;
- $\det A>0$ e $\operatorname{tr}A^2-4\det A>0$ → **nodo**:
    - $\operatorname{tr}A<0$ → stabile,
    - $\operatorname{tr}A>0$ → instabile;
- $\det A>0$ e $\operatorname{tr}A^2-4\det A<0$ → **fuoco** (autovalori
  complessi coniugati):
    - $\operatorname{tr}A<0$ → fuoco stabile (oscillazioni smorzate),
    - $\operatorname{tr}A>0$ → fuoco instabile;
- $\operatorname{tr}A=0$, $\det A>0$ → **centro** (oscillazioni perfette).

Il **grafico dei 6 casi canonici** è uno dei migliori "pro-memoria" della
modellistica.


In [ ]:
# Classificazione visuale dei 6 punti di equilibrio 2D
def draw_phase_portrait(ax, A, title, lim=2.0):
    f = lambda t, x: A @ x
    plot_vector_field(ax, f, (-lim, lim), (-lim, lim), n=18)
    # Traiettorie da varie condizioni iniziali
    ics = []
    for a in np.linspace(-lim, lim, 7):
        ics += [[a, lim], [a, -lim], [lim, a], [-lim, a]]
    plot_trajectories(ax, f, ics, t_max=4.0, color=COL["main"], lw=0.9)
    # Equilibrio
    ax.plot(0, 0, "o", color=COL["accent"], ms=8, zorder=10)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_title(title, fontsize=10)
    ax.set_aspect("equal")

cases = [
    (np.array([[-1.0,  0.0],
               [ 0.0, -2.0]]),  "Nodo stabile (autoval. reali < 0)"),
    (np.array([[ 1.0,  0.0],
               [ 0.0,  2.0]]),  "Nodo instabile (autoval. reali > 0)"),
    (np.array([[ 1.0,  0.0],
               [ 0.0, -1.0]]),  "Sella (autoval. di segno opposto)"),
    (np.array([[-0.4, -1.0],
               [ 1.0, -0.4]]),  "Fuoco stabile (compl. con Re<0)"),
    (np.array([[ 0.4, -1.0],
               [ 1.0,  0.4]]),  "Fuoco instabile (compl. con Re>0)"),
    (np.array([[ 0.0, -1.0],
               [ 1.0,  0.0]]),  "Centro (autoval. immag. puri)"),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (A, title) in zip(axes.flat, cases):
    draw_phase_portrait(ax, A, title)
plt.suptitle("Classificazione degli equilibri di un sistema lineare 2D", fontsize=13)
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — i 6 ritratti di fase canonici. Le frecce
del campo vettoriale mostrano la direzione istantanea di evoluzione, le
linee blu sono traiettorie integrate dal campo.

**Come si interpreta** — per qualsiasi modello lineare 2D, una volta
calcolati traccia e determinante della matrice $A$ si può dire *immediatamente*
in quale di questi 6 regimi siamo. Per modelli **non lineari** si calcola
il Jacobiano in equilibrio e si applica la stessa classificazione (è la
**linearizzazione** della Parte 5).


### Il piano traccia–determinante

Spesso si visualizza la classificazione su un piano con
$\operatorname{tr}A$ in ascissa e $\det A$ in ordinata.


In [ ]:
# Diagramma traccia-determinante delle classi di equilibri 2D
fig, ax = plt.subplots(figsize=(9, 7))
tr = np.linspace(-4, 4, 400)
det_par = (tr**2) / 4   # parabola tr^2 = 4 det

ax.plot(tr, det_par, color="black", lw=1.5, label=r"$\mathrm{tr}^2 = 4\det$")
ax.axhline(0, color="black", lw=1.0)
ax.axvline(0, color="black", lw=1.0)

# Regioni colorate
ax.fill_between(tr, det_par, 6, where=(tr < 0), color="#aed6f1", alpha=0.6, label="fuochi stabili")
ax.fill_between(tr, det_par, 6, where=(tr > 0), color="#f1948a", alpha=0.6, label="fuochi instabili")
ax.fill_between(tr, 0, det_par, where=(tr < 0), color="#a9dfbf", alpha=0.6, label="nodi stabili")
ax.fill_between(tr, 0, det_par, where=(tr > 0), color="#f5cba7", alpha=0.6, label="nodi instabili")
ax.fill_between(tr, -6, 0, color="#d2b4de", alpha=0.4, label="selle")
# Asse det=0: degeneri; asse tr=0 con det>0: centri
ax.axvline(0, color="purple", lw=1.5, alpha=0.7)
ax.text(0.05, 4.5, "centri (tr=0, det>0)", color="purple", fontsize=10)

ax.set_xlim(-4, 4); ax.set_ylim(-3, 6)
ax.set_xlabel("traccia $\\mathrm{tr}\\,A$")
ax.set_ylabel("determinante $\\det A$")
ax.set_title("Piano traccia-determinante: classificazione degli equilibri 2D")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout(); plt.show()


## 4.5 Feedback e Nyquist

Un sistema **in retroazione negativa** ha la forma:

```
   r(t) ---> [ + ] --u-->[ G(s) ]--y-->
                  ^              |
                  |              |
                  +---[ H(s) ]<--+
```

La funzione di trasferimento del **loop chiuso** è

$$
G_{cl}(s)=\frac{G(s)}{1+G(s)H(s)}.
$$

I **poli** del loop chiuso sono gli zeri di $1+L(s)$ con $L(s)=G(s)H(s)$
**loop gain**. Per garantire stabilità il **criterio di Nyquist** dice:

> Tracciato $L(j\omega)$ per $\omega\in(-\infty,+\infty)$, il numero di
> giri del diagramma attorno al punto $-1$ (in senso orario) deve essere
> uguale al numero di poli instabili di $L$ (a circuito aperto). Per un
> $L$ a circuito aperto stabile la condizione si riduce a: il diagramma
> **non deve circondare** $-1$.

In presenza di **ritardo puro** $T$, $L(s)\to L(s)\,e^{-sT}$. Sul diagramma
di Nyquist questo aggiunge una rotazione che dipende dalla frequenza:
oltre una certa $\omega$ il diagramma comincia ad avvolgere $-1$ e il loop
**perde stabilità**. La causa fisica: il segnale di feedback arriva *in
ritardo*, quindi può rinforzare l'errore invece di cancellarlo. Lo studieremo
nei modelli di controllo ventilatorio (Cheyne–Stokes).


In [ ]:
# Schema a blocchi di un sistema in retroazione
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.set_xlim(-0.5, 12); ax.set_ylim(-1.5, 4); ax.set_aspect("equal"); ax.axis("off")

# Riferimento -> sommatore
draw_arrow(ax, (-0.2, 2.0), (1.3, 2.0), color=COL["main"])
ax.text(0.3, 2.3, "$r(t)$", color=COL["main"])
# Sommatore (cerchio con +)
ax.add_patch(Circle((1.6, 2.0), 0.3, fill=False, lw=1.6))
ax.text(1.6, 2.0, "+", ha="center", va="center", fontsize=14)
ax.text(1.35, 1.55, "−", ha="center", va="center", fontsize=14)
# Sommatore -> G(s)
draw_arrow(ax, (1.9, 2.0), (3.4, 2.0), color=COL["main"], label="$u(t)$", offset=(0, 0.2))
draw_box(ax, 3.4, 1.5, 1.8, 1.0, "Plant\n$G(s)$", fc="#eaf2f8", ec=COL["main"])
# G -> ritardo
draw_arrow(ax, (5.2, 2.0), (6.5, 2.0), color=COL["main"])
draw_box(ax, 6.5, 1.5, 1.8, 1.0, "Ritardo\n$e^{-sT}$", fc="#fdebd0", ec=COL["warn"])
# delay -> uscita
draw_arrow(ax, (8.3, 2.0), (10.0, 2.0), color=COL["main"], label="$y(t)$", offset=(0.2, 0.2))
ax.plot(10.0, 2.0, "o", color="black", ms=6)
# Tap del feedback
wire(ax, (10.0, 2.0), (10.0, 0.5), color=COL["accent"])
wire(ax, (10.0, 0.5), (5.7, 0.5), color=COL["accent"])
draw_box(ax, 4.0, 0.0, 1.7, 1.0, "Sensore\n$H(s)$", fc="#fef9e7", ec=COL["ok"])
wire(ax, (4.0, 0.5), (1.6, 0.5), color=COL["accent"])
draw_arrow(ax, (1.6, 0.5), (1.6, 1.7), color=COL["accent"])
ax.text(2.8, 0.7, "feedback negativo", color=COL["accent"], fontsize=10)

ax.set_title("Sistema lineare con retroazione e ritardo")
plt.tight_layout(); plt.show()


In [ ]:
# Nyquist con e senza ritardo per L(s) = K/(tau s + 1)
K, tau = 1.5, 1.0
T_delay_list = [0.0, 0.5, 1.5, 3.0]
omega = np.logspace(-2, 1.5, 1200)

fig, ax = plt.subplots(figsize=(7.5, 7.5))
for Td, c in zip(T_delay_list, [COL["main"], COL["ok"], COL["warn"], COL["accent"]]):
    H = K / (1j*omega*tau + 1) * np.exp(-1j*omega*Td)
    ax.plot(H.real, H.imag, lw=2, color=c, label=f"$T={Td}$ s")
    ax.plot(H.real, -H.imag, lw=1, color=c, ls=":")  # ramo per omega<0
ax.plot(-1, 0, "x", color="red", ms=14, mew=3, label="punto critico (-1, 0)")
ax.axhline(0, color="gray", lw=0.6); ax.axvline(0, color="gray", lw=0.6)
ax.set_xlabel("Re $L(j\\omega)$"); ax.set_ylabel("Im $L(j\\omega)$")
ax.set_title("Nyquist: effetto del ritardo $e^{-sT}$ sulla stabilita'")
ax.legend(); ax.set_aspect("equal")
ax.set_xlim(-2.5, 2); ax.set_ylim(-2.5, 2.5)
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — al crescere di $T$ la curva di Nyquist
ruota e si avvolge sempre più. Per $T$ piccolo non gira intorno al punto
$-1$ → loop stabile. Per $T$ grande inizia a girarci attorno → loop
**instabile**: oscillazioni autoindotte (è il meccanismo dietro
Cheyne–Stokes!).

**Perché è importante** — un controllo fisiologico (es. la ventilazione che
risponde alla PaCO$_2$ misurata in *ritardo* dai chemocettori) può essere
intrinsecamente stabile a circuito aperto eppure **diventare instabile**
quando il loop si chiude. Il ritardo è uno dei "killer" silenziosi di
stabilità nei sistemi biologici.


### Derivazione esplosa della funzione di trasferimento a circuito chiuso

Consideriamo lo schema:

```
   r(t) --->[+]----u-----[ G(s) ]----y----+--->
             ^                            |
             |                            |
             +---------[ H(s) ]<----------+
```

con $r$ riferimento, $y$ uscita misurata, $u$ azione di controllo. Nel
sommatore $u = r - H\,y$ (feedback **negativo**, da cui il segno).

In Laplace, con condizioni iniziali nulle:

$$
Y = G\,U = G(R - H Y) \;\Longrightarrow\; Y(1 + GH) = G R
\;\Longrightarrow\; G_{cl}=\frac{Y}{R}=\frac{G}{1+GH}.
$$

**Tre osservazioni** che ricorrono in tutti i modelli con feedback:

1. **I poli del closed-loop sono gli zeri di** $1+G(s)H(s)$. Quindi la
   stabilità del loop chiuso si studia analizzando dove si annulla
   $1+L(s)$ con $L=GH$.
2. **Errore a regime** a un riferimento $r=$ costante:
   $\;e_\infty = \lim_{s\to 0} s\,E(s) = \lim_{s\to 0} \frac{s\,R(s)}{1+L(s)} = \frac{r}{1+L(0)}$.
   Quindi $L(0)\to\infty$ (presenza di un integratore) elimina l'errore a
   regime: è il "perché" del termine integrale nei controllori PID.
3. **Sensitività** alle perturbazioni d'ingresso $d$: vale
   $Y/D = G/(1+L)$. Un loop ad alto guadagno *riduce* l'effetto di $d$
   sull'uscita: è la ragione per cui la natura "ama" i feedback alti
   (baroriflesso, regolazione glicemica, ...).

### Criterio di Routh–Hurwitz (versione operativa)

Per un polinomio caratteristico $p(s)=a_n s^n + a_{n-1} s^{n-1} + \cdots + a_0$
**a coefficienti reali**, vale:

> Tutte le radici hanno parte reale negativa *se e solo se* tutti gli
> elementi della prima colonna della **tabella di Routh** sono positivi.

Costruzione della tabella per $n=4$:

| | | | | |
|---|---|---|---|---|
| $s^4$ | $a_4$ | $a_2$ | $a_0$ | |
| $s^3$ | $a_3$ | $a_1$ | | |
| $s^2$ | $b_1$ | $b_2$ | | |
| $s^1$ | $c_1$ | | | |
| $s^0$ | $a_0$ | | | |

con

$$
b_1=\frac{a_3 a_2 - a_4 a_1}{a_3},\quad
b_2=\frac{a_3 a_0 - a_4 \cdot 0}{a_3}=a_0,\quad
c_1=\frac{b_1 a_1 - a_3 b_2}{b_1}.
$$

Il sistema è stabile sse $a_4, a_3, b_1, c_1, a_0 > 0$.

**Condizione necessaria di facile uso**: tutti i coefficienti $a_i$ devono
essere positivi (se ne manca uno o cambia segno, almeno una radice ha
parte reale $\ge 0$). È un controllo che si fa "a occhio" prima di costruire
la tabella.


In [ ]:
# Routh-Hurwitz: implementazione e verifica su un polinomio cubico
def routh_table(coeffs):
    # Costruisce la tabella di Routh data la lista di coefficienti
    # dal grado piu' alto al piu' basso, e ritorna la PRIMA COLONNA.
    coeffs = list(coeffs)
    n = len(coeffs)
    rows = [coeffs[0::2], coeffs[1::2]]
    # rendi le righe della stessa lunghezza
    max_len = max(len(rows[0]), len(rows[1]))
    rows = [r + [0.0]*(max_len - len(r)) for r in rows]
    while len(rows) < n:
        prev1, prev2 = rows[-2], rows[-1]
        new_row = []
        for j in range(1, max_len):
            num = prev2[0] * prev1[j] - prev1[0] * prev2[j]
            new_row.append(num / prev2[0] if prev2[0] != 0 else 0.0)
        new_row.append(0.0)
        rows.append(new_row)
    return [r[0] for r in rows]

# Polinomio: p(s) = s^3 + 2 s^2 + 3 s + 4   (esempio stabile)
p1 = [1, 2, 3, 4]
col1 = routh_table(p1)
print("Polinomio:", p1)
print("Prima colonna Routh:", col1)
print("Stabile? ", all(x > 0 for x in col1))
print("Radici (verifica):", np.roots(p1))

# Polinomio instabile: p(s) = s^3 + s^2 + s + 6
p2 = [1, 1, 1, 6]
col1 = routh_table(p2)
print("\nPolinomio:", p2)
print("Prima colonna Routh:", col1)
print("Stabile? ", all(x > 0 for x in col1))
print("Radici (verifica):", np.roots(p2))


### Criterio di Nyquist — schema della dimostrazione

**Principio dell'argomento (Cauchy)**: se $f(s)$ è una funzione meromorfa
e $\Gamma$ è una curva chiusa che non passa né per zeri né per poli di $f$,
allora

$$
\frac{1}{2\pi j}\oint_\Gamma \frac{f'(s)}{f(s)}\,ds = N - P,
$$

dove $N$ = numero di **zeri** dentro $\Gamma$ e $P$ = numero di **poli**
dentro $\Gamma$, contati con molteplicità.

Geometricamente: il numero di giri di $f(\Gamma)$ attorno all'origine è
$N - P$.

**Applicazione al feedback**: prendiamo $f(s) = 1 + L(s)$. Gli zeri di
$1+L$ sono i poli del closed-loop; i poli di $1+L$ coincidono con i poli
di $L$ (open-loop). Scelta $\Gamma$ = "contorno di Nyquist" (semicerchio
destro), si ottiene:

> **Criterio di Nyquist**: il numero $Z$ di poli instabili del closed-loop
> è $Z = N + P$, dove $N$ è il numero di giri *orari* del diagramma
> $L(j\omega)$ attorno al punto $-1$, e $P$ è il numero di poli instabili
> di $L$ (open-loop).
> Stabilità $\Leftrightarrow Z = 0$.

**Caso ricorrente**: $L$ open-loop stabile ($P=0$). Allora la condizione
è $N=0$: *il diagramma di Nyquist non deve circondare $-1$*. È la
versione "amichevole" che usiamo costantemente.

**Effetto del ritardo** $e^{-sT}$: aggiunge una rotazione di $-\omega T$
radianti a ogni frequenza. Aumentando $T$, il diagramma "gira" e prima o
poi inizia a circondare $-1$ → il loop perde stabilità. È la matematica
di Cheyne–Stokes che abbiamo già visto.


# Parte 5 — Sistemi non lineari

Quasi tutto ciò che è davvero biologico è **non lineare**: saturazioni
enzimatiche, canali ionici, capacità di carico, soglie, feedback. La
matematica del lineare resta indispensabile *vicino agli equilibri*, ma le
fenomenologie più ricche (bistabilità, oscillazioni autonomi, caos)
emergono solo nel non lineare.

In questa parte:

- **linearizzazione** rigorosa via Jacobiano + Hartman–Grobman;
- **1D non lineare** + potenziale + biforcazioni elementari (saddle-node,
  transcritical, pitchfork);
- **2D non lineare**: nullcline, ciclo limite, teorema di
  Poincaré–Bendixson, **Van der Pol**;
- **biforcazione di Hopf**;
- **caos deterministico**: Lorenz, Rössler, sensibilità alle condizioni
  iniziali.

## 5.1 Linearizzazione e Jacobiano: richiamo operativo

Data $\dot{\mathbf{x}}=\mathbf{f}(\mathbf{x})$, sia $\mathbf{x}^*$ un
**equilibrio** ($\mathbf{f}(\mathbf{x}^*)=\mathbf{0}$). Posto
$\boldsymbol{\xi}=\mathbf{x}-\mathbf{x}^*$, lo sviluppo di Taylor del primo
ordine restituisce

$$
\dot{\boldsymbol{\xi}}=\underbrace{J(\mathbf{x}^*)}_{=A}\boldsymbol{\xi}
+\mathcal{O}(\|\boldsymbol{\xi}\|^2),
$$

con $J_{ij}=\partial f_i/\partial x_j$ valutato in $\mathbf{x}^*$. Per
equilibri **iperbolici** (parte reale degli autovalori $\ne 0$), il
**teorema di Hartman–Grobman** garantisce equivalenza topologica locale
con il sistema lineare $\dot{\boldsymbol{\xi}}=A\boldsymbol{\xi}$.


In [ ]:
# Esempio: pendolo non smorzato dot theta = omega, dot omega = -g/L sin(theta)
g_over_L = 9.81 / 1.0

def pendolo(t, z):
    th, om = z
    return np.array([om, -g_over_L * np.sin(th)])

# Equilibri: (0, 0) e (pi, 0). Calcoliamo il Jacobiano in entrambi.
def J_pendolo(theta, omega):
    return np.array([[0.0, 1.0],
                     [-g_over_L * np.cos(theta), 0.0]])

for eq in [(0.0, 0.0), (np.pi, 0.0)]:
    J = J_pendolo(*eq)
    vals = np.linalg.eigvals(J)
    print(f"Equilibrio ({eq[0]:.2f}, {eq[1]:.2f}):")
    print("  Jacobiano:\n", J)
    print("  Autovalori:", vals)
    if all(np.abs(v.real) < 1e-9 for v in vals):
        print("  -> centro (non iperbolico, H-G non si applica)")
    elif any(v.real > 0 for v in vals):
        print("  -> instabile (sella o nodo/fuoco instabile)")
    else:
        print("  -> stabile")
    print()


## 5.1.1 Visualizzazione: cosa significa "linearizzare"?

Per fissare l'idea di linearizzazione, mostriamo un esempio scalare:
prendiamo $\dot x = \sin(x)$ vicino all'equilibrio $x^*=\pi$. Vicino a
$x^*$, $\sin(x)\approx -(x-x^*)$ (con segno meno perché siamo all'altro
zero di $\sin$). Quindi la linearizzazione è $\dot\xi=-\xi$, che predice
decadimento esponenziale. Vediamo quanto è valida questa
approssimazione per condizioni iniziali via via più lontane.


In [ ]:
# Linearizzazione vs sistema esatto: dot x = sin(x) attorno a x* = pi
def f_exact(t, x):  return np.array([np.sin(x[0])])
def f_linear(t, xi): return np.array([-xi[0]])    # linearizzato attorno a pi

T = 8.0
t_grid = np.linspace(0, T, 400)
fig, ax = plt.subplots(figsize=(11, 5.5))

for delta, c in zip([0.2, 0.6, 1.0, 1.5, 2.5],
                     [COL["main"], COL["ok"], COL["warn"], COL["accent"], COL["extra"]]):
    # Sistema esatto: parto da x = pi - delta (sotto equilibrio)
    sol_ex = solve_ivp(f_exact, (0, T), [np.pi - delta], t_eval=t_grid, max_step=0.05)
    # Sistema lineare: parto da xi = -delta
    sol_li = solve_ivp(f_linear, (0, T), [-delta], t_eval=t_grid, max_step=0.05)

    ax.plot(t_grid, sol_ex.y[0], color=c, lw=2, label=f"esatto, $\\delta={delta}$")
    ax.plot(t_grid, np.pi + sol_li.y[0], color=c, lw=1.2, ls="--")

ax.axhline(np.pi, color="gray", ls=":", label=r"equilibrio $x^* = \pi$")
ax.set_xlabel("tempo"); ax.set_ylabel("x(t)")
ax.set_title("Sistema $\\dot x=\\sin(x)$ vs linearizzato\nlinee continue = esatto, tratteggiate = lineare")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — per $\delta$ piccoli ($\delta=0.2$, 0.6) le
due curve quasi coincidono: l'approssimazione lineare funziona benissimo.
Per $\delta$ grandi (1.5, 2.5) le curve divergono significativamente:
l'approssimazione lineare sopravaluta o sottovaluta la velocità di ritorno
all'equilibrio. **Lezione**: la linearizzazione racconta la verità solo
in un *intorno* dell'equilibrio. È *locale*.

## 5.2 1D non lineare: il "potenziale" $V(x)$

Per $\dot x=f(x)$ scalare, se $f=-dV/dx$, allora $V$ è un **potenziale**:
$\dot V= f \cdot \dot x = -(\dot x)^2\le 0$. Il sistema "scende" sempre nel
potenziale e si ferma a un *minimo locale*.

Esempio: $\dot x=x-x^3$. Il potenziale è $V(x)=-x^2/2+x^4/4$. Tre equilibri
$x=0$ (massimo, *instabile*) e $x=\pm 1$ (minimi, *stabili*). Sistema
**bistabile**: la condizione iniziale determina in quale pozzo cade.


In [ ]:
# Sistema bistabile: dx/dt = x - x^3
def f(x): return x - x**3
def V(x): return -x**2/2 + x**4/4

x = np.linspace(-1.6, 1.6, 600)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# Pannello 1: f(x) e equilibri
ax = axes[0]
ax.plot(x, f(x), color=COL["main"], lw=2)
ax.axhline(0, color="gray", lw=0.7)
ax.plot([-1, 0, 1], [0, 0, 0], "o", ms=10,
        color="white", mec=COL["accent"], mew=2)
ax.annotate("stabile", (-1, 0), xytext=(-1.3, 0.25),
            arrowprops=dict(arrowstyle="->", color=COL["accent"]), color=COL["accent"])
ax.annotate("instabile", (0, 0), xytext=(0.15, 0.4),
            arrowprops=dict(arrowstyle="->", color=COL["accent"]), color=COL["accent"])
ax.annotate("stabile", (1, 0), xytext=(0.6, 0.25),
            arrowprops=dict(arrowstyle="->", color=COL["accent"]), color=COL["accent"])
ax.set_xlabel("$x$"); ax.set_ylabel("$f(x)=\\dot x$")
ax.set_title(r"Campo $\dot x = x - x^3$")

# Pannello 2: potenziale V(x)
ax = axes[1]
ax.plot(x, V(x), color=COL["main"], lw=2)
ax.axhline(0, color="gray", lw=0.7)
ax.plot([-1, 0, 1], [V(-1), V(0), V(1)], "o", ms=10,
        color="white", mec=COL["accent"], mew=2)
ax.set_xlabel("$x$"); ax.set_ylabel(r"$V(x) = -x^2/2 + x^4/4$")
ax.set_title("Potenziale: i minimi sono i due stati stabili")

plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — a sinistra il campo $f(x)$ con i tre
zeri $-1, 0, +1$; a destra il potenziale a doppia buca. Il sistema
"rotola" verso il minimo più vicino. È il modello canonico di **bistabilità**,
e descrive ad esempio l'innesco di un potenziale d'azione visto come
transizione da uno stato di riposo a uno stato eccitato (versione drasticamente
semplificata).


## 5.3 Biforcazioni elementari

Una **biforcazione** è un cambio *qualitativo* nel ritratto di fase al
variare di un parametro $r$. I tre prototipi 1D:

### Saddle-node: $\dot x = r + x^2$

- $r<0$: due equilibri $x_\pm=\pm\sqrt{-r}$, uno stabile e uno instabile;
- $r=0$: collidono e annichiliscono;
- $r>0$: nessun equilibrio.

### Transcritica: $\dot x = r x - x^2$

- equilibri $x=0$ e $x=r$;
- si **scambiano** stabilità in $r=0$.

### Pitchfork supercritica: $\dot x = r x - x^3$

- $r<0$: unico equilibrio $x=0$ stabile;
- $r>0$: $x=0$ instabile, nascono $x=\pm\sqrt{r}$ stabili (**rottura di
  simmetria**, bistabilità).


In [ ]:
# Diagrammi di biforcazione 1D classici
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Saddle-node
ax = axes[0]
r_arr = np.linspace(-2, 0, 200)
ax.plot(r_arr, -np.sqrt(-r_arr), color=COL["main"], lw=2, label="stabile")
ax.plot(r_arr,  np.sqrt(-r_arr), color=COL["accent"], lw=2, ls="--", label="instabile")
ax.axvline(0, color="gray", lw=0.7); ax.axhline(0, color="gray", lw=0.7)
ax.set_xlabel("parametro $r$"); ax.set_ylabel("$x^*$")
ax.set_title(r"Saddle-node: $\dot x = r + x^2$")
ax.legend()

# Transcritica
ax = axes[1]
r_arr = np.linspace(-2, 2, 200)
ax.plot(r_arr, np.zeros_like(r_arr), color=COL["main"], lw=2)
ax.plot(r_arr, np.zeros_like(r_arr), color=COL["accent"], lw=0)  # placeholder
ax.plot(r_arr[r_arr < 0], np.zeros_like(r_arr[r_arr < 0]), color=COL["main"],   lw=2, label="$x=0$ stabile")
ax.plot(r_arr[r_arr > 0], np.zeros_like(r_arr[r_arr > 0]), color=COL["accent"], lw=2, ls="--", label="$x=0$ instabile")
ax.plot(r_arr[r_arr < 0], r_arr[r_arr < 0], color=COL["accent"], lw=2, ls="--", label="$x=r$ instabile")
ax.plot(r_arr[r_arr > 0], r_arr[r_arr > 0], color=COL["main"],   lw=2, label="$x=r$ stabile")
ax.axvline(0, color="gray", lw=0.7); ax.axhline(0, color="gray", lw=0.7)
ax.set_xlabel("parametro $r$"); ax.set_ylabel("$x^*$")
ax.set_title(r"Transcritica: $\dot x = r x - x^2$")
ax.legend(fontsize=8)

# Pitchfork supercritica
ax = axes[2]
r_arr = np.linspace(-2, 2, 200)
ax.plot(r_arr[r_arr < 0], np.zeros_like(r_arr[r_arr < 0]), color=COL["main"], lw=2)
ax.plot(r_arr[r_arr > 0], np.zeros_like(r_arr[r_arr > 0]), color=COL["accent"], lw=2, ls="--")
r_pos = r_arr[r_arr > 0]
ax.plot(r_pos,  np.sqrt(r_pos), color=COL["main"], lw=2)
ax.plot(r_pos, -np.sqrt(r_pos), color=COL["main"], lw=2)
ax.axvline(0, color="gray", lw=0.7); ax.axhline(0, color="gray", lw=0.7)
ax.set_xlabel("parametro $r$"); ax.set_ylabel("$x^*$")
ax.set_title(r"Pitchfork supercritica: $\dot x = r x - x^3$")

plt.suptitle("Biforcazioni 1D: $x^*(r)$, linee continue = stabili, tratteggiate = instabili",
             fontsize=12)
plt.tight_layout(); plt.show()


**Cosa mostrano i grafici** — al variare del parametro $r$, gli equilibri
(curve continue/tratteggiate) appaiono, scompaiono o cambiano stabilità.
Sono i 3 "schemi" che ricorrono praticamente in ogni modello dinamico.

**Perché sono importanti** — molti fenomeni fisiologici sono interpretabili
come biforcazioni: comparsa di un ritmo cardiaco (Hopf), passaggio
quiescente↔spiking di un neurone (saddle-node on invariant circle),
transizione tra stati di pesca metabolica.

## 5.4 Sistemi 2D: nullcline, ciclo limite e Poincaré–Bendixson

Per $\dot x = f(x,y),\;\dot y = g(x,y)$:

- la **nullclina di $x$** è $\{f=0\}$ (luogo dove $\dot x=0$, traiettorie
  verticali);
- la **nullclina di $y$** è $\{g=0\}$ (traiettorie orizzontali);
- gli **equilibri** sono le intersezioni delle due nullcline.

**Teorema di Poincaré–Bendixson**: in $\mathbb{R}^2$, se un'orbita resta
confinata in una regione compatta priva di equilibri, allora *deve*
avvicinarsi a un'**orbita periodica** (ciclo limite). Conseguenza: in 2D
non c'è caos — il caos richiede $\ge 3$ dimensioni.

### Van der Pol come prototipo di ciclo limite

L'equazione di Van der Pol nasce in elettronica (oscillatore con triodo)
ma è il modello canonico di **oscillazione autosostenuta** con uno smorzamento
non lineare:

$$
\ddot x - \mu(1-x^2)\dot x + x = 0,\quad \mu>0.
$$

Per $|x|<1$ lo smorzamento è negativo (sistema *guadagna* energia); per
$|x|>1$ è positivo (sistema *perde* energia). Il bilancio crea un unico
ciclo limite stabile. Forma di stato:

$$
\dot x_1=x_2,\qquad \dot x_2=\mu(1-x_1^2)x_2 - x_1.
$$


In [ ]:
# Van der Pol: phase plane + serie temporale
def van_der_pol(t, x, mu=1.0):
    return np.array([x[1], mu*(1 - x[0]**2)*x[1] - x[0]])

mu = 1.0
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]; ax.set_aspect("equal")
plot_vector_field(ax, lambda t, x: van_der_pol(t, x, mu), (-3, 3), (-4, 4), n=22)
plot_trajectories(ax, lambda t, x: van_der_pol(t, x, mu),
                  ics=[[0.1, 0.1], [-2.5, 0], [2.5, 0], [0, 3.5], [0, -3.5]],
                  t_max=30, color=COL["main"])
ax.set_xlim(-3, 3); ax.set_ylim(-4, 4)
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.set_title(fr"Van der Pol $\mu={mu}$: ciclo limite stabile")

ax = axes[1]
for ic, c in zip([[0.1, 0.1], [3.0, 0.0], [0.0, 3.5]],
                  [COL["main"], COL["accent"], COL["ok"]]):
    sol = solve_ivp(lambda t, x: van_der_pol(t, x, mu), (0, 25), ic,
                    dense_output=True, max_step=0.05)
    ax.plot(sol.t, sol.y[0], color=c, lw=1.6, label=f"IC = {ic}")
ax.set_xlabel("tempo"); ax.set_ylabel("$x_1$")
ax.set_title("$x_1(t)$ per diverse condizioni iniziali: convergono allo stesso ciclo")
ax.legend()
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — a sinistra il ritratto di fase: tutte le
traiettorie (da dentro o da fuori) tendono allo stesso anello chiuso, il
**ciclo limite**. A destra le storie temporali: dopo un transitorio, le
oscillazioni hanno tutte la stessa ampiezza e periodo, *indipendentemente*
dalla condizione iniziale. Questa è la definizione di **oscillatore
autosostenuto**.

## 5.5 Biforcazione di Hopf

Quando un parametro varia e una coppia di autovalori complessi attraversa
l'asse immaginario passando da $\Re(\lambda)<0$ a $\Re(\lambda)>0$, succedono
due cose:

1. l'equilibrio cambia stabilità (fuoco stabile → instabile);
2. nasce (o muore) un **ciclo limite** di piccola ampiezza, che cresce
   come $\sqrt{r-r_c}$ vicino alla biforcazione.

Questo è il meccanismo per cui un sistema *senza* oscillazioni a basso
guadagno inizia improvvisamente a oscillare quando il guadagno supera una
soglia (es. tremore essenziale, ritmi respiratori instabili, comparsa di
fibrillazione).


In [ ]:
# Biforcazione di Hopf supercritica:
#   dot x = mu x - y - x (x^2 + y^2)
#   dot y = x + mu y - y (x^2 + y^2)
# Equilibrio (0,0): autovalori mu +- i. Ciclo limite di raggio sqrt(mu) per mu>0.
def hopf(t, z, mu):
    x, y = z
    r2 = x*x + y*y
    return np.array([mu*x - y - x*r2,
                     x + mu*y - y*r2])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
for ax, mu in zip(axes, [-0.5, 0.0, 0.6]):
    plot_vector_field(ax, lambda t, z: hopf(t, z, mu), (-1.5, 1.5), (-1.5, 1.5), n=20)
    plot_trajectories(ax, lambda t, z: hopf(t, z, mu),
                      ics=[[1.0, 0.0], [-1.0, 0.1], [0.05, 0.0]],
                      t_max=40, color=COL["main"])
    ax.plot(0, 0, "o", color=COL["accent"], ms=8)
    if mu > 0:
        th = np.linspace(0, 2*np.pi, 200)
        ax.plot(np.sqrt(mu)*np.cos(th), np.sqrt(mu)*np.sin(th),
                color=COL["ok"], lw=2, ls="--")
        ax.text(0.05, np.sqrt(mu)+0.05, "ciclo limite", color=COL["ok"])
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_aspect("equal")
    ax.set_title(fr"$\mu={mu}$")
plt.suptitle("Biforcazione di Hopf supercritica: nasce un ciclo limite per $\\mu>0$", fontsize=12)
plt.tight_layout(); plt.show()


### Enunciato rigoroso e idea di prova di Hartman–Grobman

**Teorema (Hartman–Grobman, 1960)**. Sia $\mathbf{f}:\mathbb{R}^n\to\mathbb{R}^n$
una funzione $C^1$ e $\mathbf{x}^*$ un equilibrio **iperbolico** (cioè
$\mathbf{f}(\mathbf{x}^*)=\mathbf{0}$ e tutti gli autovalori di
$A=J(\mathbf{x}^*)$ hanno parte reale $\ne 0$). Allora esistono intorni
$U$ di $\mathbf{x}^*$ e $V$ di $\mathbf{0}$ e un **omeomorfismo**
$h:U\to V$ tale che $h$ mappa le traiettorie del sistema **non lineare**
$\dot{\mathbf{x}}=\mathbf{f}(\mathbf{x})$ nelle traiettorie del sistema
**lineare** $\dot{\boldsymbol{\xi}}=A\boldsymbol{\xi}$, preservando il
parametro tempo.

**Cosa significa concretamente**:

- Il *ritratto qualitativo* (forma delle traiettorie, stabilità, sella vs
  nodo vs fuoco) del non lineare vicino a $\mathbf{x}^*$ è *uguale* a
  quello del lineare.
- Le costanti di tempo (autovalori) sono **le stesse**.
- Non garantisce uguaglianza *geometrica* esatta, ma sì topologica.

**Idea di prova** (in 4 passi, non formale):

1. Scrivi $\mathbf{f}(\mathbf{x})=A\boldsymbol{\xi}+g(\boldsymbol{\xi})$
   con $g$ resto del primo ordine: $\|g(\boldsymbol{\xi})\|=o(\|\boldsymbol{\xi}\|)$.
2. Definisci l'operatore $T$ sui flussi:
   $h$ deve coniugare $\Phi^t_{nl}$ (flusso non lineare) con $\Phi^t_{lin}=e^{At}$.
3. In un piccolo intorno l'operatore $T$ è una contrazione (per la
   piccolezza di $g$ e la iperbolicità: $e^{At}$ contrae alcune direzioni e
   espande altre, ma nessuna è "neutra").
4. Per il **teorema del punto fisso di Banach**, esiste un unico $h$
   continuo che coniuga i due flussi. Si dimostra poi che $h$ è iniettiva
   e ha inversa continua, cioè è un omeomorfismo.

**Quando l'ipotesi salta** (autovalore con $\Re=0$): l'equilibrio è
**non iperbolico**, il teorema non si applica e per studiare la dinamica
locale servono *manifold center* (Center Manifold Theorem). È il caso di:

- centri lineari (Lotka–Volterra);
- biforcazioni di Hopf "al punto critico";
- biforcazione saddle-node "al punto critico".

In tutti questi casi il termine non lineare *determina* il comportamento:
*non lo si può ignorare*.


### Teorema di Poincaré–Bendixson — enunciato e conseguenze

**Teorema (Poincaré–Bendixson)**. Sia $\dot{\mathbf{x}}=\mathbf{f}(\mathbf{x})$
un sistema $C^1$ in $\mathbb{R}^2$. Sia $\mathcal{T}\subset\mathbb{R}^2$
una regione **compatta** (chiusa e limitata) **positivamente invariante**
(le traiettorie che entrano non escono), che **non contiene equilibri**.
Allora ogni orbita $\mathbf{x}(t)$ contenuta in $\mathcal{T}$ ha come
$\omega$-limite un'**orbita periodica**.

**Conseguenza pratica**: per dimostrare l'esistenza di un ciclo limite in
un sistema 2D è sufficiente:

1. costruire una regione "a buco" che contenga un equilibrio instabile
   (es. un fuoco instabile) ma escluda altri equilibri;
2. mostrare che sul bordo esterno della regione le traiettorie *entrano*;
3. mostrare che sul bordo interno (attorno all'equilibrio instabile) le
   traiettorie *escono*.

Per Poincaré–Bendixson la regione "a buco" è positivamente invariante e
senza equilibri → c'è un ciclo limite dentro.

**Esempio**: Van der Pol. Si dimostra rigorosamente l'esistenza di un
**unico** ciclo limite costruendo un anello positivamente invariante.

**Conseguenza più importante**: in $\mathbb{R}^2$ **non esiste caos**.
Il caos richiede $\ge 3$ dimensioni perché solo lì c'è "abbastanza spazio"
per traiettorie limitate, non periodiche, non convergenti. Da qui la
"dimensione 3" magica di Lorenz, Rössler, Hodgkin-Huxley (3 gating + V).


## 5.6 Caos deterministico

In $\mathbb{R}^3$ può esistere un **attrattore caotico**: insieme limite di
volume zero, su cui le traiettorie sono limitate e *sensibili alle condizioni
iniziali* (due IC vicine si separano esponenzialmente nel tempo).

### Sistema di Lorenz

$$
\dot x = \sigma(y-x),\quad \dot y = x(\rho-z)-y,\quad \dot z = xy-\beta z.
$$

Parametri canonici di Lorenz (1963): $\sigma=10$, $\rho=28$, $\beta=8/3$.
Il famoso "attrattore a farfalla".

### Sistema di Rössler

$$
\dot x=-y-z,\quad \dot y=x+a y,\quad \dot z=b+z(x-c).
$$

Più semplice di Lorenz (solo una non linearità $xz$), ma con la stessa
fenomenologia.


In [ ]:
# Lorenz: attrattore e sensitivita' alle condizioni iniziali
def lorenz(t, X, sigma=10, rho=28, beta=8/3):
    x, y, z = X
    return np.array([sigma*(y - x), x*(rho - z) - y, x*y - beta*z])

T = 35.0
t_eval = np.linspace(0, T, 6000)
sol1 = solve_ivp(lorenz, (0, T), [1.0, 1.0, 1.0],   t_eval=t_eval, rtol=1e-9, atol=1e-12)
sol2 = solve_ivp(lorenz, (0, T), [1.0, 1.0, 1.0001], t_eval=t_eval, rtol=1e-9, atol=1e-12)

fig = plt.figure(figsize=(14, 5.5))
# Sinistra: attrattore 3D
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot(sol1.y[0], sol1.y[1], sol1.y[2], color=COL["main"], lw=0.7)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("Attrattore di Lorenz")

# Destra: x(t) per due IC quasi uguali
ax = fig.add_subplot(1, 2, 2)
ax.plot(sol1.t, sol1.y[0], color=COL["main"], lw=1.2, label="IC = (1,1,1)")
ax.plot(sol2.t, sol2.y[0], color=COL["accent"], lw=1.2, label="IC = (1,1,1.0001)")
ax.set_xlabel("tempo"); ax.set_ylabel("$x(t)$")
ax.set_title("Sensibilita' alle condizioni iniziali")
ax.legend()
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — a sinistra il celebre "attrattore a
farfalla". A destra: due traiettorie inizialmente *quasi identiche* (differenza
di $10^{-4}$ in $z(0)$) seguono percorsi simili per $\approx 15$ unità di
tempo, poi divergono completamente. Questa è la **sensibilità alle
condizioni iniziali**: anche se le equazioni sono *deterministiche*, le
previsioni a lungo termine diventano *impossibili* perché qualsiasi
incertezza iniziale viene amplificata esponenzialmente.

**Implicazione fisiologica** — non tutti i ritmi biologici irregolari sono
"rumore": il cuore aritmico, l'EEG epilettico, le fluttuazioni respiratorie
hanno spesso componenti caotiche. Caratterizzare la dimensione frattale
dell'attrattore è uno strumento di analisi clinica.


In [ ]:
# Rössler: attrattore e serie temporale
def rossler(t, X, a=0.2, b=0.2, c=5.7):
    x, y, z = X
    return np.array([-y - z, x + a*y, b + z*(x - c)])

T = 150
t_eval = np.linspace(0, T, 12000)
sol = solve_ivp(rossler, (0, T), [0.1, 0.0, 0.0], t_eval=t_eval, rtol=1e-9, atol=1e-12)

fig = plt.figure(figsize=(13, 5))
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.plot(sol.y[0], sol.y[1], sol.y[2], color=COL["extra"], lw=0.6)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("Attrattore di Rössler")

ax = fig.add_subplot(1, 2, 2)
ax.plot(sol.t, sol.y[0], color=COL["extra"], lw=0.8)
ax.set_xlabel("tempo"); ax.set_ylabel("$x(t)$")
ax.set_title("Componente x(t): bande aperiodiche")
plt.tight_layout(); plt.show()


# Parte 6 — Dinamica delle popolazioni

I modelli di popolazione sono i **prototipi più puliti** della modellistica
biologica: lo stato è una densità (o un conteggio), la dinamica esprime
bilanci di natalità/mortalità, e gli equilibri hanno un'interpretazione
biologica immediata. Useremo qui anche le tecniche di nondimensionalizzazione,
linearizzazione, e classificazione 2D imparate nelle parti precedenti.

## 6.1 Modello di Malthus (crescita esponenziale)

Se le nascite e le morti per individuo sono costanti, $N$ obbedisce a

$$
\dot N = (b - d)\,N = r\,N,\qquad N(0)=N_0,
$$

con $r=b-d$ **tasso di crescita intrinseco**. Soluzione: $N(t)=N_0 e^{rt}$.

- $r>0$ → crescita esponenziale (proliferazione di batteri in fase
  esponenziale, popolazione umana in regime preindustriale);
- $r<0$ → decadimento esponenziale (decadimento radioattivo, popolazione
  di specie minacciate senza riproduzione).

**Limiti del modello**: ignora risorse limitate. Crescita illimitata è
biologicamente assurda → serve un termine di saturazione.

## 6.2 Modello logistico (Verhulst)

$$
\boxed{\;\dot N = r N \left(1 - \frac{N}{K}\right)\;}
$$

- $r$: tasso intrinseco;
- $K$: **capacità portante** dell'ambiente;
- per $N\ll K$: $\dot N\approx rN$ (Malthus);
- per $N\to K$: $\dot N\to 0$ (saturazione).

### Soluzione analitica esplicita

Per separazione di variabili (lo dimostriamo per completezza):

$$
\int \frac{dN}{N(1-N/K)} = \int r\,dt.
$$

Decomposizione in fratti semplici:

$$
\frac{1}{N(1-N/K)}=\frac{1}{N}+\frac{1/K}{1-N/K}.
$$

Integrando: $\ln N - \ln(1-N/K)=rt + c$, cioè $\frac{N}{1-N/K}=A e^{rt}$.
Risolvendo per $N$ con $N(0)=N_0$:

$$
N(t)=\frac{K\,N_0\,e^{rt}}{K + N_0(e^{rt}-1)}=\frac{K}{1 + \left(\frac{K}{N_0}-1\right)e^{-rt}}.
$$

### Nondimensionalizzazione

$n=N/K$, $\tau=rt$:

$$
\frac{dn}{d\tau}=n(1-n).
$$

Un solo equilibrio non banale $n^*=1$, stabile.


In [ ]:
# Logistica: soluzione analitica + simulazione + variazione di K
def logistic_analytical(t, N0, r, K):
    return K / (1 + (K/N0 - 1) * np.exp(-r*t))

r = 0.6
N0 = 0.1
t = np.linspace(0, 25, 400)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for K, c in zip([1, 2, 5, 10], [COL["main"], COL["ok"], COL["accent"], COL["extra"]]):
    axes[0].plot(t, logistic_analytical(t, N0, r, K), color=c, lw=2, label=f"K={K}")
    axes[0].axhline(K, color=c, lw=0.5, ls=":")
axes[0].set_xlabel("tempo"); axes[0].set_ylabel("$N(t)$")
axes[0].set_title("Logistica al variare di $K$ ($r=0.6$, $N_0=0.1$)")
axes[0].legend()

for r_val, c in zip([0.2, 0.5, 1.0, 2.0], [COL["main"], COL["ok"], COL["accent"], COL["extra"]]):
    axes[1].plot(t, logistic_analytical(t, N0, r_val, 5), color=c, lw=2, label=f"r={r_val}")
axes[1].axhline(5, color="gray", ls=":")
axes[1].set_xlabel("tempo"); axes[1].set_ylabel("$N(t)$")
axes[1].set_title("Logistica al variare di $r$ ($K=5$, $N_0=0.1$)")
axes[1].legend()
plt.tight_layout(); plt.show()


**Cosa mostrano i grafici** — a sinistra: la capacità portante $K$
fissa il valore *asintotico*; a destra: il tasso $r$ controlla la *rapidità*
con cui $N$ vi si avvicina. La forma a "S" (sigmoide logistica) è
universale: la curva di crescita di una colonia batterica, di un tumore,
di un mercato, segue la stessa equazione.

## 6.3 Lotka–Volterra: preda–predatore

$$
\dot x = a x - b x y,\qquad \dot y = -c y + d x y,
$$

- $x$ prede, $y$ predatori;
- $a$ = tasso di crescita delle prede in assenza di predatori;
- $c$ = tasso di mortalità dei predatori in assenza di prede;
- $b, d$ = efficienza dell'interazione preda–predatore.

### Equilibri

$\dot x=0$ e $\dot y=0$ dà $(0,0)$ (estinzione totale) e
$(x^*, y^*)=(c/d,\,a/b)$ (coesistenza).

### Linearizzazione

$$
J=\begin{bmatrix} a - b y & -b x \\ d y & -c + d x\end{bmatrix}.
$$

In $(c/d, a/b)$:

$$
J^* = \begin{bmatrix} 0 & -bc/d \\ ad/b & 0\end{bmatrix},
$$

autovalori $\lambda=\pm i\sqrt{ac}$ → **centro** (oscillazioni perfette in
linearizzato). In realtà il sistema *non lineare* ha orbite *chiuse*
attorno all'equilibrio, ma di ampiezza che dipende dall'IC: non è un ciclo
limite.

### Holling (predatore con saturazione)

Il termine $bxy$ è irrealistico: ogni predatore può "mangiare" solo un
numero finito di prede per unità di tempo. **Risposta funzionale di
Holling tipo II**:

$$
\dot x = a x \Big(1-\frac{x}{K}\Big) - \frac{\beta x}{1 + h x}\,y,\qquad
\dot y = \frac{\varepsilon \beta x}{1+h x}\,y - c y.
$$

Con questa modifica, il sistema può perdere stabilità dell'equilibrio
non banale e *acquisire* un ciclo limite (oscillazioni autosostenute con
ampiezza intrinseca al sistema, non più all'IC).


In [ ]:
# Lotka-Volterra: tempo e phase plane, e variazione delle condizioni iniziali
a, b, c, d = 1.0, 0.5, 0.75, 0.25
def lv(t, z): x, y = z; return np.array([a*x - b*x*y, -c*y + d*x*y])

x_eq, y_eq = c/d, a/b
T = 25
t_eval = np.linspace(0, T, 4000)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
for ic, col_ in zip([[1.0, 0.5], [2.0, 1.5], [5.0, 1.0]],
                     [COL["main"], COL["ok"], COL["accent"]]):
    sol = solve_ivp(lv, (0, T), ic, t_eval=t_eval, rtol=1e-9, atol=1e-12)
    ax.plot(sol.t, sol.y[0], color=col_, lw=1.4, label=f"prede x, IC={ic}")
    ax.plot(sol.t, sol.y[1], color=col_, lw=1.4, ls="--")
ax.set_xlabel("tempo"); ax.set_ylabel("popolazione")
ax.set_title("Lotka-Volterra: oscillazioni dipendenti dall'IC")
ax.legend(fontsize=8)

ax = axes[1]; ax.set_aspect("equal")
plot_vector_field(ax, lv, (0.1, 6), (0.1, 4.5), n=18)
plot_trajectories(ax, lv, ics=[[1, 0.5], [2.0, 1.5], [5.0, 1.0], [4.5, 3.5]],
                  t_max=25, color=COL["main"])
ax.plot(x_eq, y_eq, "o", color=COL["accent"], ms=10)
ax.annotate(f"($c/d$, $a/b$)=({x_eq:.1f},{y_eq:.1f})",
            xy=(x_eq, y_eq), xytext=(x_eq+0.6, y_eq+0.6),
            arrowprops=dict(arrowstyle="->", color=COL["accent"]),
            color=COL["accent"])
ax.set_xlabel("prede $x$"); ax.set_ylabel("predatori $y$")
ax.set_title("Phase plane: orbite chiuse attorno all'equilibrio")
plt.tight_layout(); plt.show()


In [ ]:
# Preda-predatore con risposta funzionale Holling II + logistica per le prede
a, K, beta, h, eps, c_p = 1.0, 6.0, 1.2, 0.6, 0.6, 0.5

def holling(t, z):
    x, y = z
    Phi = beta * x / (1 + h * x)
    return np.array([a*x*(1 - x/K) - Phi*y,
                     eps*Phi*y - c_p*y])

T = 80; t_eval = np.linspace(0, T, 6000)
sol = solve_ivp(holling, (0, T), [1.0, 0.5], t_eval=t_eval, rtol=1e-9, atol=1e-12)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(sol.t, sol.y[0], color=COL["main"],   lw=1.6, label="prede")
axes[0].plot(sol.t, sol.y[1], color=COL["accent"], lw=1.6, label="predatori")
axes[0].set_xlabel("tempo"); axes[0].set_ylabel("popolazione")
axes[0].set_title("Holling II: nasce un CICLO LIMITE stabile")
axes[0].legend()

axes[1].set_aspect("equal")
plot_vector_field(axes[1], holling, (0.1, 6.5), (0.05, 3.5), n=20)
axes[1].plot(sol.y[0], sol.y[1], color=COL["main"], lw=2)
axes[1].set_xlabel("prede $x$"); axes[1].set_ylabel("predatori $y$")
axes[1].set_title("Ritratto di fase: orbita periodica autosostenuta")
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — A differenza del Lotka–Volterra "puro",
il modello con Holling II ha un *vero* ciclo limite: tutte le orbite (purché
non sull'equilibrio instabile interno) convergono a un'unica orbita periodica,
indipendentemente dalle condizioni iniziali. Questo è ciò che accade quando
si introduce una **saturazione** biologicamente realistica nel termine di
predazione.

**Cosa cambia con i parametri** — la presenza/assenza del ciclo limite
dipende criticamente da $K$, $h$, $\beta$. C'è una biforcazione di Hopf:
sotto soglia l'equilibrio interno è stabile (oscillazioni si smorzano);
sopra soglia, l'equilibrio perde stabilità e nasce il ciclo.


### Quantità conservata di Lotka–Volterra (dimostrazione esplicita)

Per il sistema di Lotka–Volterra $\dot x = ax - bxy,\;\dot y = -cy + dxy$
con $x,y>0$, esiste una **funzione conservata** $H(x,y)$ tale che
$\dot H = 0$. Cerchiamola.

Dividendo le due equazioni:

$$
\frac{dy}{dx}=\frac{y(-c+dx)}{x(a-by)}.
$$

Separazione di variabili:

$$
\frac{a-by}{y}\,dy = \frac{-c+dx}{x}\,dx
\;\Longleftrightarrow\;
\left(\frac{a}{y}-b\right)dy = \left(-\frac{c}{x}+d\right)dx.
$$

Integrando entrambi i membri:

$$
a\ln y - by = -c\ln x + dx + \text{cost}.
$$

Riarrangiando:

$$
\boxed{\;H(x,y) = dx - c\ln x + by - a\ln y = \text{cost}\;}
$$

Quindi le traiettorie di Lotka–Volterra giacciono sui **livelli di $H$**.
Verifichiamolo:

$$
\dot H = (d - c/x)\dot x + (b - a/y)\dot y
= (d - c/x)x(a-by) + (b - a/y)y(-c+dx)
= (dx - c)(a-by) + (by - a)(-c+dx),
$$

raccogliendo $(dx-c)$:

$$
\dot H = (dx-c)(a-by) - (dx-c)(a-by) = 0. \quad\square
$$

Conseguenze:
- Le orbite sono **chiuse** (livelli di $H$ sono curve chiuse attorno
  all'equilibrio $(c/d, a/b)$ → centro).
- Il **periodo** delle oscillazioni dipende dalla quale curva di livello
  si percorre (cioè dalle condizioni iniziali) → orbite non isocrone.
- Piccole perturbazioni del modello (es. risposta funzionale saturante
  come Holling II) rompono questa conservazione e possono creare cicli
  limite isolati.


In [ ]:
# Verifica numerica della conservazione di H(x,y) lungo le orbite di Lotka-Volterra
a_lv2, b_lv2, c_lv2, d_lv2 = 1.0, 0.5, 0.75, 0.25
def lv_check(t, z):
    x, y = z
    return np.array([a_lv2*x - b_lv2*x*y, -c_lv2*y + d_lv2*x*y])

def H(x, y):
    return d_lv2*x - c_lv2*np.log(x) + b_lv2*y - a_lv2*np.log(y)

# Integriamo da varie condizioni iniziali
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax_phase, ax_H = axes
for ic, c in zip([[1.0, 0.5], [2.0, 1.5], [4.0, 1.5]],
                  ["#1f4e79", "#1e8449", "#c0392b"]):
    sol = solve_ivp(lv_check, (0, 30), ic, dense_output=True, max_step=0.02,
                    rtol=1e-10, atol=1e-12)
    ax_phase.plot(sol.y[0], sol.y[1], color=c, lw=1.6, label=f"IC = {ic}")
    H_traj = H(sol.y[0], sol.y[1])
    ax_H.plot(sol.t, H_traj - H_traj[0], color=c, lw=1.6, label=f"IC = {ic}")

ax_phase.set_xlabel("prede x"); ax_phase.set_ylabel("predatori y")
ax_phase.set_title("Orbite chiuse di Lotka-Volterra"); ax_phase.legend()
ax_H.set_xlabel("tempo"); ax_H.set_ylabel("H(t) - H(0)")
ax_H.set_title("Conservazione di H lungo le traiettorie (~ 0)"); ax_H.legend()
plt.tight_layout(); plt.show()

print("Conservazione di H: scostamento massimo per la prima IC:",
      np.max(np.abs(H(sol.y[0], sol.y[1]) - H(sol.y[0,0], sol.y[1,0]))))


# Parte 7 — Modelli fisiologici

In questa parte applichiamo *tutto* il materiale precedente a modelli
quantitativi della fisiologia umana. Per ogni modello seguiamo lo stesso
schema:

**A. Contesto fisiologico** — perché il modello esiste, quale fenomeno
spiega.
**B. Variabili, parametri, unità** — tabella completa.
**C. Ipotesi e semplificazioni** — esplicitate.
**D. Derivazione** — dai bilanci di massa/carica/Kirchhoff alle equazioni.
**E. Forma matriciale** — quando possibile.
**F. Equilibri**.
**G. Linearizzazione** — Jacobiano se il modello è non lineare.
**H. Analisi di stabilità**.
**I. Implementazione "a mano"** — passo passo, didattica.
**J. Implementazione robusta con `solve_ivp`**.
**K. Grafici** — annotati.
**L. Interpretazione fisiologica dei grafici**.
**M. Esercizi**.

I modelli trattati:

1. Circolazione sistemica a 3 compartimenti.
2. Cardiovascolare con baroriflesso (controllo dinamico).
3. Scambio di soluto fra compartimenti.
4. Emodialisi (lineare e non lineare con shift osmotici).
5. Meccanica respiratoria.
6. Ventilazione alveolare e spazio morto.
7. Scambio dei gas.
8. Controllo chemocettoriale della ventilazione (Cheyne–Stokes).
9. Elettrofisiologia cellulare: Nernst, membrana RC.
10. Voltage clamp.
11. Modello di Hodgkin–Huxley e potenziale d'azione.


## 7.1 Circolazione sistemica a 3 compartimenti

### A. Contesto fisiologico

La circolazione sistemica è una "tubatura" complessa, ma a livello di
modello concentrato (parametri *lumped*) si può ridurre a tre compartimenti:

- **arteria sistemica** (compliance bassa $C_{sa}$, alta pressione $\sim 100$ mmHg);
- **vena sistemica** (compliance alta $C_{sv}$, pressione bassa $\sim 5$ mmHg);
- **atrio destro** (pressione $\sim 0$ mmHg).

Tra questi compartimenti scorre il sangue attraverso resistenze (arteriole
+ capillari + venule). Il cuore destro pompa dall'atrio destro verso le
arterie con portata $Q_{co}=K\,p_{ra}$ (semplificazione lineare).

### B. Variabili, parametri, unità

| Simbolo | Significato | Unità tipica |
|---|---|---|
| $p_{sa}$ | pressione arteriosa sistemica | mmHg |
| $p_{sv}$ | pressione venosa sistemica | mmHg |
| $p_{ra}$ | pressione atriale destra | mmHg |
| $C_{sa}, C_{sv}, C_{ra}$ | compliance | mL/mmHg |
| $R_{sa}, R_{sv}$ | resistenze sistemica e venosa | mmHg·s/mL |
| $K$ | "gain" cardiaco (Frank–Starling) | mL/s/mmHg |
| $I_{ext}$ | infusione/emorragia | mL/s (>0 infusione) |

### C. Ipotesi

1. Compartimenti puntiformi (parametri concentrati).
2. Resistenze lineari (legge di Ohm).
3. Frank–Starling lineare: $Q_{co}=K\,p_{ra}$.
4. Compliance costanti (no attivazione muscolare ai vasi).
5. Compliance dell'atrio destro $C_{ra}$ molto piccola (rigidità relativa).

### D. Derivazione

Per ogni compartimento, **bilancio di volume**:
$\dot V_i = Q_{in,i}-Q_{out,i}$. Usando $V_i=C_i\,p_i$:

$$
\begin{aligned}
C_{sa}\,\dot p_{sa} &= Q_{co} - (p_{sa}-p_{sv})/R_{sa} + I_{ext},\\
C_{sv}\,\dot p_{sv} &= (p_{sa}-p_{sv})/R_{sa} - (p_{sv}-p_{ra})/R_{sv},\\
C_{ra}\,\dot p_{ra} &= (p_{sv}-p_{ra})/R_{sv} - Q_{co},
\end{aligned}
$$

con $Q_{co}=K\,p_{ra}$. Lineare in $(p_{sa},p_{sv},p_{ra})$.

### E. Forma matriciale

$$
\dot{\mathbf{p}}=A\,\mathbf{p}+ B\,I_{ext},\qquad
\mathbf{p}=\begin{bmatrix} p_{sa}\\ p_{sv}\\ p_{ra}\end{bmatrix},
$$

con

$$
A=\begin{bmatrix}
-1/(R_{sa}C_{sa}) & 1/(R_{sa}C_{sa}) & K/C_{sa}\\
1/(R_{sa}C_{sv}) & -1/(R_{sa}C_{sv}) - 1/(R_{sv}C_{sv}) & 1/(R_{sv}C_{sv})\\
0 & 1/(R_{sv}C_{ra}) & -1/(R_{sv}C_{ra}) - K/C_{ra}
\end{bmatrix},\quad
B=\begin{bmatrix} 1/C_{sa}\\ 0\\ 0\end{bmatrix}.
$$

### F. Equilibrio (con $I_{ext}=I_0$)

$\mathbf{p}^*=-A^{-1}B I_0$. In condizioni nominali $p_{sa}^*\approx 95$ mmHg,
$p_{sv}^*\approx 5$ mmHg, $p_{ra}^*\approx 2$ mmHg.

### G–H. Stabilità

Tutti gli autovalori di $A$ hanno parte reale negativa → equilibrio
asintoticamente stabile. Lo verifichiamo numericamente.


In [ ]:
# === Modello cardiovascolare sistemico a 3 compartimenti ===
# Parametri "nominali" plausibili (ordini di grandezza didattici)
C_sa, C_sv, C_ra = 1.5, 50.0, 5.0   # mL/mmHg
R_sa, R_sv      = 0.95, 0.05         # mmHg*s/mL
K               = 4.0                # mL/s/mmHg (Frank-Starling)

A = np.array([
    [-1/(R_sa*C_sa),  1/(R_sa*C_sa),                       K/C_sa],
    [ 1/(R_sa*C_sv), -1/(R_sa*C_sv) - 1/(R_sv*C_sv),       1/(R_sv*C_sv)],
    [ 0.0,            1/(R_sv*C_ra),                       -1/(R_sv*C_ra) - K/C_ra],
])
B = np.array([1.0/C_sa, 0.0, 0.0])

print("Autovalori del modello cardiocircolatorio (devono avere Re<0):")
print(np.linalg.eigvals(A))

# Punto di equilibrio per infusione costante I_ext0
I_ext_baseline = 0.0   # niente infusione: equilibrio = 0... aggiungiamo gravita' modellata come offset
# Per avere un equilibrio fisiologico, aggiungiamo invece un "unstressed volume" effettivo
# tramite I_ext positivo finche' il sistema raggiunge ~ pressioni reali.
# Per semplicita': lasciamo come baseline I_ext=0 e applichiamo emorragia poi.

# Simulazione: a t=20 s applichiamo un'emorragia per 30 s
T = 80.0
def I_ext(t):
    if 20 <= t <= 50: return -25.0   # mL/s
    return 0.0

def rhs(t, p): return A @ p + B * I_ext(t)

p0 = np.array([95.0, 5.0, 2.0])
sol = solve_ivp(rhs, (0, T), p0, dense_output=True, max_step=0.05)

fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)
ax = axes[0]
ax.plot(sol.t, sol.y[0], color=COL["main"],   lw=2, label="$p_{sa}$ (arteriosa)")
ax.plot(sol.t, sol.y[1], color=COL["ok"],     lw=2, label="$p_{sv}$ (venosa)")
ax.plot(sol.t, sol.y[2], color=COL["accent"], lw=2, label="$p_{ra}$ (atrio dx)")
ax.axvspan(20, 50, alpha=0.15, color="red")
ax.text(35, 92, "emorragia\n-25 mL/s", ha="center", color="red", fontsize=9)
ax.set_ylabel("pressione [mmHg]"); ax.legend(loc="center right")
ax.set_title("Modello cardiovascolare lineare: risposta a un'emorragia")

ax = axes[1]
Q_co = K * sol.y[2]
ax.plot(sol.t, Q_co, color=COL["extra"], lw=2)
ax.axvspan(20, 50, alpha=0.15, color="red")
ax.set_ylabel("$Q_{co}$ [mL/s]"); ax.set_xlabel("tempo [s]")
ax.set_title("Portata cardiaca = $K\\cdot p_{ra}$")
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — durante l'emorragia (zona rossa) tutte le
pressioni scendono e la portata cardiaca crolla. Quando l'emorragia
finisce, il sistema **ritorna lentamente** verso l'equilibrio (autovalori
con parti reali negative).

**Limite del modello** — in un paziente reale c'è un **baroriflesso** che
*compenserebbe* l'emorragia aumentando resistenze e contrattilità.
Aggiungere questo controllo è la prossima sezione.


### Analogo elettrico del modello cardiocircolatorio

Disegniamo lo schema "equivalente" del modello cardiovascolare come rete di
$R$, $C$, generatori. È il modo più chiaro per leggere il sistema.


In [ ]:
# Schema del modello cardiovascolare a 3 compartimenti come rete RC
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.set_xlim(-0.5, 13); ax.set_ylim(-1.0, 5.5); ax.set_aspect("equal"); ax.axis("off")

# Linea superiore = nodo arterioso, venoso, atriale
y_top = 4.5
wire(ax, (1.0, y_top), (4.0, y_top))   # Psa
wire(ax, (5.5, y_top), (8.5, y_top))   # Psv (con R_sa nel mezzo)
wire(ax, (10.0, y_top), (12.0, y_top)) # Pra

# Nodo arterioso
ax.plot(2.5, y_top, "o", color="black", ms=6)
ax.text(2.5, y_top+0.35, "$P_{sa}$", color=COL["main"], ha="center", fontsize=12)
# Resistenza sistemica
resistor(ax, (4.0, y_top), (5.5, y_top), label="$R_{sa}$")
# Nodo venoso
ax.plot(7.0, y_top, "o", color="black", ms=6)
ax.text(7.0, y_top+0.35, "$P_{sv}$", color=COL["main"], ha="center", fontsize=12)
# Resistenza venosa
resistor(ax, (8.5, y_top), (10.0, y_top), label="$R_{sv}$")
# Nodo atriale
ax.plot(11.0, y_top, "o", color="black", ms=6)
ax.text(11.0, y_top+0.35, "$P_{ra}$", color=COL["main"], ha="center", fontsize=12)

# Compliance verticali verso terra
y_ground = 0.5
wire(ax, (0.5, y_top), (0.5, y_ground))
wire(ax, (12.5, y_top), (12.5, y_ground))
wire(ax, (0.5, y_ground), (12.5, y_ground))
ground(ax, (6.5, y_ground))

# C_sa (sotto nodo sa)
wire(ax, (2.5, y_top), (2.5, 3.4))
capacitor(ax, (2.5, 3.4), (2.5, 2.0), label="$C_{sa}$")
wire(ax, (2.5, 2.0), (2.5, y_ground))

# C_sv
wire(ax, (7.0, y_top), (7.0, 3.4))
capacitor(ax, (7.0, 3.4), (7.0, 2.0), label="$C_{sv}$")
wire(ax, (7.0, 2.0), (7.0, y_ground))

# C_ra
wire(ax, (11.0, y_top), (11.0, 3.4))
capacitor(ax, (11.0, 3.4), (11.0, 2.0), label="$C_{ra}$")
wire(ax, (11.0, 2.0), (11.0, y_ground))

# Cuore: generatore di corrente che porta sangue da P_ra a P_sa (Q_co = K*P_ra)
# Disegniamo come pompa = generatore di corrente controllato
wire(ax, (12.0, y_top), (12.5, y_top))
wire(ax, (12.5, y_top), (12.5, 4.0))
wire(ax, (12.5, 4.0), (-0.2, 4.0))
isource(ax, (-0.2, 4.0), (-0.2, y_top), label=r"$Q_{co}=K\,P_{ra}$")
wire(ax, (-0.2, y_top), (1.0, y_top))

# Iniezione/emorragia I_ext sul nodo sa
wire(ax, (2.5, y_top), (2.5, 5.4))
ax.annotate("", xy=(2.5, y_top + 0.05), xytext=(2.5, 5.4),
            arrowprops=dict(arrowstyle="->", color=COL["accent"], lw=1.5))
ax.text(2.7, 5.0, "$I_{ext}$ (infusione/emorragia)",
        color=COL["accent"], fontsize=10)

ax.set_title("Modello cardiocircolatorio sistemico: rete RC equivalente",
             fontsize=12)
plt.tight_layout(); plt.show()


**Come leggere lo schema** — i tre nodi superiori (in alto) sono le
pressioni $P_{sa}, P_{sv}, P_{ra}$. Ogni compartimento ha una *compliance*
$C$ (capacitore verso la terra "pressione 0") e gli scambi tra compartimenti
avvengono attraverso *resistenze* $R$. Il cuore destro è schematizzato come
una **sorgente di corrente controllata**: spinge $Q_{co}=K\,P_{ra}$ mL/s di
sangue dal nodo $P_{ra}$ al nodo $P_{sa}$ (qui per chiarezza il loop esce a
destra e rientra a sinistra). L'**emorragia** è modellata come $I_{ext}<0$
che esce dal nodo arterioso.

**Dallo schema alle equazioni** — KCL ai 3 nodi superiori produce
direttamente le tre ODE del modello. Ogni "$C_i\dot P_i$" sale dalla
capacità verso il nodo; le correnti orizzontali $P_i - P_j)/R$ scorrono
attraverso le resistenze.


### Quadro fisiologico e clinico del modello cardiovascolare

Il modello a 3 compartimenti è una **stilizzazione**: nel paziente reale ci
sono valvole (unidirezionalità del flusso), pulsatilità, geometria
ventricolare che cambia nel ciclo. Le modifiche cliniche dei parametri:

| Patologia | $R_{sa}$ | $K$ (gain cardiaco) | $C_{sa}$ |
|---|---|---|---|
| Ipertensione cronica | ↑↑ | normale o ↑ | ↓ (rigidità arteriosa) |
| Shock settico | ↓↓ | ↓ (depressione miocardica) | normale |
| Insufficienza cardiaca | ↑ | ↓↓ | ↓ |
| Emorragia massiva | normale → ↑ (baroriflesso) | ↑ (compenso) | normale |
| Beta-bloccanti | normale | ↓ | normale |
| Vasocostrittori (noradrenalina) | ↑ | normale | normale o ↑ |

### Misure cliniche delle variabili di stato

- $p_{sa}$ → bracciale o catetere arterioso (mmHg).
- $p_{ra}$ (CVP) → catetere venoso centrale (mmHg).
- $Q_{co}$ → termodiluizione, ecocardiografia, PiCCO.
- Compliance arteriosa → onda di polso, PWV (pulse wave velocity).

### Il ciclo pressione–volume ventricolare (PV-loop)

Il modello *lumped* non lo cattura, ma è essenziale per leggere
ecocardiografie. Disegniamo un PV-loop stilizzato del ventricolo sinistro
(modello di Suga–Sagawa).


In [ ]:
# PV-loop didattico del ventricolo sinistro (Suga-Sagawa)
# Linee di riferimento:
#   - ESPVR (End-Systolic Pressure-Volume Relation): retta P = Ees*(V - V0)
#   - EDPVR (End-Diastolic): curva passiva non lineare
V = np.linspace(20, 200, 400)
Ees = 2.5
V0_es = 15
ESPVR = Ees * (V - V0_es)
EDPVR = 0.5 * (np.exp(0.02*(V - 20)) - 1)

EDV, ESV = 130.0, 60.0
P_dia = 0.5 * (np.exp(0.02*(EDV - 20)) - 1)
P_aor = 80.0

# Costruisco il poligono del loop (4 fasi)
loop_V, loop_P = [], []
V_fill = np.linspace(ESV, EDV, 60)
P_fill = 0.5 * (np.exp(0.02*(V_fill - 20)) - 1)
loop_V += list(V_fill); loop_P += list(P_fill)
loop_V += [EDV]*30; loop_P += list(np.linspace(P_dia, P_aor, 30))
V_eject = np.linspace(EDV, ESV, 60)
P_eject = P_aor + 30*np.sin(np.linspace(0, np.pi, 60))
loop_V += list(V_eject); loop_P += list(P_eject)
loop_V += [ESV]*30; loop_P += list(np.linspace(P_eject[-1], P_dia, 30))

fig, ax = plt.subplots(figsize=(9.5, 6.5))
ax.plot(V, ESPVR, color=COL["accent"], lw=1.5, ls="--",
        label=r"ESPVR (end-syst.) — pendenza $E_{es}$ = contrattilità")
ax.plot(V, EDPVR, color=COL["ok"], lw=1.5, ls="--",
        label="EDPVR (end-diast.) — compliance passiva")
ax.plot(loop_V, loop_P, color=COL["main"], lw=2.2, label="ciclo PV")
ax.set_xlim(0, 200); ax.set_ylim(-5, 160)
ax.annotate("riempimento\n(diastole)", xy=(100, 5), xytext=(60, 20),
            arrowprops=dict(arrowstyle="->"))
ax.annotate("contrazione\nisovolumica", xy=(130, 45), xytext=(150, 35),
            arrowprops=dict(arrowstyle="->"))
ax.annotate("eiezione", xy=(95, 110), xytext=(130, 120),
            arrowprops=dict(arrowstyle="->"))
ax.annotate("rilassamento\nisovolumico", xy=(60, 50), xytext=(20, 70),
            arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("Volume ventricolare [mL]")
ax.set_ylabel("Pressione ventricolare [mmHg]")
ax.set_title("PV-loop del ventricolo sinistro (Suga–Sagawa)")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()


**Cosa rappresenta** — il ciclo è percorso *in senso orario* a ogni
battito. Le 4 fasi:

1. **Riempimento ventricolare** (orizzontale in basso): il sangue entra
   dall'atrio sinistro; $V$ cresce lungo la curva EDPVR (compliance
   passiva del miocardio).
2. **Contrazione isovolumica**: le valvole sono chiuse; $V$ costante a
   $EDV$; pressione sale rapidamente.
3. **Eiezione**: la valvola aortica si apre; $V$ scende mentre $P$ resta
   alta; la pressione finale (ESP) sta sulla retta ESPVR.
4. **Rilassamento isovolumico**: valvola aortica chiusa, $V$ resta a
   $ESV$, pressione scende rapidamente.

**Indici clinici derivati dal PV-loop**:

- **Stroke Volume**: $SV = EDV - ESV$ (volume eiettato per battito);
- **Frazione di eiezione**: $EF = SV/EDV$ (normale > 0.55, scompenso < 0.40);
- **Lavoro cardiaco**: area racchiusa dal loop (≈ stroke work);
- **Contrattilità**: pendenza $E_{es}$ della ESPVR (indipendente dal carico).

**Connessione con il modello a 3 compartimenti** — il modello lumped *non
separa* sistole e diastole: usa una relazione media $Q_{co}=K\,p_{ra}$ che
approssima la legge di Frank–Starling (più riempimento → più gittata). Per
modellare il PV-loop serve aggiungere una variabile *elastanza temporale*
$E(t)$ (modello di Suga).


## 7.2 Baroriflesso: cardiovascolare con controllo dinamico

### A. Contesto fisiologico

Il sistema nervoso autonomo regola **dinamicamente** il sistema cardiocircolatorio
per mantenere $p_{sa}$ vicina a un setpoint:

- aumenta la **resistenza sistemica** $R_{sa}$ (vasocostrizione);
- aumenta il **gain cardiaco** $K$ (inotropismo).

Modelliamo entrambe come stati dinamici di primo ordine con costante di tempo
$\tau$ e set-point dipendente da $p_{sa}$.

### B–C. Variabili e ipotesi

Aggiungiamo due stati di controllo $R_{sa}(t)$ e $K(t)$:

$$
\tau_R\,\dot R_{sa} = R_{sa}^{ss}(p_{sa}) - R_{sa},\qquad
\tau_K\,\dot K     = K^{ss}(p_{sa}) - K,
$$

con le funzioni stazionarie a sigmoide (saturazione):

$$
R_{sa}^{ss}(p) = R_{sa}^{0} + \Delta R\,\sigma\!\big(\tfrac{p^* - p}{\sigma_p}\big),\quad
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

$\Delta R$ è il range di adattamento, $p^*$ il setpoint, $\sigma_p$ la
sensibilità (mmHg per unità di sigmoide).

Il sistema completo è **non lineare** ma è ancora gestibile con `solve_ivp`.


In [ ]:
# === Modello cardiovascolare con baroriflesso ===
# Parametri "nominali"
C_sa, C_sv, C_ra = 1.5, 50.0, 5.0
R_sv = 0.05
tau_R, tau_K = 6.0, 10.0     # costanti di tempo del baroriflesso (s)
R0, dR = 0.5, 1.0            # R_sa nominale e ampiezza di modulazione
K0, dK = 3.0, 3.0            # gain cardiaco nominale e modulazione
p_setpoint = 95.0
sigma_p = 8.0

def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

def Rsa_ss(p_sa): return R0 + dR * sigmoid((p_setpoint - p_sa)/sigma_p)
def K_ss(p_sa):   return K0 + dK * sigmoid((p_setpoint - p_sa)/sigma_p)

def cv_baro(t, X, I_ext_fn):
    p_sa, p_sv, p_ra, R_sa, K = X
    Q_co = K * p_ra
    dpsa = (Q_co - (p_sa - p_sv)/R_sa + I_ext_fn(t)) / C_sa
    dpsv = ((p_sa - p_sv)/R_sa - (p_sv - p_ra)/R_sv) / C_sv
    dpra = ((p_sv - p_ra)/R_sv - Q_co) / C_ra
    dRsa = (Rsa_ss(p_sa) - R_sa) / tau_R
    dK   = (K_ss(p_sa)   - K)   / tau_K
    return np.array([dpsa, dpsv, dpra, dRsa, dK])

# Confronto: con e senza baroriflesso. Senza = blocchiamo R_sa e K a costanti.
def cv_no_baro(t, X, I_ext_fn):
    p_sa, p_sv, p_ra = X[:3]
    R_sa = R0 + dR*0.5    # valore nominale fisso
    K    = K0 + dK*0.5
    Q_co = K * p_ra
    dpsa = (Q_co - (p_sa - p_sv)/R_sa + I_ext_fn(t)) / C_sa
    dpsv = ((p_sa - p_sv)/R_sa - (p_sv - p_ra)/R_sv) / C_sv
    dpra = ((p_sv - p_ra)/R_sv - Q_co) / C_ra
    return np.array([dpsa, dpsv, dpra, 0.0, 0.0])

T = 120.0
def I_ext(t):
    if 30 <= t <= 70: return -20.0
    return 0.0

X0 = np.array([95.0, 5.0, 2.0, R0+dR*0.5, K0+dK*0.5])
sol_b = solve_ivp(lambda t, X: cv_baro(t, X, I_ext),   (0, T), X0, dense_output=True, max_step=0.05)
sol_n = solve_ivp(lambda t, X: cv_no_baro(t, X, I_ext),(0, T), X0, dense_output=True, max_step=0.05)

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
ax = axes[0]
ax.plot(sol_b.t, sol_b.y[0], color=COL["main"], lw=2, label="$p_{sa}$ CON baroriflesso")
ax.plot(sol_n.t, sol_n.y[0], color=COL["accent"], lw=2, ls="--", label="$p_{sa}$ SENZA")
ax.axhline(p_setpoint, color="gray", ls=":")
ax.axvspan(30, 70, color="red", alpha=0.1)
ax.set_ylabel("$p_{sa}$ [mmHg]"); ax.legend()
ax.set_title("Effetto del baroriflesso durante emorragia (zona rossa)")

ax = axes[1]
ax.plot(sol_b.t, sol_b.y[3], color=COL["main"], lw=2, label="$R_{sa}(t)$ con baroriflesso")
ax.axhline(R0+dR*0.5, color=COL["accent"], ls="--", label="$R_{sa}$ fisso (senza)")
ax.set_ylabel("$R_{sa}$ [mmHg·s/mL]"); ax.legend()

ax = axes[2]
ax.plot(sol_b.t, sol_b.y[4], color=COL["main"], lw=2, label="$K(t)$")
ax.axhline(K0+dK*0.5, color=COL["accent"], ls="--", label="$K$ fisso")
ax.set_ylabel("$K$ [mL/s/mmHg]"); ax.set_xlabel("tempo [s]"); ax.legend()
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — con baroriflesso, in risposta all'emorragia il
sistema *aumenta* sia $R_{sa}$ (vasocostrizione) sia $K$ (contrattilità);
risultato: la $p_{sa}$ scende molto meno e si riassesta vicino al setpoint.
Senza baroriflesso, $p_{sa}$ crolla.

**Cosa cambia se varia il guadagno** $\Delta R, \Delta K$ — un guadagno alto
compensa di più ma può rendere il loop oscillante. Provate a raddoppiare
$\Delta K$: si vedono sovraregolazioni.

**Perché è importante** — questo è il **primo esempio** di sistema
fisiologico con un controllore biologico che genera dinamica di
adattamento, e introduce il concetto fondamentale di *setpoint* + *attuatore lento*.


### Anatomia e fisiologia del baroriflesso (per davvero)

Il **baroriflesso arterioso** parte dai meccanocettori nel **seno carotideo**
(biforcazione della carotide comune) e nell'**arco aortico**. Sono
"stiramento-sensibili": più la parete si distende per la pressione, più
scaricano. Le fibre afferenti viaggiano:

- dal seno carotideo nel **nervo glossofaringeo** (IX);
- dall'arco aortico nel **nervo vago** (X);

fino al **nucleo del tratto solitario (NTS)** nel bulbo, che proietta sui
nuclei autonomici (RVLM, NA) regolando simpatico e parasimpatico.

Risposta canonica a una **caduta** di $p_{sa}$:

| Fase | Cosa succede |
|---|---|
| 1 | ↓ scarica afferente dai barocettori |
| 2 | ↓ inibizione del simpatico, ↓ inibizione del parasimpatico nel NTS |
| 3 | ↑ tono simpatico cardiaco e periferico, ↓ tono vagale |
| 4 | ↑ FC, ↑ contrattilità (gain $K$), ↑ resistenze arteriolari ($R_{sa}$), ↑ venocostrizione |

**Costanti di tempo** (motivo per cui modelliamo con $\tau_R, \tau_K$ diversi):

- vasocostrizione (alfa-1 sui vasi): 5–10 s;
- cronotropia/inotropia (beta-1 cardiaco): 1–3 s, più veloce;
- riassorbimento renale di Na e attivazione del RAAS: minuti–ore.

Il modello a 2 stati di controllo collassa questo spettro; per ricerca si
separano almeno simpatico veloce e RAAS lento.

### Casi clinici dove il baroriflesso "fallisce"

- **Disautonomia diabetica**: cadono i guadagni → **ipotensione ortostatica** (sviene quando si alza).
- **Shock spinale**: blocco del simpatico → ipotensione + paradossale bradicardia (vago intatto).
- **Terapia con beta-bloccanti**: limita la risposta cronotropa → fatica adattiva sotto sforzo.
- **Atrofia multisistemica / Parkinson avanzato**: degenerazione NTS → perdita completa di compensazione barorecettoriale.

### Perché una sigmoide?

Nel modello abbiamo usato $\sigma(z)=1/(1+e^{-z})$ per le funzioni
$R_{sa}^{ss}(p_{sa})$ e $K^{ss}(p_{sa})$. Questo non è arbitrario: in
fisiologia *quasi tutte* le relazioni dose-risposta sono **a S**, perché:

- a basse $p$ il sistema satura "verso l'alto" (massimo sforzo simpatico);
- a $p$ vicine al set-point la pendenza è massima (massima sensibilità);
- a $p$ molto alte satura "verso il basso" (massimo rilassamento).

La curva di Hill $y(x)=x^n/(K^n + x^n)$ è la versione "ad ascisse positive"
della stessa idea. Vediamole insieme: l'idea è la stessa che useremo per
saturazione dell'emoglobina, gating di canali ionici, legame
recettore–ligando.


In [ ]:
# Sigmoide logistica vs funzione di Hill: visualizzazione comparativa
z = np.linspace(-6, 6, 400)
sig = 1/(1 + np.exp(-z))

# Hill su x > 0
x = np.linspace(0, 10, 400)
K = 3.0
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))

ax = axes[0]
ax.plot(z, sig, color=COL["main"], lw=2.4)
ax.axhline(0.5, color="gray", ls=":")
ax.axvline(0, color="gray", ls=":")
ax.set_xlabel("z"); ax.set_ylabel(r"$\sigma(z)$")
ax.set_title("Sigmoide logistica $\\sigma(z)=1/(1+e^{-z})$")

ax = axes[1]
for n, c in zip([1, 2, 4, 8, 20],
                 [COL["main"], COL["ok"], COL["warn"], COL["accent"], COL["extra"]]):
    y = x**n / (K**n + x**n)
    ax.plot(x, y, color=c, lw=2, label=f"n={n}")
ax.axvline(K, color="gray", ls=":")
ax.text(K+0.2, 0.05, f"$K={K}$", color="gray")
ax.set_xlabel("x"); ax.set_ylabel("y(x)")
ax.set_title("Funzione di Hill al variare di $n$ (cooperatività)")
ax.legend()

# Curva di dissociazione dell'emoglobina
PO2 = np.linspace(0, 100, 400)
P50 = 27.0; n_Hb = 2.7
SaO2 = PO2**n_Hb / (P50**n_Hb + PO2**n_Hb)
ax = axes[2]
ax.plot(PO2, SaO2, color=COL["main"], lw=2.2)
ax.axhline(0.5, color="gray", ls=":")
ax.axvline(P50, color="gray", ls=":")
ax.plot(P50, 0.5, "o", color=COL["accent"], ms=10)
ax.annotate(f"$P_{{50}}={P50}$ mmHg", xy=(P50, 0.5), xytext=(P50+10, 0.4),
            arrowprops=dict(arrowstyle="->", color=COL["accent"]),
            color=COL["accent"])
ax.axhline(0.98, color="gray", lw=0.5)
ax.text(70, 0.99, "98% (arteriosa)", color="gray", fontsize=9)
ax.axhline(0.75, color="gray", lw=0.5)
ax.text(35, 0.76, "75% (venosa)", color="gray", fontsize=9)
ax.set_xlabel("$P_{O_2}$ [mmHg]"); ax.set_ylabel("$S_{aO_2}$")
ax.set_title("Dissociazione emoglobina (Hill, $n=2.7$)")
plt.tight_layout(); plt.show()


**Connessione con tutti i modelli del notebook**:

- **Baroriflesso** (sezione 7.2): saturazione di $R_{sa}$ e $K$ tramite $\sigma$.
- **Variabili di gating HH** (sezione 7.10): $m_\infty(V), h_\infty(V), n_\infty(V)$ sono sigmoidi in $V$.
- **Curva ventilatoria** (sezione 7.6): in versione realistica è una sigmoide $\dot V_A(P_{ACO_2})$ con saturazione superiore.
- **Curva di Hb** (sopra): cooperatività delle 4 subunità → $n\approx 2.7$.

> *Quando in un grafico vedi una S, c'è sempre una **storia di saturazione**
> dietro. Le sigmoidi sono i mattoni delle leggi non lineari della
> fisiologia.*


## 7.3 Scambio di soluto tra due compartimenti

### A–C. Contesto e ipotesi

Due compartimenti (es. intracellulare e extracellulare, oppure plasma e
interstizio) separati da una membrana semipermeabile a un soluto. Volumi
$V_1, V_2$ costanti. Concentrazioni $c_1, c_2$. Flusso di soluto
proporzionale al gradiente (legge di Fick semplificata):

$$
J = P_s\,A\,(c_1 - c_2),
$$

con $P_s$ permeabilità del soluto e $A$ area di membrana. Bilanci di massa:

$$
V_1\,\dot c_1 = -J,\qquad V_2\,\dot c_2 = +J.
$$

In forma compatta, $\dot c_1 = -k\,(c_1-c_2)$, $\dot c_2=+k'(c_1-c_2)$ con
$k=P_s A/V_1$, $k'=P_s A/V_2$.

### D–F. Equilibrio e quantità conservata

La somma $V_1 c_1 + V_2 c_2$ (massa totale) si conserva. Sottraendo le due
equazioni si ottiene $\dot{(c_1-c_2)}=-(k+k')(c_1-c_2)$ → la **differenza**
decade esponenzialmente con costante di tempo $\tau=1/(k+k')$. All'equilibrio
$c_1=c_2=c_\infty$ con $c_\infty=(V_1c_1(0)+V_2c_2(0))/(V_1+V_2)$.


In [ ]:
# Scambio di soluto a 2 compartimenti
P_s_A = 0.05        # cm^3/s -> permeabilita' * area
V1, V2 = 1.0, 4.0   # L

def transport(t, c):
    c1, c2 = c
    J = P_s_A * (c1 - c2)
    return np.array([-J/V1, J/V2])

T = 60
sol = solve_ivp(transport, (0, T), [10.0, 0.0], dense_output=True, max_step=0.1)

c_inf = (V1*10 + V2*0) / (V1 + V2)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sol.t, sol.y[0], color=COL["main"],   lw=2, label="$c_1$ (compartimento piccolo)")
ax.plot(sol.t, sol.y[1], color=COL["accent"], lw=2, label="$c_2$ (compartimento grande)")
ax.axhline(c_inf, color="gray", ls=":", label=f"$c_\\infty={c_inf:.2f}$")
# Massa totale (deve essere costante)
mass = V1*sol.y[0] + V2*sol.y[1]
ax.plot(sol.t, mass / (V1 + V2), color="black", lw=1, ls="-.", alpha=0.5,
        label="massa totale / (V1+V2) (cost.)")
ax.set_xlabel("tempo [s]"); ax.set_ylabel("conc. [mmol/L]")
ax.set_title("Equilibrazione di un soluto tra 2 compartimenti (Fick)")
ax.legend(); plt.tight_layout(); plt.show()

print(f"Costante di tempo teorica: tau = {1/(P_s_A*(1/V1 + 1/V2)):.2f} s")


## 7.4 Emodialisi: due compartimenti con osmolarità

### A. Contesto fisiologico

In emodialisi un paziente con insufficienza renale viene "depurato" facendo
scorrere il sangue (compartimento extracellulare effettivo: $V_e$) attraverso
una membrana semipermeabile a contatto con una soluzione dializzante (volume
$V_d$, ricco di Na, K, ecc., concentrazioni di urea ≈ 0). Il modello classico
considera due compartimenti corporei (intracellulare $V_i$, extracellulare $V_e$)
e si concentra su urea (soluto principale da rimuovere) e su volume per gli
effetti osmotici.

### B. Variabili e parametri

| Simbolo | Significato | Unità |
|---|---|---|
| $c_i, c_e$ | conc. urea intra/extra | mmol/L |
| $V_i, V_e$ | volumi compartimenti | L |
| $K_{ie}$ | clearance tra i comparti | L/min |
| $K_d$ | clearance del dializzatore | L/min |
| $c_d$ | conc. urea nel dializzante (≈ 0) | mmol/L |
| $Q_{UF}$ | ultrafiltrazione (sottrazione di volume) | L/min |

### D. Bilanci (caso lineare, volumi costanti)

$$
\begin{aligned}
V_i\,\dot c_i &= K_{ie}(c_e - c_i),\\
V_e\,\dot c_e &= K_{ie}(c_i - c_e) - K_d(c_e - c_d).
\end{aligned}
$$

In assenza di ultrafiltrazione e con $c_d=0$:

$$
\begin{bmatrix}\dot c_i\\ \dot c_e\end{bmatrix}=
\begin{bmatrix} -K_{ie}/V_i & K_{ie}/V_i\\ K_{ie}/V_e & -(K_{ie}+K_d)/V_e\end{bmatrix}
\begin{bmatrix} c_i\\ c_e\end{bmatrix}.
$$

### F. Equilibrio (con $c_d=0$): $c_i=c_e=0$. Stabile (autovalori negativi).

### Caso non lineare: shift osmotico di volume

Se varia la concentrazione di soluti osmoticamente attivi, **acqua passa**
tra i compartimenti per mantenere l'osmolarità. La versione non lineare
include:

$$
\begin{aligned}
\dot V_i &= P_w\,A\,(\Pi_i - \Pi_e),\\
\dot V_e &= -\dot V_i - Q_{UF},
\end{aligned}
$$

con $\Pi$ osmolarità totale. La concentrazione si calcola sempre come
"massa / volume", quindi i bilanci diventano:

$$
\frac{d(V_i c_i)}{dt}=K_{ie}(c_e-c_i),\qquad
\frac{d(V_e c_e)}{dt}=K_{ie}(c_i-c_e)-K_d(c_e-c_d).
$$

Per chiarezza didattica simuliamo prima il **caso lineare** (volumi costanti),
poi il **caso non lineare** con shift osmotico.


In [ ]:
# === Emodialisi: schema a due compartimenti (illustrazione) ===
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.set_xlim(-0.5, 13); ax.set_ylim(-0.5, 5); ax.set_aspect("equal"); ax.axis("off")

# Compartimento intracellulare
intra = Rectangle((0.5, 1.0), 3.5, 3.0, fc="#d4efdf", ec=COL["ok"], lw=1.8, alpha=0.6)
ax.add_patch(intra)
ax.text(2.25, 3.7, "INTRACELL.\n$V_i$", ha="center", color=COL["ok"], fontsize=11, fontweight="bold")
ax.text(2.25, 2.5, "$c_i$", ha="center", color="black", fontsize=14)

# Compartimento extracellulare
extra = Rectangle((4.8, 1.0), 3.5, 3.0, fc="#d6eaf8", ec=COL["main"], lw=1.8, alpha=0.6)
ax.add_patch(extra)
ax.text(6.55, 3.7, "EXTRACELL.\n$V_e$", ha="center", color=COL["main"], fontsize=11, fontweight="bold")
ax.text(6.55, 2.5, "$c_e$", ha="center", color="black", fontsize=14)

# Dialysate
dial = Rectangle((9.0, 1.0), 3.5, 3.0, fc="#fdebd0", ec=COL["warn"], lw=1.8, alpha=0.6)
ax.add_patch(dial)
ax.text(10.75, 3.7, "DIALIZZATORE\n$V_d$", ha="center", color=COL["warn"], fontsize=11, fontweight="bold")
ax.text(10.75, 2.5, "$c_d\\approx 0$", ha="center", color="black", fontsize=14)

# Membrane (linee tratteggiate verticali)
ax.plot([4.0, 4.0], [1.0, 4.0], color="black", ls="--", lw=1.5)
ax.plot([4.8, 4.8], [1.0, 4.0], color="black", ls="--", lw=1.5)
ax.text(4.4, 0.6, "membrana\ncellulare", ha="center", fontsize=8, color=COL["grey"])
ax.plot([8.3, 8.3], [1.0, 4.0], color="black", ls="--", lw=1.5)
ax.plot([9.0, 9.0], [1.0, 4.0], color="black", ls="--", lw=1.5)
ax.text(8.65, 0.6, "membrana\ndialisi", ha="center", fontsize=8, color=COL["grey"])

# Frecce di scambio
draw_arrow(ax, (4.05, 3.2), (4.75, 3.2), color=COL["accent"], label="$K_{ie}(c_i-c_e)$", offset=(0, 0.15))
draw_arrow(ax, (8.35, 3.2), (8.95, 3.2), color=COL["accent"], label="$K_d(c_e-c_d)$", offset=(0, 0.15))
# Acqua osmotica
draw_arrow(ax, (4.5, 1.6), (4.3, 1.6), color="#2874a6", label="acqua (osmosi)", offset=(0, -0.4))
# Ultrafiltrazione
draw_arrow(ax, (8.5, 1.6), (8.8, 1.6), color="#7d3c98", label="$Q_{UF}$", offset=(0.4, -0.4))

ax.set_title("Modello compartimentale di emodialisi", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# === Emodialisi: simulazione del caso LINEARE (volumi fissi) ===
K_ie = 0.7   # L/min
K_d  = 0.3   # L/min
V_i, V_e = 25.0, 14.0   # L (uomo 70 kg ca.)
c_d = 0.0
c_i0, c_e0 = 25.0, 25.0   # mmol/L (uremico)

A_dial = np.array([
    [-K_ie/V_i,         K_ie/V_i],
    [ K_ie/V_e, -(K_ie + K_d)/V_e],
])
B_dial = np.array([0.0, K_d/V_e * c_d])

T = 240   # min, 4 ore
def rhs_lin(t, c): return A_dial @ c + B_dial

sol_lin = solve_ivp(rhs_lin, (0, T), [c_i0, c_e0], dense_output=True, max_step=0.5)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sol_lin.t, sol_lin.y[0], color=COL["main"],   lw=2, label="$c_i$ intracellulare")
ax.plot(sol_lin.t, sol_lin.y[1], color=COL["accent"], lw=2, label="$c_e$ extracellulare")
ax.set_xlabel("tempo [min]"); ax.set_ylabel("urea [mmol/L]")
ax.set_title("Emodialisi (modello lineare): rimozione di urea in 4 h")
ax.legend(); plt.tight_layout(); plt.show()

print("Autovalori del sistema lineare:", np.linalg.eigvals(A_dial))


In [ ]:
# === Emodialisi NON LINEARE con shift osmotico di volume ===
# Per evitare stiffness estrema modelliamo direttamente lo shift d'acqua come
# rilassamento veloce verso l'iso-osmolarita': la membrana cellulare a passare
# acqua e' molto piu' rapida rispetto alla scala temporale della dialisi.
# (P_w grande ma stiffness gestita con metodo implicito BDF.)
P_w       = 0.005      # L/min per (mOsm/L) - osmotic permeability surface-area product
K_d_urea  = 0.3        # L/min - clearance dializzatore per urea
K_ie_urea = 0.7        # L/min - exchange cell-blood
K_ie_Na   = 0.3        # L/min - exchange Na cell-blood (per la quota mobile)
c_d_urea  = 0.0        # urea nel dialisato
c_d_Na    = 140.0      # Na nel dialisato (fisiologico)
K_d_Na    = 0.1        # piccola clearance Na (Na ovviamente passa attraverso la membrana)

def hemodialysis(t, X, Q_UF=0.005):
    Vi, Ve, mUi, mUe, mNi, mNe = X
    # protezione numerica
    Vi = max(Vi, 0.5); Ve = max(Ve, 0.5)
    cUi, cUe = mUi/Vi, mUe/Ve
    cNi, cNe = mNi/Vi, mNe/Ve
    # osmolarita' (urea + sodio, fattore 2 per la dissociazione di NaCl)
    Pi_i = cUi + 2*cNi
    Pi_e = cUe + 2*cNe
    Jw   = P_w * (Pi_i - Pi_e)         # >0 -> acqua da intra a extra
    dVi  = -Jw
    dVe  = +Jw - Q_UF                   # Q_UF rimuove acqua da extra
    # Soluti (bilancio in mmol)
    dmUi = -K_ie_urea*(cUi - cUe)
    dmUe = +K_ie_urea*(cUi - cUe) - K_d_urea*(cUe - c_d_urea)
    dmNi = -K_ie_Na*(cNi - cNe)
    dmNe = +K_ie_Na*(cNi - cNe) - K_d_Na*(cNe - c_d_Na)
    return np.array([dVi, dVe, dmUi, dmUe, dmNi, dmNe])

# Stato iniziale: paziente uremico con eccesso di volume nell'extracellulare
Vi0, Ve0 = 25.0, 17.0     # 3 L di eccesso in Ve rispetto al normale
cU0 = 30.0                # urea alta
cN0 = 138.0
X0 = np.array([Vi0, Ve0, cU0*Vi0, cU0*Ve0, cN0*Vi0, cN0*Ve0])

T = 240
sol = solve_ivp(lambda t,X: hemodialysis(t,X, Q_UF=0.005),
                (0, T), X0, method="BDF",
                rtol=1e-6, atol=1e-9, dense_output=True)

Vi_t, Ve_t = sol.y[0], sol.y[1]
cUi_t = sol.y[2]/Vi_t; cUe_t = sol.y[3]/Ve_t
cNi_t = sol.y[4]/Vi_t; cNe_t = sol.y[5]/Ve_t

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
ax = axes[0]
ax.plot(sol.t, cUi_t, color=COL["main"],   lw=2, label="urea intra")
ax.plot(sol.t, cUe_t, color=COL["accent"], lw=2, label="urea extra")
ax.set_ylabel("urea [mmol/L]"); ax.legend()
ax.set_title("Emodialisi non lineare con shift osmotico di volume")

ax = axes[1]
ax.plot(sol.t, cNi_t, color=COL["main"],   lw=2, label="Na intra")
ax.plot(sol.t, cNe_t, color=COL["accent"], lw=2, label="Na extra")
ax.set_ylabel("[Na+] [mmol/L]"); ax.legend()

ax = axes[2]
ax.plot(sol.t, Vi_t, color=COL["main"],   lw=2, label="$V_i$ intra")
ax.plot(sol.t, Ve_t, color=COL["accent"], lw=2, label="$V_e$ extra")
ax.set_ylabel("volume [L]"); ax.set_xlabel("tempo [min]"); ax.legend()
plt.tight_layout(); plt.show()

print(f"Integrazione: success={sol.success}, n_steps={len(sol.t)}")


**Cosa mostra il grafico** — la rimozione di urea è esponenziale; il
sodio cambia poco; i volumi si riassestano per effetto dell'osmosi
indotta dalla rimozione di urea (un soluto osmoticamente attivo) e per
effetto della ultrafiltrazione. Nei pazienti reali questo squilibrio
osmotico può causare il "**disequilibrium syndrome**": l'urea esce
velocemente dall'extracellulare ma più lentamente dall'intracellulare
(soprattutto cerebrale), creando ipertonia intracellulare e shift d'acqua.

**Cosa cambia col dialisato** — abbassando il sodio del dialisato si
sottrae sodio al paziente; alzandolo si trattiene sodio (e acqua per
osmosi). È un trade-off clinico.


### Cinetiche multi-compartimentali dell'urea e sindrome da disequilibrio

Il modello a 2 compartimenti è la **base concettuale** del $Kt/V$
(parametro di adeguatezza dialitica usato in clinica), ma l'urea in realtà
ha cinetica **trifasica**:

| Compartimento | Costante di tempo | Note |
|---|---|---|
| plasma | minuti | rapidissimo |
| interstizio + ECF | decine di minuti | flusso ematico-dipendente |
| intracellulare (specialmente muscolo) | ore | il vero "limite" |

Il fenomeno del **rebound post-dialisi** — l'urea plasmatica risale dopo
la fine della seduta — è la conseguenza dell'equilibrazione lenta dal
compartimento intracellulare. Modelli **single-pool** sottostimano la dose
*effettiva* di urea raggiunta nel paziente; per questo si usa l'indice
$Kt/V^{eq}$ (equilibrated).

### Sindrome da disequilibrio dialitico (DDS)

Sintomi: cefalea, nausea, agitazione, fino a convulsioni nei casi gravi.
Meccanismo passo-passo:

1. L'urea è rimossa rapidamente dall'**ECF** (dializzatore efficiente).
2. Resta in cellula, specialmente nei **neuroni** (bassa permeabilità della BBB all'urea, presenza di osmoliti idiogenici).
3. Osmolarità intracellulare > ECF → **acqua entra nei neuroni**.
4. Edema cerebrale → sintomi neurologici.

**Prevenzione clinica**:

- prima dialisi del paziente uremico: clearance ridotte, sedute brevi ("ramping up");
- dialisato con **sodio elevato** per ridurre il gradiente osmotico (mannitolo nel dialisato in casi estremi);
- dialisi più **frequenti** e meno aggressive piuttosto che lunghe e intense.

### Effetto della concentrazione di K$^+$ nel dialisato

La $[K^+]_d$ è uno dei parametri **clinicamente più critici**:

| $[K^+]_d$ | Quando si usa | Rischio |
|---|---|---|
| 0–1 mEq/L | iperkaliemia severa con minaccia aritmica | aritmie da iperpolarizzazione cardiaca |
| 2 mEq/L | iperkaliemia grave | come sopra, meno marcato |
| 3 mEq/L | standard quotidiano | bilanciato |
| 4 mEq/L | normokaliemia, dieta-controllata | poca rimozione → progressione iperkaliemia |

Tutto si riconduce all'**equazione di Nernst** (sezione 7.8): $E_K$
dipende da $[K^+]_o$. Una rimozione troppo rapida porta $E_K$ a valori più
negativi → membrane cardiache iperpolarizzate → blocchi di conduzione e
aritmie pericolose. Nei pazienti con farmaci che prolungano il QT
(amiodarone) il margine è ancora più stretto.

### Indici di adeguatezza usati in clinica

- **$Kt/V$**: clearance × tempo / volume di distribuzione. Target $\ge 1.2$ per dialisi trisettimanale, $\ge 1.8$ giornaliera.
- **URR (Urea Reduction Ratio)**: $1 - c_{e,\text{post}}/c_{e,\text{pre}}$. Target $\ge 0.65$.
- **$\beta_2$-microglobulina**: marker di rimozione "media molecola"; correlata a mortalità a lungo termine.


## 7.5 Meccanica respiratoria

### A. Contesto fisiologico

L'apparato respiratorio è una rete di tubi (laringe, trachea, bronchi)
che termina negli alveoli. Modello *lumped* a 5 nodi: bocca/larynx,
trachea, bronchi, alveoli, "pleura" (driver di pressione esterna).
Ognuno è un compartimento con compliance, separato dal successivo da una
resistenza.

### B–C. Variabili e ipotesi

| Simbolo | Significato | Unità |
|---|---|---|
| $P_m$ | pressione alla bocca | cmH$_2$O |
| $P_l$ | laringe | cmH$_2$O |
| $P_b$ | bronchi | cmH$_2$O |
| $P_A$ | alveolare | cmH$_2$O |
| $P_{pl}$ | pleurica (driver) | cmH$_2$O |
| $R_i$ | resistenze viste in serie tra nodi | cmH$_2$O·s/L |
| $C_i$ | compliance dei "vasi d'aria" | L/cmH$_2$O |

Ipotesi: flusso laminare lineare ($\Delta P = R Q$), compliance costanti,
gas incomprimibile (semplificazione importante!).

### D. Equazioni

$P_m=0$ (atmosfera). Per ciascun nodo interno con compliance $C_i$:
$C_i\dot P_i = Q_{in,i} - Q_{out,i}$.

In forma matriciale (dimostrazione completa nelle dispense del corso):
$\dot{\mathbf{P}} = A\,\mathbf{P} + B\,P_{pl}(t)$ con $\mathbf{P}=(P_l,P_b,P_A,P_{pl})$
e $P_{pl}(t)$ definito dal driver muscolare/respiratoria.

In versione *didattica* concentriamo tutto in un singolo modello del
secondo ordine RLC (R aerodinamica + I dell'aria + C polmonare),
sufficiente a catturare la dinamica volumetrica:

$$
P_{pl}(t) - P_{atm} = R\,\dot V + (V - V_0)/C + I_a\,\ddot V .
$$

Per la versione "5 compartimenti" del corso, vedi sotto.


In [ ]:
# Schema della meccanica respiratoria (5 compartimenti)
fig, ax = plt.subplots(figsize=(13, 5))
ax.set_xlim(-0.5, 13); ax.set_ylim(-1, 5); ax.set_aspect("equal"); ax.axis("off")

# Nodi sulla linea superiore
nodes = [("Bocca\n$P_m=0$", 0.5),
         ("Laringe\n$P_l$",  3.0),
         ("Trachea\n$P_t$",  5.5),
         ("Bronchi\n$P_b$",  8.0),
         ("Alveoli\n$P_A$", 10.5)]
y_top = 4.2
for (label, x) in nodes:
    ax.plot(x, y_top, "o", color="black", ms=8)
    ax.text(x, y_top+0.5, label, ha="center", fontsize=10)

# Resistenze (zigzag)
for (x0, x1, label) in [(0.5, 3.0, "$R_l$"),
                        (3.0, 5.5, "$R_t$"),
                        (5.5, 8.0, "$R_b$"),
                        (8.0, 10.5, "$R_A$")]:
    resistor(ax, (x0, y_top), (x1, y_top), label=label)

# Compliance verticali
for (x, label) in [(3.0, "$C_l$"), (5.5, "$C_t$"), (8.0, "$C_b$"), (10.5, "$C_A$")]:
    wire(ax, (x, y_top), (x, 3.0))
    capacitor(ax, (x, 3.0), (x, 1.6), label=label)
    wire(ax, (x, 1.6), (x, 0.5))
# Linea terra
wire(ax, (0.5, 0.5), (12, 0.5))
ground(ax, (6.5, 0.5))

# Driver pleurico (generatore di pressione esterno alle compliance)
ax.text(12.4, 2.3, "Pressione\npleurica\n$P_{pl}(t)$", color=COL["accent"], fontsize=10)
draw_arrow(ax, (12.0, 4.0), (10.7, 4.0), color=COL["accent"], mutation_scale=14)

ax.set_title("Meccanica respiratoria: rete a 5 nodi (bocca→alveoli) con compliance e resistenze",
             fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# === Meccanica respiratoria semplificata a 1 grado di liberta' ===
# Modello: P_pl - P_atm = R*Vdot + (V-V0)/C  (trascuriamo l'inerzia)
# V(t): volume polmonare; Pdriver: pressione muscolare-pleurica
R_resp = 2.0    # cmH2O*s/L
C_resp = 0.1    # L/cmH2O
V0     = 0.0    # volume di riferimento
freq   = 0.25   # Hz (15 atti/min)
P_amp  = 6.0    # cmH2O

def P_driver(t):
    # Sinusoide negativa: durante l'inspirazione P_pl si abbassa
    return -P_amp * (1 - np.cos(2*np.pi*freq*t)) / 2

def lung(t, V):
    Vdot = (P_driver(t) - (V[0] - V0)/C_resp) / R_resp
    return np.array([Vdot])

T = 12
sol = solve_ivp(lung, (0, T), [0.0], dense_output=True, max_step=0.05)
V = sol.y[0]
Q = np.gradient(V, sol.t)
Pdriver_arr = np.array([P_driver(t) for t in sol.t])

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
axes[0].plot(sol.t, Pdriver_arr, color=COL["accent"], lw=2)
axes[0].set_ylabel("$P_{pl}$ [cmH2O]")
axes[0].set_title("Pressione muscolo-pleurica (sinusoidale)")
axes[1].plot(sol.t, V, color=COL["main"], lw=2)
axes[1].set_ylabel("Volume polmonare [L]")
axes[1].set_title("Volume polmonare (risposta del sistema)")
axes[2].plot(sol.t, Q, color=COL["ok"], lw=2)
axes[2].set_ylabel("Flusso $\\dot V$ [L/s]")
axes[2].set_xlabel("tempo [s]")
axes[2].set_title("Flusso d'aria")
plt.tight_layout(); plt.show()


## 7.6 Ventilazione alveolare, spazio morto e scambio gassoso

### A. Spazio morto vs ventilazione alveolare

Volume corrente $V_T$ = volume di aria scambiata per atto respiratorio.
Solo una parte arriva agli alveoli e partecipa allo scambio: il resto resta
nello **spazio morto** $V_D$ (vie aeree di conduzione). Quindi

$$
V_A = V_T - V_D,
$$

e la **ventilazione alveolare** in L/min è $\dot V_A = f\,(V_T - V_D)$, con
$f$ frequenza respiratoria.

### B. Equazione alveolare per CO$_2$ (steady state)

Bilancio di massa: in ingresso l'aria ha $F_{ICO_2}\approx 0$; in uscita
ha $F_{ACO_2}$. La produzione metabolica di CO$_2$ è $\dot V_{CO_2}$.
All'equilibrio:

$$
\dot V_A\,F_{ACO_2} = \dot V_{CO_2}.
$$

In termini di pressione parziale ($P_{ACO_2} = F_{ACO_2}\,(P_B - P_{H_2O})$):

$$
\boxed{\;P_{ACO_2} \approx \frac{\dot V_{CO_2}}{\dot V_A}\cdot 0.863\;\text{(mmHg)}\;}
$$

L'analoga per O$_2$ (con consumo) dà l'iperbole ventilatoria. Visualizziamole.


In [ ]:
# Iperboli di CO2 e O2 vs ventilazione alveolare V_A (l/min)
VCO2 = 200.0       # ml/min, produzione metabolica
VO2  = 250.0       # ml/min, consumo
F_IO2 = 0.21
P_B   = 760.0
P_H2O = 47.0
P_IO2 = F_IO2 * (P_B - P_H2O)

VA = np.linspace(2, 15, 400)  # L/min
PACO2 = VCO2 / VA * 0.863
PAO2  = P_IO2 - VO2/VA * 0.863

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(VA, PACO2, color=COL["accent"], lw=2.5, label=r"$P_{ACO_2}$ (CO$_2$)")
ax.plot(VA, PAO2,  color=COL["main"],   lw=2.5, label=r"$P_{AO_2}$ (O$_2$)")
ax.axhline(40, color="gray", ls=":")
ax.axvline(5.25, color="gray", ls=":")
ax.text(5.35, 70, "set-point\nfisiologico\n$\\dot V_A\\approx 5.25$ L/min",
        color="gray", fontsize=10)
ax.set_xlabel(r"ventilazione alveolare $\dot V_A$ [L/min]")
ax.set_ylabel("pressione parziale alveolare [mmHg]")
ax.set_title("Iperboli ventilatorie: dipendenza di $P_A$ da $\\dot V_A$")
ax.legend(); plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — l'iperbole della CO$_2$ è ripida vicino a
basse ventilazioni (iperventilare anche poco abbassa molto $P_{ACO_2}$) e
*piatta* per ventilazioni alte; quella dell'O$_2$ è quasi piatta dopo la
saturazione.

### Patologie del sistema respiratorio nel linguaggio del modello

Il sistema respiratorio reale è una **rete asimmetrica** di vie aeree (modello
di Weibel a 23 generazioni). Patologie tipiche e i loro effetti sul modello
"lumped" $R$, $C$:

| Patologia | $R$ vie aeree | $C$ polmonare | Note cliniche |
|---|---|---|---|
| BPCO / enfisema | ↑↑ | ↑↑ (distruzione setti) | air trapping; iperinflazione; FEV$_1$↓↓ |
| Asma acuto | ↑↑↑ | normale | reversibile con broncodilatatori (β2-agonisti, ipratropio) |
| Fibrosi polmonare | normale | ↓↓↓ | polmoni "rigidi"; lavoro respiratorio alto; FEV$_1$/FVC normale |
| ARDS | ↑ | ↓↓ | parenchima eterogeneo, "baby lung"; PaO$_2$/FiO$_2$ ↓ |
| Pneumotorace | normale | ↓ effettiva | volume funzionale ridotto; rumori respiratori assenti |
| Edema polmonare | ↑ moderato | ↓↓ | rantoli, ortopnea, P-wedge ↑ |

### Costante di tempo respiratoria e auto-PEEP

$\tau_{resp} = R\,C$. In un polmone normale $\tau \approx 0.2$ s. Per
espirazione completa servono $\sim 3\tau \approx 0.6$ s. Se la frequenza
respiratoria è troppo alta (es. ventilazione meccanica con $T_E$ corto),
l'aria **non riesce a uscire del tutto** → "**auto-PEEP**" (positive end-expiratory pressure intrinseca) e iperinflazione progressiva. È il classico problema clinico nei pazienti BPCO ventilati.

> Stessa identica matematica del circuito RC: $\dot V = -V/\tau$ con
> espirazione interrotta prima del completamento → si accumula volume
> residuo.

### Spazio morto $V_D$ e dead space ratio $V_D/V_T$

Il rapporto $V_D/V_T$ è clinicamente cruciale. **Equazione di Bohr modificata**:

$$
\frac{V_D}{V_T} = \frac{P_{ACO_2} - P_{ECO_2}}{P_{ACO_2}},
$$

dove $P_{ECO_2}$ è la pressione di CO$_2$ nell'aria espirata mediata. Sale in:

- **embolia polmonare** (zone ventilate non perfuse → "spazio morto alveolare");
- **enfisema** (distruzione dei capillari alveolari);
- **ARDS** (aree non scambianti per shunt);
- **ipoperfusione** (shock cardiogeno con bassa portata).

### Capnografia clinica

In ICU si misura $P_{ETCO_2}$ (end-tidal CO$_2$) come surrogato di
$P_{ACO_2}$: si plotta CO$_2$ vs tempo o vs volume. La forma della curva
indica:

- plateau orizzontale = scambio efficace;
- pendenza positiva = malattia ostruttiva (eterogeneità del riempimento);
- $P_{ETCO_2}$ molto < $P_{aCO_2}$ → spazio morto aumentato (es. embolia).

## 7.7 Controllo chemocettoriale della ventilazione: Cheyne–Stokes

I chemocettori (carotidei e centrali) "leggono" $P_{ACO_2}$ e regolano la
ventilazione. Schematicamente, $\dot V_A = G\,(P_{ACO_2} - P_{th}) + V_0$
con un **ritardo** $\tau_d$ (tempo che il sangue impiega ad arrivare ai
chemocettori). Loop chiuso instabile se $G\,\tau_d$ supera una soglia:
**Cheyne–Stokes** (oscillazioni crescenti-decrescenti del respiro).


In [ ]:
# Modello didattico di controllo ventilatorio con ritardo (delay differential)
# Stato: P_A_CO2. Controllo: V_A = V0 + G * (P_A(t - tau_d) - P_th).
# Per semplicita' implementiamo con buffer circolare.

VCO2 = 200.0
G = 1.2          # L/min per mmHg
P_th = 38.0      # soglia (mmHg)
V0 = 1.0         # ventilazione basale (L/min)

class ControlSim:
    def __init__(self, tau_d, dt=0.05, T=120, P0=44.0):
        self.tau_d = tau_d; self.dt = dt
        self.N = int(T/dt) + 1
        self.t = np.linspace(0, T, self.N)
        self.PA = np.full(self.N, P0)
        self.VA = np.zeros(self.N)
    def run(self):
        n_delay = max(1, int(self.tau_d / self.dt))
        for k in range(self.N - 1):
            P_chemo = self.PA[max(0, k - n_delay)]
            VA = max(0.5, V0 + G * (P_chemo - P_th))
            self.VA[k] = VA
            # P_A_CO2 evolve secondo equazione alveolare semplificata
            P_inf = VCO2 / (VA*1000/60.0) * 0.863 * 1000  # converto ai mmHg
            # Dinamica primo ordine con tau breve
            tau_lung = 5.0
            self.PA[k+1] = self.PA[k] + self.dt / tau_lung * (P_inf - self.PA[k])
        return self.t, self.PA, self.VA

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
for tau_d, c in zip([2.0, 8.0, 15.0],
                     [COL["main"], COL["warn"], COL["accent"]]):
    sim = ControlSim(tau_d=tau_d)
    t, PA, VA = sim.run()
    axes[0].plot(t, PA, color=c, lw=1.6, label=f"$\\tau_d={tau_d}$ s")
    axes[1].plot(t, VA, color=c, lw=1.6)
axes[0].axhline(P_th, color="gray", ls=":")
axes[0].set_ylabel("$P_{ACO_2}$ [mmHg]"); axes[0].legend()
axes[0].set_title("Controllo ventilatorio chemocettoriale al variare del ritardo")
axes[1].set_ylabel("$\\dot V_A$ [L/min]"); axes[1].set_xlabel("tempo [s]")
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — con ritardo breve ($\tau_d=2$ s) il loop
è stabile e il sistema converge a un equilibrio. Aumentando $\tau_d$
compaiono prima oscillazioni smorzate, poi (oltre soglia) **oscillazioni
autoperpetuanti** del tipo Cheyne–Stokes: ventilazione che cresce e cala
ciclicamente, con $P_{ACO_2}$ che le segue in controfase.

**Perché è importante** — nei pazienti con insufficienza cardiaca avanzata
o lesioni del SNC, il tempo di transito sangue→chemocettori si allunga
oppure il guadagno del feedback aumenta, ed entrambi gli effetti possono
oltrepassare la soglia di Hopf: nasce il respiro di Cheyne–Stokes.


## 7.8 Elettrofisiologia cellulare: equazione di Nernst

### A. Contesto

Ogni canale ionico permeabile alla sola specie $X$ raggiunge un equilibrio
elettrochimico quando il lavoro elettrico (campo che spinge gli ioni)
bilancia il gradiente di concentrazione (diffusione). Quel potenziale di
equilibrio è il **potenziale di Nernst** $E_X$:

$$
\boxed{\;E_X = \frac{RT}{zF}\ln\frac{[X]_o}{[X]_i}\;}
$$

| Simbolo | Significato | Valore tipico |
|---|---|---|
| $R$ | costante dei gas | 8.314 J/(mol·K) |
| $T$ | temperatura assoluta | 310 K (37°C) |
| $z$ | carica dello ione | ±1 (Na+, K+, Cl-), ±2 (Ca++) |
| $F$ | Faraday | 96485 C/mol |
| $[X]_o, [X]_i$ | conc. extra/intra | mM |

### D. Derivazione (versione corta)

All'equilibrio elettrochimico, l'**energia libera** per mole di ione X è
nulla:

$$
\Delta G = zF\Delta V + RT \ln\frac{[X]_o}{[X]_i} = 0 \implies
\Delta V = -\frac{RT}{zF}\ln\frac{[X]_o}{[X]_i} = \frac{RT}{zF}\ln\frac{[X]_o}{[X]_i}
$$

con la convenzione standard $\Delta V = V_{in} - V_{out}$ ribaltata nei
testi (qui usiamo $V$ del compartimento "in" rispetto a quello "out").

A 37°C la formula in pratica si scrive

$$
E_X[\text{mV}] = \frac{61.5}{z}\,\log_{10}\frac{[X]_o}{[X]_i}.
$$


In [ ]:
# Potenziali di Nernst per Na, K, Cl a 37 C e effetto di [X]_o
R, T, F = 8.314, 310.0, 96485.0
def nernst(z, Co, Ci): return (R*T)/(z*F) * np.log(Co/Ci) * 1e3   # mV

# Valori "tipici" delle concentrazioni neuronali
ions = {
    "Na+":  (+1, 145.0, 12.0),
    "K+":   (+1,   4.0, 155.0),
    "Cl-":  (-1, 120.0,   4.0),
    "Ca2+": (+2,   2.0, 1e-4),
}
print("Potenziali di Nernst a 37 C:")
for nm, (z, Co, Ci) in ions.items():
    print(f"  {nm:5s} z={z:+d}  [X]_o={Co}  [X]_i={Ci:.1e}  E = {nernst(z, Co, Ci):+7.2f} mV")

# Effetto del K extra (iperkaliemia)
K_o_range = np.linspace(2, 12, 200)
E_K = (R*T)/(1*F) * np.log(K_o_range/155.0) * 1e3
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(K_o_range, E_K, color=COL["main"], lw=2)
ax.axvline(4.0, color="gray", ls=":")
ax.text(4.1, -100, "valore normale\n[K+]o = 4 mM", color="gray")
ax.set_xlabel("[K+]_o [mM]"); ax.set_ylabel("$E_K$ [mV]")
ax.set_title("Potenziale di Nernst per K+ vs concentrazione extracellulare")
plt.tight_layout(); plt.show()


**Implicazione clinica** — un'iperkaliemia (es. [K+]o da 4 a 8 mM)
*depolarizza* $E_K$ di $\sim$18 mV. Poiché il potenziale di riposo
neuronale e cardiaco è dominato da $E_K$, questo cambiamento può portare
ad aritmie pericolose.

## 7.9 Voltage clamp

In *voltage clamp* lo sperimentatore impone $V_m$ a un valore desiderato
e misura la corrente totale necessaria a mantenerlo. Implementazione
nel modello: si tiene $V_m=V_{cmd}$ costante, e la corrente totale di
membrana iniettata dall'amplificatore è esattamente $-I_{ion}(V_m,t)$
(con il segno definito dalla convenzione). In Python si simula
"congelando" la dinamica di $V$ e calcolando $I_{ion}$.


### Derivazione di Nernst dalla termodinamica (potenziale elettrochimico)

Per uno ione $X$ di carica $z$ in soluzione, definiamo il **potenziale
elettrochimico**

$$
\bar\mu_X = \underbrace{\mu_X^0 + RT\ln a_X}_{\text{chimico}} + \underbrace{zF V}_{\text{elettrico}},
$$

dove $a_X\approx [X]$ è l'attività (≈ concentrazione), $V$ il potenziale
elettrico, $R$ la costante dei gas, $T$ temperatura assoluta, $F$ Faraday.

**Equilibrio termodinamico** tra interno ($i$) e esterno ($o$) di una
membrana permeabile **solo** a $X$: $\bar\mu_X^{(i)}=\bar\mu_X^{(o)}$,
cioè

$$
\mu_X^0 + RT\ln[X]_i + zF V_i = \mu_X^0 + RT\ln[X]_o + zF V_o.
$$

Definiamo $V_m \equiv V_i - V_o$. Risolvendo:

$$
zF V_m = RT\big(\ln[X]_o - \ln[X]_i\big)
\;\Longleftrightarrow\;
\boxed{\;V_m = E_X = \frac{RT}{zF}\,\ln\frac{[X]_o}{[X]_i}\;}
$$

**Forma pratica a 37 °C** (310 K):

$$
E_X[\text{mV}] = \frac{61.5}{z}\,\log_{10}\frac{[X]_o}{[X]_i}.
$$

### Bilancio di flussi e equazione di Goldman–Hodgkin–Katz (GHK)

Se la membrana è permeabile a **più** ioni, all'equilibrio
elettrochimico **non** lo è a ciascuno separatamente, ma il
**bilancio di carica** (la corrente totale è nulla) impone:

$$
\sum_i I_i = 0 \;\Longrightarrow\;
V_m = \frac{RT}{F}\,\ln\!\left(\frac{P_{Na}[Na]_o + P_K[K]_o + P_{Cl}[Cl]_i}{P_{Na}[Na]_i + P_K[K]_i + P_{Cl}[Cl]_o}\right).
$$

(Lo ione Cl ha carica $-1$ e quindi le sue concentrazioni "extra" e "intra"
si invertono nella formula.) È l'**equazione di GHK**: dà il potenziale
di riposo quando più canali sono attivi.

> Per concentrazioni neuronali tipiche, $P_K\gg P_{Na}$ a riposo, quindi
> $V_m^{\text{rest}}\approx E_K\approx -75\,\text{mV}$.


In [ ]:
# GHK: come varia V_rest al variare dei rapporti di permeabilità
R_, T_, F_ = 8.314, 310.0, 96485.0
Na_o, Na_i = 145.0, 12.0
K_o,  K_i  =   4.0, 155.0
Cl_o, Cl_i = 120.0,   4.0

P_K = 1.0   # normalizzazione

ratios = np.logspace(-3, 0, 100)   # P_Na/P_K da 0.001 a 1
Vrest = []
for r in ratios:
    P_Na = r * P_K
    num = P_Na*Na_o + P_K*K_o + 0.45*Cl_i
    den = P_Na*Na_i + P_K*K_i + 0.45*Cl_o
    Vrest.append(R_*T_/F_ * np.log(num/den) * 1e3)

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(ratios, Vrest, color="#1f4e79", lw=2)
ax.axhline(-75, color="gray", ls=":", label="≈ E_K (-75 mV)")
ax.axhline(+60, color="gray", ls="--", label="≈ E_Na (+60 mV)")
ax.set_xlabel(r"rapporto $P_{Na}/P_K$")
ax.set_ylabel(r"$V_{rest}$ da GHK [mV]")
ax.set_title("Potenziale di riposo (GHK) al variare del rapporto P_Na/P_K")
ax.legend()
plt.tight_layout(); plt.show()


### Cinetica di gating come probabilità di stato — derivazione

Hodgkin e Huxley scrissero $\dot m = \alpha_m(V)(1-m) - \beta_m(V) m$ in
modo fenomenologico. La motivazione *fisica* sta nel modello di **canale
ionico a due stati**.

Sia un singolo canale con due conformazioni: **chiusa** $C$ e **aperta** $O$,
con costanti di transizione $\alpha(V)$ e $\beta(V)$:

$$
C \overset{\alpha(V)}{\underset{\beta(V)}{\rightleftharpoons}} O.
$$

Sia $p(t)$ la *probabilità* che il canale sia aperto al tempo $t$ (e
$1-p$ quella di essere chiuso). L'equazione di evoluzione (master
equation per uno stato):

$$
\dot p = \underbrace{\alpha(V)(1-p)}_{\text{C}\to\text{O}}
       - \underbrace{\beta(V) p}_{\text{O}\to\text{C}}.
$$

Forma equivalente "tau, p_infty":

$$
\dot p = \frac{p_\infty(V) - p}{\tau(V)},
\quad p_\infty=\frac{\alpha}{\alpha+\beta},\quad \tau=\frac{1}{\alpha+\beta}.
$$

**Connessione con la macrocorrente**: se la membrana ha $N$ canali
indipendenti e ognuno è aperto con probabilità $p$, la **conduttanza
media** è

$$
g(t,V) = N\cdot g_{\text{single}} \cdot p(t,V) = \bar g\,p(t,V),
$$

con $\bar g = N g_{\text{single}}$ conduttanza massima. Quindi nel modello
HH la "variabile di gating" $m$ è esattamente la probabilità di apertura
di un sotto-cancello.

### Perché $m^3 h$ e $n^4$?

I canali del Na hanno **3 cancelli di attivazione** ($m$) e **1 cancello
di inattivazione** ($h$) *indipendenti*. La probabilità che siano TUTTI
nello stato "passa corrente" è il prodotto $m\cdot m\cdot m\cdot h = m^3 h$.

I canali del K hanno **4 cancelli di attivazione** $n$ (struttura
tetramerica del canale Kv). La probabilità che siano tutti aperti è $n^4$.

Quindi la corrente macroscopica è:

$$
I_{Na} = \bar g_{Na}\,m^3 h\,(V - E_{Na}),\qquad
I_K = \bar g_K\,n^4\,(V - E_K).
$$

Il fattore $(V - E_i)$ è la **driving force**: zero quando $V = E_i$ (a
riposo elettrochimico, nessuna corrente netta anche se i canali sono
aperti).


## 7.10 Modello di Hodgkin–Huxley

### A. Contesto

Hodgkin e Huxley (1952, Premio Nobel) modellarono l'assone gigante del
calamaro come un circuito equivalente con:

- **capacitore** $C_m$;
- **tre branche ioniche in parallelo**: Na, K, leak;
- ciascuna conduttanza è $g_X = \bar g_X \cdot p_X(t,V)$ con $p_X$ probabilità
  di canale aperto, descritta da **variabili di gating** $m, h, n$.

### Equazioni di gating

Ogni canale ha sotto-cancelli che si aprono/chiudono con cinetica di primo
ordine:

$$
\dot x = \alpha_x(V)(1-x) - \beta_x(V)\,x,\qquad x\in\{m,h,n\}.
$$

Si possono riscrivere nella forma "$\tau$, $x_\infty$":

$$
\dot x = \frac{x_\infty(V) - x}{\tau_x(V)},\quad
x_\infty=\frac{\alpha_x}{\alpha_x+\beta_x},\quad \tau_x=\frac{1}{\alpha_x+\beta_x}.
$$

### Equazione di membrana

$$
C_m\dot V = -\bar g_{Na}\,m^3 h\,(V - E_{Na}) - \bar g_K\,n^4\,(V - E_K)
            - g_L\,(V - E_L) + I_{ext}.
$$

Il fattore $m^3$ è la probabilità che 3 cancelli "$m$" siano aperti; $h$ è
una unica probabilità di non-inattivazione; $n^4$ è la probabilità di 4
cancelli $n$ aperti.

### Circuito equivalente

Disegniamolo esplicitamente con conduttanze e batterie ioniche.


In [ ]:
# Schema del circuito equivalente di Hodgkin-Huxley
fig, ax = plt.subplots(figsize=(11.5, 5.5))
ax.set_xlim(-0.5, 9); ax.set_ylim(-0.5, 4.8); ax.set_aspect("equal"); ax.axis("off")

# Linea superiore = intracellulare (V_m)
y_top = 4.0
wire(ax, (0.5, y_top), (8.5, y_top))
# Linea inferiore = extracellulare = ground
y_bot = 0.5
wire(ax, (0.5, y_bot), (8.5, y_bot))
ground(ax, (4.5, y_bot))
ax.text(8.7, y_top, "interno ($V_m$)", color=COL["main"], fontsize=10)
ax.text(8.7, y_bot, "esterno (0)", color=COL["grey"], fontsize=10)

# Corrente esterna a sinistra
wire(ax, (0.7, y_top), (0.7, 2.5))
isource(ax, (0.7, 2.5), (0.7, 1.5), label=r"$I_{ext}$")
wire(ax, (0.7, 1.5), (0.7, y_bot))

# Capacitore al centro-sinistra
wire(ax, (2.0, y_top), (2.0, 2.6))
capacitor(ax, (2.0, 2.6), (2.0, 1.5), label=r"$C_m$")
wire(ax, (2.0, 1.5), (2.0, y_bot))

# Branca Na
wire(ax, (3.5, y_top), (3.5, 3.0))
resistor(ax, (3.5, 3.0), (3.5, 2.0), label=r"$g_{Na}=\bar g_{Na}m^3h$")
wire(ax, (3.5, 2.0), (3.5, 1.6))
vsource(ax, (3.5, 1.6), (3.5, 0.9), label=r"$E_{Na}$", plus_up=True)
wire(ax, (3.5, 0.9), (3.5, y_bot))

# Branca K
wire(ax, (5.5, y_top), (5.5, 3.0))
resistor(ax, (5.5, 3.0), (5.5, 2.0), label=r"$g_K=\bar g_K n^4$")
wire(ax, (5.5, 2.0), (5.5, 1.6))
vsource(ax, (5.5, 1.6), (5.5, 0.9), label=r"$E_K$", plus_up=False)
wire(ax, (5.5, 0.9), (5.5, y_bot))

# Branca leak
wire(ax, (7.5, y_top), (7.5, 3.0))
resistor(ax, (7.5, 3.0), (7.5, 2.0), label=r"$g_L$")
wire(ax, (7.5, 2.0), (7.5, 1.6))
vsource(ax, (7.5, 1.6), (7.5, 0.9), label=r"$E_L$", plus_up=False)
wire(ax, (7.5, 0.9), (7.5, y_bot))

# Equazione esplicita
ax.text(4.5, 4.6,
        r"$C_m\dot V = I_{ext} - g_{Na}(V-E_{Na}) - g_K(V-E_K) - g_L(V-E_L)$",
        ha="center", fontsize=12, color=COL["main"])

ax.set_title("Circuito equivalente di Hodgkin-Huxley", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# === Hodgkin-Huxley con parametri canonici (squid axon) ===
# Unita': V in mV, t in ms, C_m in uF/cm^2, g in mS/cm^2, I in uA/cm^2.
C_m = 1.0
g_Na, g_K, g_L = 120.0, 36.0, 0.3
E_Na, E_K, E_L = 50.0, -77.0, -54.387

def alpha_n(V): return 0.01*(V + 55) / (1 - np.exp(-(V + 55)/10))
def beta_n(V):  return 0.125 * np.exp(-(V + 65)/80)
def alpha_m(V): return 0.1*(V + 40) / (1 - np.exp(-(V + 40)/10))
def beta_m(V):  return 4.0 * np.exp(-(V + 65)/18)
def alpha_h(V): return 0.07 * np.exp(-(V + 65)/20)
def beta_h(V):  return 1.0/(1 + np.exp(-(V + 35)/10))

def hh_rhs(t, y, I_ext_fn):
    V, n, m, h = y
    I_Na = g_Na * m**3 * h * (V - E_Na)
    I_K  = g_K  * n**4     * (V - E_K)
    I_L  = g_L              * (V - E_L)
    dV = (I_ext_fn(t) - I_Na - I_K - I_L) / C_m
    dn = alpha_n(V)*(1-n) - beta_n(V)*n
    dm = alpha_m(V)*(1-m) - beta_m(V)*m
    dh = alpha_h(V)*(1-h) - beta_h(V)*h
    return np.array([dV, dn, dm, dh])

# Stimoli: pulse di 10 uA/cm^2 per 0.5 ms a t=5 ms, e di nuovo a t=20 ms
def I_ext(t):
    if 5 <= t <= 5.5:  return 10.0
    if 20 <= t <= 20.5: return 10.0
    return 0.0

# Condizioni iniziali al riposo
V0 = -65.0
n0 = alpha_n(V0)/(alpha_n(V0)+beta_n(V0))
m0 = alpha_m(V0)/(alpha_m(V0)+beta_m(V0))
h0 = alpha_h(V0)/(alpha_h(V0)+beta_h(V0))

T = 40
sol = solve_ivp(lambda t, y: hh_rhs(t, y, I_ext), (0, T), [V0, n0, m0, h0],
                dense_output=True, max_step=0.01, rtol=1e-8, atol=1e-10)

V = sol.y[0]; n = sol.y[1]; m = sol.y[2]; h = sol.y[3]
I_Na = g_Na * m**3 * h * (V - E_Na)
I_K  = g_K  * n**4     * (V - E_K)

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
axes[0].plot(sol.t, V, color=COL["main"], lw=1.6)
axes[0].set_ylabel("$V_m$ [mV]"); axes[0].set_title("Potenziale d'azione di Hodgkin-Huxley")
axes[0].axhline(0, color="gray", lw=0.6)

axes[1].plot(sol.t, m, color=COL["accent"], lw=1.4, label="m (Na act)")
axes[1].plot(sol.t, h, color=COL["ok"],     lw=1.4, label="h (Na inact)")
axes[1].plot(sol.t, n, color=COL["main"],   lw=1.4, label="n (K act)")
axes[1].legend(); axes[1].set_ylabel("variabili di gating")

axes[2].plot(sol.t, I_Na, color=COL["accent"], lw=1.4, label="$I_{Na}$")
axes[2].plot(sol.t, I_K,  color=COL["main"],   lw=1.4, label="$I_K$")
axes[2].axhline(0, color="gray", lw=0.6)
axes[2].set_ylabel("corrente [µA/cm²]"); axes[2].set_xlabel("tempo [ms]")
axes[2].legend(); axes[2].set_title("Correnti ioniche")
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — il primo stimolo (a $t=5$ ms) innesca un
**potenziale d'azione**: $V_m$ schizza fino a +30 mV per poi tornare a
$-75$ mV, oltrepassando temporaneamente $E_K$ (iperpolarizzazione).
Le variabili di gating mostrano il meccanismo: $m$ schizza in alto (apertura
rapida dei canali Na), poi $h$ scende (inattivazione) e $n$ sale lentamente
(apertura K) per ripolarizzare. Le correnti ioniche evidenziano il
sequence Na entrante (negativa) seguita da K uscente (positiva).

**Sul secondo stimolo** — a $t=20$ ms il sistema risponde di nuovo, ma se
si stimola troppo presto (periodo refrattario assoluto/relativo) il
neurone non spara. È esattamente il meccanismo che limita la massima
frequenza di sparo neuronale.

### Voltage clamp didattico: variabili di gating a $V$ costante


In [ ]:
# Voltage clamp didattico: blocchiamo V a vari livelli e vediamo le correnti
V_clamps = [-50, -30, 0, 30]
T = 10  # ms
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Dinamica dei gating con V fisso: dy = alpha (1-y) - beta y, scalare
def gate_dynamics(V, m0, h0, n0, T=10):
    def rhs(t, y):
        m, h, n = y
        return [alpha_m(V)*(1-m) - beta_m(V)*m,
                alpha_h(V)*(1-h) - beta_h(V)*h,
                alpha_n(V)*(1-n) - beta_n(V)*n]
    sol = solve_ivp(rhs, (0, T), [m0, h0, n0], dense_output=True, max_step=0.01)
    return sol

V_rest = -65.0
n0 = alpha_n(V_rest)/(alpha_n(V_rest)+beta_n(V_rest))
m0 = alpha_m(V_rest)/(alpha_m(V_rest)+beta_m(V_rest))
h0 = alpha_h(V_rest)/(alpha_h(V_rest)+beta_h(V_rest))

ax1, ax2 = axes
for V_c, c in zip(V_clamps, [COL["main"], COL["ok"], COL["warn"], COL["accent"]]):
    sol = gate_dynamics(V_c, m0, h0, n0, T=T)
    m, h, n = sol.y
    I_Na = g_Na * m**3 * h * (V_c - E_Na)
    I_K  = g_K  * n**4     * (V_c - E_K)
    ax1.plot(sol.t, I_Na, color=c, lw=1.6, label=f"V={V_c} mV")
    ax2.plot(sol.t, I_K,  color=c, lw=1.6, label=f"V={V_c} mV")

ax1.set_xlabel("tempo [ms]"); ax1.set_ylabel("$I_{Na}$ [µA/cm²]")
ax1.set_title("Voltage clamp: corrente $I_{Na}$"); ax1.legend()
ax2.set_xlabel("tempo [ms]"); ax2.set_ylabel("$I_K$ [µA/cm²]")
ax2.set_title("Voltage clamp: corrente $I_K$"); ax2.legend()
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — a $V$ depolarizzati la corrente Na è
**transitoria** (apre rapidamente, poi si inattiva) mentre la corrente K è
**persistente** (apre più lentamente e resta aperta finché V è alto). È
esattamente questa **separazione cinetica** che permette al potenziale
d'azione di esistere: il Na entra prima di disattivarsi, il K spegne dopo.


### Curve di gating a regime $x_\infty(V)$ e costanti di tempo $\tau_x(V)$


In [ ]:
V_range = np.linspace(-100, 50, 300)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for x_name, ax, ay, ay_inf in [("m", alpha_m, beta_m, None),
                                ("h", alpha_h, beta_h, None),
                                ("n", alpha_n, beta_n, None)]:
    a = np.array([ax(V) for V in V_range])
    b = np.array([ay(V) for V in V_range])
    inf = a / (a + b)
    tau = 1.0 / (a + b)
    axes[0].plot(V_range, inf, lw=2, label=f"${x_name}_\\infty(V)$")
    axes[1].plot(V_range, tau, lw=2, label=f"$\\tau_{x_name}(V)$")
axes[0].set_xlabel("V [mV]"); axes[0].set_ylabel("steady-state"); axes[0].set_title("Aperture stazionarie $x_\\infty(V)$")
axes[0].legend()
axes[1].set_xlabel("V [mV]"); axes[1].set_ylabel(r"$\tau$ [ms]"); axes[1].set_title("Costanti di tempo $\\tau_x(V)$")
axes[1].legend()
plt.tight_layout(); plt.show()


**Lettura delle curve** — $m_\infty$ e $n_\infty$ sono curve sigmoidali
crescenti in $V$ (più depolarizzo, più si aprono Na e K); $h_\infty$ è
decrescente (depolarizzazione → inattivazione Na). $\tau_m$ è molto più
piccolo di $\tau_n$ e $\tau_h$: i canali Na *attivano* prima di tutto, K
e l'inattivazione Na vengono dopo. Questa separazione cinetica è ciò che
permette al sistema di "ciclare".


### Dalla biofisica HH alla farmacologia clinica

Il modello HH originale descrive l'assone gigante del calamaro a **6 °C**.
Per neuroni di mammifero a 37 °C tutte le costanti di tempo vanno
**ridotte** di un fattore $Q_{10}\approx 3$ per ogni 10 °C → cinetiche ≈ $3^{3.1}\approx 30$ volte più veloci.

### Modulazione farmacologica (gli "ingredienti" della clinica)

| Farmaco / tossina | Bersaglio | Effetto sul modello | Uso clinico |
|---|---|---|---|
| **Tetrodotossina (TTX)** | $g_{Na}$ | blocco selettivo → no spike | tossina (pesce palla) |
| **Tetraetilammonio (TEA)** | $g_K$ | blocca K → spike "allargati" | research tool |
| **Lidocaina** | $g_{Na}$ inattivati | use-dependent block | anestetico locale, antiaritmico classe Ib |
| **Carbamazepina, fenitoina** | $g_{Na}$ inattivati | stabilizzano inattivazione | antiepilettici (state-dependent) |
| **Tossina dello scorpione (α)** | inattivazione $h$ | rallenta $h$ → spike prolungati | tossicologica |
| **Riluzolo** | $g_{Na}$ persistente | riduce eccitotossicità | SLA (allunga la vita) |
| **Amiodarone** | multibranca (K, Ca, Na) | prolunga refrattario | antiaritmico classe III |
| **Lamotrigina** | $g_{Na}$ inattivati | stabilizza | antiepilettico, stabilizzatore dell'umore |

### Patofisiologia neurologica e cardiaca nel linguaggio di HH

- **Epilessia focale**: spesso ↓ $g_K$ (loss-of-function di canali Kv) → soglia di sparo abbassata, sparo persistente.
- **Paralisi periodica iperkaliemica**: mutazioni di $g_{Na}$ con inattivazione difettosa → muscoli incapaci di ripolarizzare → debolezza durante episodi di K↑.
- **Sindrome del QT lungo** (cardiologia): anomalie dei canali del K cardiaco (HERG, KvLQT1) → ripolarizzazione lenta → fenestre di aritmia, torsione di punta.
- **Sclerosi multipla**: demielinizzazione → propagazione saltatoria compromessa → blocco di conduzione (gli "spike" non superano la zona demielinizzata).
- **Brugada syndrome**: $g_{Na}$ cardiaco con loss-of-function → epicardio early repolarization → reentry → morte improvvisa.

> La **lezione di HH** è che la *stessa equazione differenziale* in due
> popolazioni di pazienti diversi può comportarsi diversamente perché *i
> parametri delle conduttanze* cambiano. Un farmaco moderno bersaglia un
> singolo coefficiente in una singola branca.

### Classi di eccitabilità (Hodgkin, 1948)

Hodgkin classificò le cellule eccitabili in:

- **classe I**: il rate di sparo cresce **con continuità** da 0 sopra soglia → biforcazione **saddle-node on invariant circle** (SNIC). Tipica di motoneuroni a sparo lento.
- **classe II**: il sistema passa **improvvisamente** da silenzio a sparo a una frequenza minima > 0 → biforcazione di **Hopf subcritica**. Il classico HH è **classe II**.

In clinica certe forme di disturbi del ritmo somigliano alla classe II (oscillazioni con periodo ben definito, "tutto o niente") vs classe I (modulazione fine della frequenza, gain del sistema variabile).

### Come si misurano in laboratorio le grandezze HH

- $V_m$ → microelettrodi intracellulari o patch-clamp (gold standard);
- $I_{Na}, I_K$ separati → voltage clamp + farmaci selettivi (TTX per isolare $I_K$, TEA per isolare $I_{Na}$);
- $m, h, n$ → non si misurano direttamente: si **inferiscono** dalla cinetica della corrente durante voltage clamp con protocolli a step.

### Modelli successivi (oltre HH)

- **Morris–Lecar (1981)**: HH ridotto a 2 variabili → studio della biforcazione.
- **FitzHugh–Nagumo**: ulteriore semplificazione (Van der Pol relaxation).
- **Hindmarsh–Rose**: 3 variabili, riproduce bursting.
- **Modelli cardiaci** (Beeler–Reuter, Luo–Rudy, Ten Tusscher): aggiungono Ca$^{2+}$ intracellulare e SR.


# Parte 8 — Metodi numerici avanzati

Abbiamo già usato Eulero esplicito, Eulero implicito e RK4 nella Parte 1.
Qui li approfondiamo, in particolare per problemi *stiff* (come HH e
modelli farmacocinetici lenti+veloci) e per il **root finding** (necessario
agli impliciti e ai punti di equilibrio dei modelli non lineari).

## 8.1 Confronto definitivo: esplicito vs implicito su HH


### Derivazione di Runge–Kutta del 4° ordine — costruzione formale

Vogliamo approssimare $\mathbf{x}(t+\Delta t)$ usando solo valutazioni di
$\mathbf{f}$ dentro l'intervallo $[t, t+\Delta t]$. Sviluppando in Taylor:

$$
\mathbf{x}(t+\Delta t) = \mathbf{x}(t) + \Delta t\,\dot{\mathbf{x}}
+ \frac{\Delta t^2}{2}\ddot{\mathbf{x}}
+ \frac{\Delta t^3}{6}\dddot{\mathbf{x}}
+ \frac{\Delta t^4}{24}\overset{....}{\mathbf{x}}
+ O(\Delta t^5).
$$

L'idea di Runge è **mescolare** valutazioni di $\mathbf{f}$ a punti
intermedi così che, sviluppando ogni $\mathbf{k}_i$ in Taylor e
sostituendo, **i termini fino a $O(\Delta t^4)$ coincidano** con la serie
esatta. Si trovano i coefficienti $b_i, c_i, a_{ij}$ del **tableau di
Butcher**.

Per RK4 classico:

| $c$ | $A$ | |
|---|---|---|
| 0 | | |
| 1/2 | 1/2 | |
| 1/2 | 0   | 1/2 |
| 1 | 0   | 0   | 1 |
| $b$ | 1/6 | 2/6 | 2/6 | 1/6 |

Cioè:

$$
\begin{aligned}
\mathbf{k}_1&=\mathbf{f}(t, \mathbf{x}),\\
\mathbf{k}_2&=\mathbf{f}(t+\tfrac{\Delta t}{2}, \mathbf{x}+\tfrac{\Delta t}{2}\mathbf{k}_1),\\
\mathbf{k}_3&=\mathbf{f}(t+\tfrac{\Delta t}{2}, \mathbf{x}+\tfrac{\Delta t}{2}\mathbf{k}_2),\\
\mathbf{k}_4&=\mathbf{f}(t+\Delta t, \mathbf{x}+\Delta t\,\mathbf{k}_3),\\
\mathbf{x}_{n+1}&=\mathbf{x}_n + \tfrac{\Delta t}{6}(\mathbf{k}_1+2\mathbf{k}_2+2\mathbf{k}_3+\mathbf{k}_4).
\end{aligned}
$$

**Errore locale**: $O(\Delta t^5)$. **Errore globale**: $O(\Delta t^4)$.
Vince Eulero (globale $O(\Delta t)$) per accuratezza, con costo solo 4
valutazioni di $\mathbf{f}$ per passo.

### Regione di stabilità lineare di Eulero esplicito vs implicito

Applichiamo i due metodi all'equazione test scalare $\dot x = \lambda x$,
$\lambda\in\mathbb{C}$.

**Eulero esplicito**: $x_{n+1} = (1 + \lambda \Delta t)\,x_n$.
Stabile (i.e. $|x_n|$ non esplode) se $|1 + \lambda \Delta t| \le 1$, cioè
dentro al **cerchio di raggio 1 centrato in $-1$** nel piano $\lambda\Delta t$.
È un dischetto piccolo: per $\lambda$ molto negativo il passo $\Delta t$
deve essere piccolo.

**Eulero implicito**: $x_{n+1} = x_n/(1 - \lambda\Delta t)$.
Stabile se $|1 - \lambda\Delta t| \ge 1$, cioè *fuori* dal cerchio di raggio
1 centrato in $+1$. Questa regione contiene **tutto il semipiano sinistro
$\Re(\lambda\Delta t)<0$**. È **A-stabile**: ogni problema
asintoticamente stabile è simulato senza vincoli sul passo.


In [ ]:
# Visualizzazione delle regioni di stabilità
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

x = np.linspace(-3, 3, 600)
y = np.linspace(-3, 3, 600)
X, Y = np.meshgrid(x, y)
Z = X + 1j*Y

# Eulero esplicito: stabile sse |1+Z| <= 1
ax = axes[0]
stab_exp = np.abs(1 + Z) <= 1
ax.contourf(X, Y, stab_exp.astype(int), levels=[0.5, 1.5], colors=["#aed6f1"])
ax.contour(X, Y, stab_exp.astype(int), levels=[0.5], colors=["#1f4e79"])
ax.axhline(0, color="gray", lw=0.6); ax.axvline(0, color="gray", lw=0.6)
ax.set_xlabel(r"Re$(\lambda \Delta t)$"); ax.set_ylabel(r"Im$(\lambda \Delta t)$")
ax.set_title("Regione di stabilità: Eulero esplicito\n(disco centrato in -1, raggio 1)")
ax.set_xlim(-3,3); ax.set_ylim(-3,3); ax.set_aspect("equal")

# Eulero implicito: stabile sse |1-Z| >= 1
ax = axes[1]
stab_imp = np.abs(1 - Z) >= 1
ax.contourf(X, Y, stab_imp.astype(int), levels=[0.5, 1.5], colors=["#a9dfbf"])
ax.contour(X, Y, stab_imp.astype(int), levels=[0.5], colors=["#1e8449"])
ax.axhline(0, color="gray", lw=0.6); ax.axvline(0, color="gray", lw=0.6)
ax.set_xlabel(r"Re$(\lambda \Delta t)$"); ax.set_ylabel(r"Im$(\lambda \Delta t)$")
ax.set_title("Regione di stabilità: Eulero implicito\n(tutto il piano TRANNE disco in +1)")
ax.set_xlim(-3,3); ax.set_ylim(-3,3); ax.set_aspect("equal")

plt.tight_layout(); plt.show()
print("L'implicito copre tutto il semipiano sinistro -> A-stabile.")


In [ ]:
# Esempio: integrare HH per breve tempo con RK45 (esplicito adattivo) vs BDF (implicito)
import time

def I_ext_test(t):
    return 7.0 if 5 <= t <= 50 else 0.0

V0 = -65.0
n0 = alpha_n(V0)/(alpha_n(V0)+beta_n(V0))
m0 = alpha_m(V0)/(alpha_m(V0)+beta_m(V0))
h0 = alpha_h(V0)/(alpha_h(V0)+beta_h(V0))

T = 50
methods = ["RK45", "Radau", "BDF"]
print(f"{'metodo':<8} {'wall (s)':<10} {'#step':<7} {'max V (mV)':<10}")
fig, ax = plt.subplots(figsize=(10, 4.6))
for m_name, c in zip(methods, [COL["main"], COL["ok"], COL["accent"]]):
    t0 = time.time()
    sol = solve_ivp(lambda t, y: hh_rhs(t, y, I_ext_test),
                    (0, T), [V0, n0, m0, h0],
                    method=m_name, rtol=1e-6, atol=1e-9)
    wt = time.time() - t0
    print(f"{m_name:<8} {wt:<10.3f} {len(sol.t):<7} {sol.y[0].max():<10.2f}")
    ax.plot(sol.t, sol.y[0], lw=1.4, color=c, label=m_name)
ax.set_xlabel("t [ms]"); ax.set_ylabel("V [mV]")
ax.set_title("HH integrato con metodi diversi (stessa tolleranza)")
ax.legend(); plt.tight_layout(); plt.show()


## 8.2 Root finding per equilibri di modelli non lineari

Per un modello non lineare $\dot{\mathbf{x}}=\mathbf{f}(\mathbf{x})$,
l'equilibrio si trova risolvendo $\mathbf{f}(\mathbf{x}^*)=\mathbf{0}$ con
metodi numerici (Newton, hybr di scipy). La derivata $J=\partial f/\partial x$
è la matrice Jacobiana: si può anche dare a `root` analiticamente per
velocizzare e stabilizzare.


In [ ]:
# Trovare equilibri del modello cardiovascolare con baroriflesso
def steady_state_baro(I0):
    def F(X):
        p_sa, p_sv, p_ra, R_sa, K = X
        Q_co = K * p_ra
        return np.array([
            (Q_co - (p_sa - p_sv)/R_sa + I0)/C_sa,
            ((p_sa - p_sv)/R_sa - (p_sv - p_ra)/R_sv)/C_sv,
            ((p_sv - p_ra)/R_sv - Q_co)/C_ra,
            (Rsa_ss(p_sa) - R_sa)/tau_R,
            (K_ss(p_sa)   - K)/tau_K,
        ])
    X0 = np.array([95, 5, 2, R0+dR*0.5, K0+dK*0.5])
    sol = root(F, X0, method="hybr")
    return sol.x

I_list = np.linspace(-30, 30, 21)
PA = []; R_curve = []; K_curve = []
for I0 in I_list:
    x = steady_state_baro(I0)
    PA.append(x[0]); R_curve.append(x[3]); K_curve.append(x[4])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
axes[0].plot(I_list, PA, color=COL["main"], lw=2, marker="o")
axes[0].axhline(p_setpoint, color="gray", ls=":")
axes[0].set_xlabel("$I_{ext}$ [mL/s]"); axes[0].set_ylabel("$p_{sa}^*$ [mmHg]")
axes[0].set_title("Pressione a regime vs infusione")
axes[1].plot(I_list, R_curve, color=COL["accent"], lw=2, marker="o")
axes[1].set_xlabel("$I_{ext}$"); axes[1].set_ylabel("$R_{sa}^*$"); axes[1].set_title("$R_{sa}$ a regime")
axes[2].plot(I_list, K_curve, color=COL["ok"], lw=2, marker="o")
axes[2].set_xlabel("$I_{ext}$"); axes[2].set_ylabel("$K^*$"); axes[2].set_title("$K$ a regime")
plt.tight_layout(); plt.show()


**Cosa mostra il grafico** — la pressione a regime resta vicina al
setpoint $\approx 95$ mmHg anche per ingressi molto diversi: il baroriflesso
"appiattisce" la dipendenza variando *automaticamente* $R_{sa}$ e $K$. È
la firma di un buon controllo proporzionale + integratore lento.


# Parte 9 — Esercizi guidati con soluzioni

Questa parte è progettata per **scrivere codice** e **modificare modelli**.
Per ogni esercizio: testo, traccia di soluzione, codice di riferimento.

## Esercizio A — La membrana come filtro passa-basso

Considera una membrana passiva con $C_m=1\;\mu\text{F/cm}^2$,
$g_L=0.3\;\text{mS/cm}^2$. Inietta una corrente sinusoidale a varie
frequenze:

$$
I_{ext}(t) = I_0 \sin(2\pi f t).
$$

1. Calcola analiticamente l'ampiezza della risposta di $V_m$ in funzione
   di $f$.
2. Verifica numericamente che la membrana è un **filtro passa-basso** con
   frequenza di taglio $f_c=1/(2\pi\tau)$, $\tau=C_m/g_L$.


In [ ]:
# Soluzione esercizio A
C_m, g_L, E_L = 1.0, 0.3, -65.0
tau = C_m/g_L
I0 = 1.0
freqs = np.logspace(-2, 1.0, 30)  # Hz... ma qui ms quindi unita' di freq 1/ms = kHz

# Risposta analitica: |V(f)|/I0 = 1/g_L / sqrt(1 + (2 pi f tau)^2)
H_analytic = (1/g_L) / np.sqrt(1 + (2*np.pi*freqs*tau)**2)

# Simulazione: misuriamo l'ampiezza in steady state
V_amplitude = []
for f in freqs:
    def I_ext(t): return I0 * np.sin(2*np.pi*f*t)
    def rhs(t, V): return np.array([(I_ext(t) - g_L*(V[0]-E_L))/C_m])
    T = max(50.0, 5.0/f)
    sol = solve_ivp(rhs, (0, T), [E_L], dense_output=True, max_step=min(0.05, 0.05/f))
    # Stima ampiezza nella seconda meta'
    V = sol.y[0]
    mask = sol.t > T*0.5
    V_amplitude.append((V[mask].max() - V[mask].min())/2)

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(freqs, H_analytic, color=COL["main"], lw=2, label="analitica $1/g_L\\sqrt{1+(2\\pi f \\tau)^2}$")
ax.loglog(freqs, V_amplitude, "o", color=COL["accent"], label="simulata")
fc = 1/(2*np.pi*tau)
ax.axvline(fc, color="gray", ls=":")
ax.text(fc*1.1, 1, f"$f_c=1/(2\\pi\\tau)={fc:.3f}$", color="gray")
ax.set_xlabel("frequenza [1/ms]"); ax.set_ylabel("ampiezza |V(f)| [mV per µA/cm²]")
ax.set_title("Esercizio A: membrana passiva = filtro passa-basso")
ax.legend(); plt.tight_layout(); plt.show()


## Esercizio B — Analisi di stabilità di un modello non lineare

Considera un'equazione cinetica di Goldbeter (semplificata):

$$
\dot x = v_0 - \frac{V_M x^n}{K^n + x^n}.
$$

Per quali valori di $v_0$ il sistema ha 1, 2 o 3 equilibri? Studia la
stabilità.


In [ ]:
# Soluzione esercizio B
V_M, K, n_hill = 1.0, 0.5, 4

def f(x, v0): return v0 - V_M * x**n_hill / (K**n_hill + x**n_hill)

# Curve f(x) per vari v0
x = np.linspace(0, 2, 400)
fig, ax = plt.subplots(figsize=(9.5, 5))
for v0, c in zip([0.1, 0.3, 0.6, 0.9],
                  [COL["main"], COL["ok"], COL["warn"], COL["accent"]]):
    ax.plot(x, f(x, v0), color=c, lw=2, label=f"$v_0={v0}$")
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("$x$"); ax.set_ylabel("$f(x)=\\dot x$")
ax.set_title("Numero di equilibri al variare di $v_0$ (intersezioni con asse 0)")
ax.legend(); plt.tight_layout(); plt.show()

# Bisezione per trovare equilibri stabili
from scipy.optimize import brentq
print("Equilibri per v0=0.6:")
roots = []
for xa, xb in [(0.01, 0.4), (0.4, 1.0), (1.0, 2.0)]:
    try:
        r = brentq(lambda x: f(x, 0.6), xa, xb)
        roots.append(r)
        df_num = (f(r+1e-5, 0.6) - f(r-1e-5, 0.6)) / 2e-5
        print(f"  x*={r:.4f}, df/dx={df_num:+.3f} -> {'stabile' if df_num<0 else 'instabile'}")
    except ValueError:
        pass


## Esercizio C — Dal Hodgkin–Huxley al rate di sparo

Misura la frequenza di sparo del modello HH al variare di $I_{ext}$
costante. Disegna il "curva F–I" (frequency vs current).


In [ ]:
# Soluzione esercizio C
def measure_firing_rate(I_amp, T=200):
    def I_ext(t): return I_amp if t > 10 else 0.0
    sol = solve_ivp(lambda t, y: hh_rhs(t, y, I_ext), (0, T), [V0, n0, m0, h0],
                    dense_output=True, max_step=0.02, rtol=1e-7, atol=1e-9)
    V = sol.y[0]; t = sol.t
    # Conta spike: pendenze positive che attraversano 0 mV
    crossings = np.where((V[:-1] < 0) & (V[1:] > 0))[0]
    crossings = [c for c in crossings if t[c] > 30]
    if len(crossings) < 2: return 0
    times = t[crossings]
    return 1000.0 / np.mean(np.diff(times))  # Hz

I_vals = np.linspace(0, 25, 18)
rates = [measure_firing_rate(I) for I in I_vals]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(I_vals, rates, "o-", color=COL["main"], lw=2, ms=8)
ax.set_xlabel("$I_{ext}$ [µA/cm²]"); ax.set_ylabel("frequenza di sparo [Hz]")
ax.set_title("Curva F-I del modello di Hodgkin-Huxley")
ax.axvline(6.0, color="gray", ls=":", label="soglia di sparo $\\approx 6$ µA/cm²")
ax.legend(); plt.tight_layout(); plt.show()


**Interpretazione** — la curva F-I del modello HH è "saltatoria":
sotto soglia la cellula non spara; sopra soglia spara a una frequenza
*finita* (≈ 50-60 Hz) e poi cresce lentamente. È una transizione tipica
da "quiescente" a "spike" tramite una biforcazione **subcritica di Hopf** (il
modello HH classico ha proprio questa caratteristica).


## Esercizio D — Risposta di un compartimento vascolare a una sinusoide

Modella un singolo compartimento vascolare con $C=1.5$ mL/mmHg e
$R=0.95$ mmHg·s/mL. Iniettagli un flusso sinusoidale a varie frequenze
(da 0.1 a 5 Hz). Misura:

1. l'ampiezza della pressione di uscita rispetto a quella d'ingresso;
2. lo sfasamento.

Confronta con la **funzione di trasferimento analitica** del compartimento RC.

**Suggerimento didattico**: il compartimento vascolare è equivalente a un
filtro **passa-basso**: a basse frequenze segue l'input quasi senza
attenuazione; ad alte frequenze attenua e introduce ritardo. La frequenza
di taglio (–3 dB) è $f_c = 1/(2\pi RC)$.


In [ ]:
# Soluzione esercizio D: risposta in frequenza del compartimento vascolare
C_vasc, R_vasc = 1.5, 0.95
tau = R_vasc * C_vasc  # costante di tempo
fc = 1/(2*np.pi*tau)
print(f"Costante di tempo: tau = {tau:.3f} s")
print(f"Frequenza di taglio (-3 dB): f_c = {fc:.3f} Hz")

# Funzione di trasferimento del compartimento: P/Q = R/(1 + j*omega*RC)
freqs = np.logspace(-2, 1.0, 30)
H_analytic = R_vasc / np.sqrt(1 + (2*np.pi*freqs*tau)**2)
phase_analytic = -np.degrees(np.arctan(2*np.pi*freqs*tau))

# Simulazione: misuriamo ampiezza e sfasamento numericamente
amplitudes = []
phases = []
Q0 = 10.0  # mL/s
for f in freqs:
    def Q(t): return Q0 * np.sin(2*np.pi*f*t)
    def rhs(t, P): return np.array([(Q(t) - P[0]/R_vasc)/C_vasc])
    T = max(30.0, 8/f)
    sol = solve_ivp(rhs, (0, T), [0.0], dense_output=True, max_step=min(0.02, 0.02/f))
    P = sol.y[0]; t = sol.t
    mask = t > T*0.6
    amp = (P[mask].max() - P[mask].min())/2 / Q0
    amplitudes.append(amp)
    # Sfasamento: trovo dove P attraversa zero in salita
    idx_input = np.argmin(np.abs(np.sin(2*np.pi*f*t[mask])))
    idx_out = np.argmin(np.abs(P[mask] - P[mask].mean()))
    phases.append(0)  # semplificazione

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ax = axes[0]
ax.loglog(freqs, H_analytic, color=COL["main"], lw=2.2, label="analitica")
ax.loglog(freqs, amplitudes, "o", color=COL["accent"], ms=7, label="simulata")
ax.axvline(fc, color="gray", ls=":")
ax.text(fc*1.05, 0.1, f"$f_c={fc:.2f}$ Hz", color="gray")
ax.set_xlabel("frequenza [Hz]"); ax.set_ylabel("|P/Q| [mmHg per mL/s]")
ax.set_title("Esercizio D: ampiezza vs frequenza\n(compartimento vascolare = passa-basso)")
ax.legend()

ax = axes[1]
ax.semilogx(freqs, phase_analytic, color=COL["main"], lw=2.2)
ax.axvline(fc, color="gray", ls=":")
ax.axhline(-45, color="gray", lw=0.6, ls="--")
ax.text(fc*1.05, -47, "−45° a $f_c$", color="gray")
ax.set_xlabel("frequenza [Hz]"); ax.set_ylabel("sfasamento [°]")
ax.set_title("Sfasamento analitico: ritardo crescente con la frequenza")
plt.tight_layout(); plt.show()


**Interpretazione fisiologica** — il sistema arterioso si comporta da
filtro passa-basso: le pulsazioni cardiache "veloci" (≈1 Hz) vengono
attenuate dalle compliance dei grossi vasi prima di arrivare ai capillari.
È il meccanismo del **Windkessel**: la compliance "smorza" la pulsatilità.

## Esercizio E — Confronto Lotka–Volterra vs Holling: chi vince?

Implementa entrambi i modelli (puro Lotka–Volterra e Holling II + logistica).
Calcola e plotta:

1. il **periodo medio delle oscillazioni** in funzione delle condizioni iniziali per LV puro;
2. il **periodo del ciclo limite** in Holling II (deve essere costante!).

Confronta come cambia la "qualità" delle oscillazioni nei due modelli al
variare di un parametro.


In [ ]:
# Soluzione esercizio E
from scipy.signal import find_peaks

# LV puro
a_lv, b_lv, c_lv, d_lv = 1.0, 0.5, 0.75, 0.25
def lv(t, z): x, y = z; return np.array([a_lv*x - b_lv*x*y, -c_lv*y + d_lv*x*y])

# Holling II + logistica (rinomino i parametri con suffisso _hol per non
# collidere con alpha_h/beta_h che sono le variabili di gating di HH)
A_hol, K_hol, beta_hol, h_hol, eps_hol, c_hol = 1.0, 6.0, 1.2, 0.6, 0.6, 0.5
def holling(t, z):
    x, y = z
    Phi = beta_hol * x / (1 + h_hol * x)
    return np.array([A_hol*x*(1 - x/K_hol) - Phi*y, eps_hol*Phi*y - c_hol*y])

# Misuro periodo
def period(model, ic, T_max=120):
    t_eval = np.linspace(0, T_max, 12000)
    sol = solve_ivp(model, (0, T_max), ic, t_eval=t_eval, rtol=1e-9, atol=1e-12)
    # picchi della prima variabile
    peaks, _ = find_peaks(sol.y[0], distance=20)
    if len(peaks) < 3: return np.nan
    return np.mean(np.diff(sol.t[peaks[-min(5, len(peaks)-1):]]))

# LV: periodo dipende dall'ampiezza iniziale
ic_amplitudes = np.linspace(0.5, 3.0, 8)
lv_periods = []
for amp in ic_amplitudes:
    ic = [c_lv/d_lv + amp, a_lv/b_lv]
    lv_periods.append(period(lv, ic))

# Holling: periodo del ciclo limite (indipendente da IC, dentro al bacino)
hol_periods = []
for amp in ic_amplitudes:
    ic = [1.0 + amp*0.3, 0.5 + amp*0.1]
    hol_periods.append(period(holling, ic))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ic_amplitudes, lv_periods, "o-", color=COL["accent"], lw=2, ms=8,
        label="Lotka–Volterra puro (periodo cresce con l'ampiezza IC)")
ax.plot(ic_amplitudes, hol_periods, "s-", color=COL["main"], lw=2, ms=8,
        label="Holling II + logistica (periodo costante = ciclo limite)")
ax.set_xlabel("'distanza' dell'IC dall'equilibrio")
ax.set_ylabel("periodo misurato")
ax.set_title("Esercizio E: LV puro (orbite chiuse) vs Holling II (ciclo limite)")
ax.legend()
plt.tight_layout(); plt.show()


**Lezione dell'esercizio** — Lotka–Volterra è strutturalmente
*fragile*: il periodo dipende dalle condizioni iniziali. Un modello
ecologico realistico (Holling) ha un **ciclo limite** vero, con periodo
intrinseco: cambia poco se sposti l'IC. È il motivo per cui in ecologia
si studia il "paradosso dell'arricchimento" (aumentare $K$ riduce la
stabilità).

## Esercizio F — Cheyne-Stokes come Hopf-instabilità

Riprendi il modello di controllo ventilatorio della Parte 7.7. Trova
*numericamente* la soglia di $\tau_d$ (ritardo) oltre la quale il loop
diventa instabile. Calcola l'ampiezza delle oscillazioni a regime
in funzione di $\tau_d$.

**Suggerimento**: per ogni $\tau_d$, simula a regime e misura
l'ampiezza picco–picco di $P_{ACO_2}$.


In [ ]:
# Soluzione esercizio F: bifurcation diagram empirico di Cheyne-Stokes

VCO2 = 200.0
G = 1.2; P_th = 38.0; V0 = 1.0

def simulate_chemo(tau_d, T=240, dt=0.05, P0=44.0):
    N = int(T/dt) + 1
    t = np.linspace(0, T, N)
    PA = np.full(N, P0); VA_arr = np.zeros(N)
    n_delay = max(1, int(tau_d/dt))
    for k in range(N - 1):
        P_chemo = PA[max(0, k - n_delay)]
        VA = max(0.5, V0 + G * (P_chemo - P_th))
        VA_arr[k] = VA
        P_inf = VCO2 / (VA*1000/60.0) * 0.863 * 1000
        tau_lung = 5.0
        PA[k+1] = PA[k] + dt/tau_lung * (P_inf - PA[k])
    return t, PA, VA_arr

tau_d_list = np.linspace(0.5, 25, 16)
amplitudes = []
for td in tau_d_list:
    t, PA, _ = simulate_chemo(td, T=300)
    # Misuro l'ampiezza nell'ultimo terzo di simulazione (regime asintotico)
    PA_late = PA[t > 200]
    amplitudes.append((PA_late.max() - PA_late.min())/2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.plot(tau_d_list, amplitudes, "o-", color=COL["main"], lw=2, ms=8)
ax.axhline(2.0, color="gray", ls=":")
ax.set_xlabel(r"ritardo $\tau_d$ [s]")
ax.set_ylabel("ampiezza picco-picco/2 di $P_{ACO_2}$ [mmHg]")
ax.set_title("Esercizio F: diagramma di biforcazione empirico Hopf")

# Tre esempi di dinamica
ax = axes[1]
for td, c in zip([3.0, 10.0, 18.0], [COL["ok"], COL["warn"], COL["accent"]]):
    t, PA, _ = simulate_chemo(td, T=200)
    ax.plot(t, PA, color=c, lw=1.5, label=fr"$\tau_d={td}$ s")
ax.set_xlabel("tempo [s]"); ax.set_ylabel(r"$P_{ACO_2}$ [mmHg]")
ax.set_title("Dinamica al variare di $\\tau_d$")
ax.legend()
plt.tight_layout(); plt.show()


**Interpretazione clinica** — il diagramma di biforcazione mostra
chiaramente la **biforcazione di Hopf**: sotto una certa soglia di $\tau_d$
le oscillazioni sono assenti (sistema asintoticamente stabile), sopra la
soglia compaiono oscillazioni di ampiezza crescente. Nei pazienti con
**scompenso cardiaco** il tempo di circolo aumenta (perché il cuore pompa
meno) → $\tau_d$ aumenta → si entra nella zona di Cheyne–Stokes.

## Esercizio G — Calibrare HH a un dato sperimentale "rumoroso"

Genera dati "sperimentali" di un potenziale d'azione HH aggiungendo
rumore, poi cerca di ri-stimare $\bar g_{Na}$ e $\bar g_K$ da quei dati.

Vediamo quanto bene la calibrazione recupera i veri valori dei parametri.


In [ ]:
# Soluzione esercizio G: calibrazione di HH da dati rumorosi
from scipy.optimize import minimize

# "Veri" parametri usati per generare i dati
gNa_true, gK_true = 120.0, 36.0

def hh_with(gNa, gK, T=25, V0=-65.0, n0=None, m0=None, h0=None):
    if n0 is None:
        n0 = alpha_n(V0)/(alpha_n(V0)+beta_n(V0))
        m0 = alpha_m(V0)/(alpha_m(V0)+beta_m(V0))
        h0 = alpha_h(V0)/(alpha_h(V0)+beta_h(V0))
    def I_ext(t): return 10.0 if 2 <= t <= 2.5 else 0.0
    def rhs(t, y):
        V, n, m, h = y
        I_Na = gNa * m**3 * h * (V - 50.0)
        I_K  = gK  * n**4     * (V - (-77.0))
        I_L  = 0.3 * (V - (-54.387))
        return [(I_ext(t) - I_Na - I_K - I_L)/1.0,
                alpha_n(V)*(1-n) - beta_n(V)*n,
                alpha_m(V)*(1-m) - beta_m(V)*m,
                alpha_h(V)*(1-h) - beta_h(V)*h]
    return solve_ivp(rhs, (0, T), [V0, n0, m0, h0], dense_output=True,
                     max_step=0.02, rtol=1e-7, atol=1e-9)

# Genero dati con i veri parametri + rumore
np.random.seed(42)
sol_true = hh_with(gNa_true, gK_true)
t_data = np.linspace(0, 25, 80)
V_clean = np.interp(t_data, sol_true.t, sol_true.y[0])
V_data = V_clean + 2.0 * np.random.randn(len(t_data))

# Funzione costo
def cost(params):
    gNa, gK = params
    try:
        sol = hh_with(gNa, gK)
        V_model = np.interp(t_data, sol.t, sol.y[0])
        return np.mean((V_model - V_data)**2)
    except Exception:
        return 1e10

# Ottimizzazione con punto iniziale "sbagliato"
res = minimize(cost, x0=[80.0, 25.0], method="Nelder-Mead",
               options={"xatol": 0.5, "fatol": 0.5, "maxiter": 100})
gNa_hat, gK_hat = res.x
print(f"Veri:  g_Na={gNa_true}, g_K={gK_true}")
print(f"Stima: g_Na={gNa_hat:.1f}, g_K={gK_hat:.1f}")
print(f"Errore relativo: {abs(gNa_hat-gNa_true)/gNa_true*100:.1f}% (Na), "
      f"{abs(gK_hat-gK_true)/gK_true*100:.1f}% (K)")

# Plot finale
fig, ax = plt.subplots(figsize=(10, 5))
sol_fit = hh_with(gNa_hat, gK_hat)
ax.plot(sol_true.t, sol_true.y[0], color="black", lw=1, alpha=0.5, label="vero (clean)")
ax.plot(t_data, V_data, "o", color=COL["accent"], ms=4, label="dati rumorosi")
ax.plot(sol_fit.t, sol_fit.y[0], color=COL["main"], lw=2,
        label=f"fit g_Na={gNa_hat:.0f}, g_K={gK_hat:.0f}")
ax.set_xlabel("t [ms]"); ax.set_ylabel("V [mV]")
ax.set_title("Esercizio G: calibrazione di HH a dati rumorosi")
ax.legend()
plt.tight_layout(); plt.show()


**Note pedagogiche sull'esercizio G** — la calibrazione di HH è
*difficile* perché:

1. Le conduttanze sono fortemente accoppiate (riducendo $g_{Na}$ e $g_K$
   insieme di pari percentuale, la forma del potenziale d'azione cambia
   poco) → **non identificabilità pratica**.
2. La funzione costo è altamente non convessa, e il sistema può non
   sparare per molte combinazioni di parametri → discontinuità della costo
   → ottimizzatori locali rischiano di rimanere intrappolati.

In ricerca si usa **MCMC bayesiano** o **simulated annealing** per HH; un
solo Nelder-Mead è solo una demo didattica.

### Tabella di sintesi degli esercizi

| Esercizio | Difficoltà | Tema centrale |
|---|---|---|
| A | ★ | Membrana passiva come filtro passa-basso |
| B | ★★ | Equilibri e biforcazioni in un modello scalare |
| C | ★★ | Curva F–I di HH e biforcazione subcritica |
| D | ★★ | Funzione di trasferimento di un compartimento vascolare |
| E | ★★★ | Orbite chiuse vs ciclo limite: confronto modelli ecologici |
| F | ★★★ | Diagramma di biforcazione empirico (Hopf) del controllo ventilatorio |
| G | ★★★★ | Calibrazione di HH a dati rumorosi e (non-)identificabilità |


# Appendice — Cheat sheet finale

### Da una equazione a un modello in Python (procedura standard)

1. Scrivi i bilanci.
2. Identifica stato $\mathbf{x}$, ingresso $\mathbf{u}$, parametri.
3. Esprimi $\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x},\mathbf{u})$.
4. Implementa `rhs(t, x)` come funzione.
5. Usa `solve_ivp` per la simulazione, con `method='RK45'` o `BDF` se stiff.
6. Trova equilibri con `scipy.optimize.root(rhs_steadystate, x0)`.
7. Per stabilità: calcola $J$ con differenze finite o analiticamente, e
   guarda gli autovalori.

### Comandi numpy / scipy "di sopravvivenza"

```python
np.linalg.eig(A)            # autovalori, autovettori
scipy.linalg.expm(A*t)      # e^{At}
scipy.optimize.root(f, x0)  # zero di f
scipy.integrate.solve_ivp(  # ODE
    rhs, (t0, tf), x0,
    t_eval=tg, method="RK45",
    rtol=1e-8, atol=1e-10)
scipy.optimize.curve_fit(model, t, y, p0=[...])
```

### Errori tipici

- **Dimensioni**: scordarsi che $1\,F = 1\,C/V$, $1\,F \cdot 1\,V/s = 1\,A$.
- **Tempo in unità sbagliate**: HH usa ms, cardio usa s. Mai confonderli.
- **Eulero esplicito su sistema stiff**: la simulazione esplode.
- **Equilibrio cercato in solitudine**: senza condizioni iniziali ragionevoli
  `root` può trovare equilibri non fisici (es. concentrazioni negative).
- **Cancellazione catastrofica** in $1-\exp(\cdot)$ vicino a zero: usare
  `np.expm1` se necessario.
- **Linearizzazione lontano dall'equilibrio**: il Jacobiano predice solo
  la dinamica locale.

### Per andare oltre

- Strogatz, *Nonlinear Dynamics and Chaos*.
- Keener & Sneyd, *Mathematical Physiology*.
- Hoppensteadt & Peskin, *Modeling and Simulation in Medicine and the Life
  Sciences*.
- Hairer, Norsett, Wanner, *Solving Ordinary Differential Equations* I/II.

> **Buona modellistica!** L'unica vera differenza tra un esercizio scritto
> sulla lavagna e un modello che spiega un fenomeno reale è la *cura* con cui
> si definiscono le variabili, si scrivono i bilanci e si verifica la
> coerenza dimensionale e numerica.
